# Technical Debt Prediction — Standalone Colab Pipeline

End-to-end reproduction of the LightGBM high-risk technical-debt classifier
on 22 Apache Java projects. The only external input is `td_V2.db` in
Google Drive.

## Section 0 — Setup

Mount Drive, configure paths, install non-pre-installed packages, and verify
the environment. **Edit `DRIVE_BASE` in cell 0.1 if your Drive layout differs.**

In [ ]:
# 0.1 — Mount Drive and copy the raw database to local disk
from google.colab import drive
drive.mount('/content/drive')

# >>> Edit this one line if your Drive layout differs <<<
DRIVE_BASE = '/content/drive/MyDrive/td_thesis'

import os, shutil
from pathlib import Path

PROJECT_ROOT = Path('/content')
os.chdir(PROJECT_ROOT)
(PROJECT_ROOT / 'data' / 'raw').mkdir(parents=True, exist_ok=True)

src_db = Path(DRIVE_BASE) / 'data' / 'raw' / 'td_V2.db'
dst_db = PROJECT_ROOT / 'data' / 'raw' / 'td_V2.db'
if not src_db.exists():
    raise FileNotFoundError(f'Place td_V2.db at: {src_db}')

# Always recopy from Drive — partial copies on free-tier Colab silently
# truncate large files. Delete any existing local copy first.
dst_db.unlink(missing_ok=True)
shutil.copy(src_db, dst_db)

src_size = src_db.stat().st_size
dst_size = dst_db.stat().st_size
if dst_size != src_size:
    raise RuntimeError(
        f'Copy truncated: source={src_size:,} bytes, local={dst_size:,} bytes. '
        'Re-run this cell.'
    )
print(f'DB ready at {dst_db}  ({dst_size / 1e9:.2f} GB, matches Drive)')


In [ ]:
# 0.2 — Project layout, global config (constants from config.py), shared helpers
import os, sys, json, time, shutil, warnings, sqlite3, re, math
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Iterable, Optional, List, Tuple, Dict
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ---- paths ----
PROJECT_ROOT = Path('/content')
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'
MODELS_DIR = PROJECT_ROOT / 'models'
for _d in (PROCESSED_DATA_DIR, FIGURES_DIR, TABLES_DIR, TABLES_DIR / 'db_samples', MODELS_DIR):
    _d.mkdir(parents=True, exist_ok=True)
TD_DATASET_PATH = RAW_DATA_DIR / 'td_V2.db'

# ---- dataset filters ----
SOURCE_FILE_EXTENSIONS = ('.java',)
PATH_EXCLUSION_PATTERNS = ('/test/', '/tests/', '/generated/', '/generated-sources/', '/target/', '/build/')

# ---- snapshot ----
SNAPSHOT_STRATEGY = 'median'
OBSERVATION_WINDOW_MONTHS = 6
LABEL_SURROGATE_WINDOW_MONTHS = 6
MIN_PRE_SNAPSHOT_COMMITS = 500
MIN_POST_SNAPSHOT_COMMITS = 50

# ---- labeling ----
BUGFIX_REGEX = r"\b(fix|bug|defect|patch|resolve|repair)\b"
JIRA_ISSUE_KEY_PATTERN = r"\b([A-Z][A-Z0-9_]+)-(\d+)\b"
THEORETICAL_WEIGHTS = {
    'S1_severity': 0.30, 'S4_bugfix': 0.25, 'S2_debt': 0.20,
    'S5_churn': 0.15, 'S3_smells': 0.05, 'S6_contributors': 0.05,
}
LABEL_RISK_THRESHOLD = 0.50
LABEL_THRESHOLD_FALLBACKS = (0.45, 0.40)
LABEL_MIN_POSITIVES_PER_PROJECT = 5
SEVERITY_LABEL_LEVELS = ('BLOCKER', 'CRITICAL')

# ---- feature catalogue (27 features, 5 families) ----
SIZE_COMPLEXITY_FEATURES = ['ncloc','complexity','cognitive_complexity','functions','classes']
STATIC_DEBT_FEATURES = ['n_code_smells','n_bugs','total_debt_minutes','issue_density','duplicated_lines_density']
HISTORICAL_FEATURES = ['total_commits_pre','code_churn_pre','recent_churn_90d','commit_frequency_30d',
                       'file_age_days','days_since_last_change','contributor_count','ownership_ratio']
GRAPH_FEATURES = ['cocg_degree','cocg_pagerank','cocg_betweenness','cocg_entropy']
PRIOR_DEFECT_FEATURES = ['bugfix_commits_pre','bugfix_commits_90d','bug_density_pre','n_jira_bugs_pre','jira_blocker_flag']
FEATURE_FAMILIES = {
    'size_complexity': SIZE_COMPLEXITY_FEATURES,
    'static_debt': STATIC_DEBT_FEATURES,
    'historical': HISTORICAL_FEATURES,
    'graph': GRAPH_FEATURES,
    'prior_defect': PRIOR_DEFECT_FEATURES,
}
ALL_FEATURES = SIZE_COMPLEXITY_FEATURES + STATIC_DEBT_FEATURES + HISTORICAL_FEATURES + GRAPH_FEATURES + PRIOR_DEFECT_FEATURES
assert len(ALL_FEATURES) == 27, f'expected 27 features, got {len(ALL_FEATURES)}'
LOG1P_FEATURES = ['ncloc','complexity','total_commits_pre','code_churn_pre','recent_churn_90d',
                  'file_age_days','total_debt_minutes','bugfix_commits_pre','n_jira_bugs_pre']
FEATURE_SELECTION_THRESHOLD = 0.001

# ---- modeling ----
RANDOM_STATE = 42
MODEL_ORDER = ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']
CV_FOLDS = 10
TUNING_TRIALS = 30
TUNING_INNER_CV_FOLDS = 5
TUNING_OBJECTIVE = 'pr_auc'
COST_EFFECTIVENESS_AT = 0.20
THRESHOLD_SWEEP_MIN = 0.05
THRESHOLD_SWEEP_MAX = 0.80
THRESHOLD_SWEEP_STEP = 0.01
THRESHOLD_DEFAULT = 0.30
THRESHOLD_MIN_POSITIVES = 3

# Lower parallelism in Colab to avoid pickling overhead.
STAGE5_N_JOBS = min(2, os.cpu_count() or 2)

# ---- display helpers ----
from IPython.display import Image, display

def show_figure(rel_path):
    p = Path(rel_path)
    if not p.exists():
        print(f'(figure not found: {p})'); return
    display(Image(filename=str(p)))

def show_table(csv_path, highlight_col=None, top=None):
    df = pd.read_csv(csv_path)
    if top is not None:
        df = df.head(top)
    num = df.select_dtypes('number').columns
    if len(num):
        df[num] = df[num].round(4)
    if highlight_col and highlight_col in df.columns:
        display(df.style.highlight_max(subset=[highlight_col], color='lightgreen'))
    else:
        display(df)

print('Paths + config ready.  PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# 0.3 — Install only the packages not pre-installed in Colab
!pip install -q xgboost>=2.0.0 lightgbm>=4.0.0 optuna>=3.5.0 shap>=0.44.0 \
    pyarrow>=14.0.0 pydriller>=2.5 imbalanced-learn>=0.11.0 \
    matplotlib-venn>=0.11.9 tabulate>=0.9.0 networkx>=3.2 igraph>=0.11
print('Dependencies installed.')


In [ ]:
# 0.4 — Verify environment
import platform
import xgboost, lightgbm, optuna, shap
print(f'Python      : {platform.python_version()}')
print(f'xgboost     : {xgboost.__version__}')
print(f'lightgbm    : {lightgbm.__version__}')
print(f'optuna      : {optuna.__version__}')
print(f'shap        : {shap.__version__}')

with sqlite3.connect(str(TD_DATASET_PATH)) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
print(f'DB tables    : {len(tables)}')
print(tables['name'].tolist())


## Section 1 — Database Inspection

Inspect the raw SQLite database schema and confirm all 10 tables are present.
Writes `results/tables/db_schema.csv`, `db_table_counts.csv`, and per-table
sample CSVs.

In [ ]:
# Pipeline code

"""
Thin SQL readers for the Technical Debt Dataset v2.0.

This module contains ONLY raw SELECT helpers - no cleaning, no aggregation,
no domain logic. Cleaning lives in ``src.data.clean`` and labeling in
``src.data.labeling``.

Every function accepts an optional ``sqlite3.Connection``; callers that open
many queries should share one connection to avoid file-open overhead on a
1.5 GB database.

References
----------
Lenarduzzi, V., Saarimaki, N., Taibi, D. (2019). The Technical Debt Dataset.
Proc. PROMISE 2019.
"""

import os
import sqlite3
import sys
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd


def _td_projects_env_allowlist() -> Optional[list[str]]:
    """Optional comma-separated subset from ``TD_PROJECTS`` (Colab demo / smoke runs).

    Entries without a colon are prefixed with ``org.apache:`` to match TD v2 IDs.
    """
    raw = os.environ.get("TD_PROJECTS", "").strip()
    if not raw:
        return None
    out: list[str] = []
    for part in raw.split(","):
        p = part.strip()
        if not p:
            continue
        out.append(p if ":" in p else f"org.apache:{p}")
    return out


# ---------------------------------------------------------------------------
# Connection helpers
# ---------------------------------------------------------------------------
def get_connection(db_path: Optional[Path] = None) -> sqlite3.Connection:
    """Open a SQLite connection to the Technical Debt Dataset."""
    db_path = Path(db_path) if db_path else TD_DATASET_PATH
    if not db_path.exists():
        raise FileNotFoundError(
            f"Database not found at {db_path}\n"
            f"Download: https://github.com/clowee/The-Technical-Debt-Dataset/releases"
        )
    return sqlite3.connect(str(db_path))


def get_table_names(conn: sqlite3.Connection) -> list[str]:
    """List all tables in the database."""
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    return [r[0] for r in cur.fetchall()]


def get_table_schema(conn: sqlite3.Connection, table: str) -> pd.DataFrame:
    """Return PRAGMA table_info rows for a table."""
    return pd.read_sql_query(f"PRAGMA table_info({table});", conn)


def get_row_count(conn: sqlite3.Connection, table: str) -> int:
    """Return COUNT(*) of a table."""
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    return cur.fetchone()[0]


def get_sample_rows(conn: sqlite3.Connection, table: str, n: int = 3) -> pd.DataFrame:
    """Return the first ``n`` rows of a table."""
    return pd.read_sql_query(f"SELECT * FROM {table} LIMIT {int(n)};", conn)


# ---------------------------------------------------------------------------
# Table-specific loaders
# ---------------------------------------------------------------------------
def list_projects(conn: sqlite3.Connection) -> list[str]:
    """Return the sorted list of distinct project IDs using GIT_COMMITS."""
    try:
        rows = pd.read_sql_query(
            "SELECT DISTINCT PROJECT_ID FROM GIT_COMMITS ORDER BY PROJECT_ID",
            conn,
        )
    except Exception:
        rows = pd.read_sql_query(
            "SELECT DISTINCT PROJECT_ID FROM SONAR_ISSUES ORDER BY PROJECT_ID",
            conn,
        )
    projects = rows["PROJECT_ID"].dropna().astype(str).tolist()
    allow = _td_projects_env_allowlist()
    if allow is not None:
        allow_set = frozenset(allow)
        projects = [p for p in projects if p in allow_set]
    return projects


def load_git_commits(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
    main_branch_only: bool = True,
    columns: Optional[list[str]] = None,
) -> pd.DataFrame:
    """Load GIT_COMMITS, optionally restricted to projects / main branch.

    Column names verified in Stage 1: ``COMMIT_HASH``, ``COMMIT_MESSAGE``,
    ``AUTHOR_DATE``, ``COMMITTER_DATE``, ``IN_MAIN_BRANCH`` (stored as text
    strings ``'True'``/``'False'``).
    """
    cols_expr = ", ".join(columns) if columns else "*"
    sql = f"SELECT {cols_expr} FROM GIT_COMMITS"
    where = []
    if main_branch_only:
        where.append("IN_MAIN_BRANCH = 'True'")
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        where.append(f"PROJECT_ID IN ({ids})")
    if where:
        sql += " WHERE " + " AND ".join(where)
    df = pd.read_sql_query(sql, conn)
    if "AUTHOR_DATE" in df.columns:
        df["AUTHOR_DATE"] = pd.to_datetime(df["AUTHOR_DATE"], errors="coerce", utc=True)
    if "COMMITTER_DATE" in df.columns:
        df["COMMITTER_DATE"] = pd.to_datetime(df["COMMITTER_DATE"], errors="coerce", utc=True)
    return df


def load_git_commits_changes(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Load GIT_COMMITS_CHANGES, optionally restricted to projects."""
    sql = "SELECT * FROM GIT_COMMITS_CHANGES"
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        sql += f" WHERE PROJECT_ID IN ({ids})"
    return pd.read_sql_query(sql, conn)


def load_sonar_measures(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Load SONAR_MEASURES; schema is wide (one column per metric)."""
    sql = "SELECT * FROM SONAR_MEASURES"
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        sql += f" WHERE PROJECT_ID IN ({ids})"
    return pd.read_sql_query(sql, conn)


def load_sonar_issues(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Load SONAR_ISSUES."""
    sql = "SELECT * FROM SONAR_ISSUES"
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        sql += f" WHERE PROJECT_ID IN ({ids})"
    return pd.read_sql_query(sql, conn)


def load_jira_issues(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load the JIRA_ISSUES table. Schema is confirmed by Stage 1."""
    return pd.read_sql_query("SELECT * FROM JIRA_ISSUES", conn)


def load_szz_fault_inducing(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load SZZ fault-inducing / fault-fixing commit links.

    Columns verified in Stage 1: ``PROJECT_ID``, ``FAULT_FIXING_COMMIT_HASH``,
    ``FAULT_INDUCING_COMMIT_HASH``.
    """
    return pd.read_sql_query("SELECT * FROM SZZ_FAULT_INDUCING_COMMITS", conn)


def load_sonar_analysis(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load SONAR_ANALYSIS (maps ANALYSIS_KEY to snapshot DATE + Git REVISION).

    Columns verified in Stage 1: ``PROJECT_ID``, ``ANALYSIS_KEY``, ``DATE``,
    ``REVISION``. This table is the bridge between ``SONAR_MEASURES`` snapshots
    and Git history (via REVISION hash and DATE).
    """
    df = pd.read_sql_query("SELECT * FROM SONAR_ANALYSIS", conn)
    if "DATE" in df.columns:
        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce", utc=True)
    return df


def load_projects_meta(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load the PROJECTS metadata table (31 rows, PROJECT_ID + GIT_LINK + JIRA_LINK)."""
    return pd.read_sql_query("SELECT * FROM PROJECTS", conn)


def load_refactorings(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load refactoring records (optional qualitative validation)."""
    try:
        return pd.read_sql_query("SELECT * FROM REFACTORING_MINER", conn)
    except Exception:
        return pd.read_sql_query("SELECT * FROM REFACTORINGS", conn)


# ---------------------------------------------------------------------------
# Windowed helpers
# ---------------------------------------------------------------------------
def commits_in_window(
    conn: sqlite3.Connection,
    project_id: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    main_branch_only: bool = True,
) -> pd.DataFrame:
    """Return commits in the half-open window (start, end] for a project."""
    parts = ["PROJECT_ID = ?", "AUTHOR_DATE > ?", "AUTHOR_DATE <= ?"]
    params: list = [project_id, start.isoformat(), end.isoformat()]
    if main_branch_only:
        parts.append("IN_MAIN_BRANCH = 'True'")
    sql = "SELECT * FROM GIT_COMMITS WHERE " + " AND ".join(parts)
    df = pd.read_sql_query(sql, conn, params=params)
    if "AUTHOR_DATE" in df.columns:
        df["AUTHOR_DATE"] = pd.to_datetime(df["AUTHOR_DATE"], errors="coerce", utc=True)
    return df


"""
Stage 1 - Database smoke test and schema inventory.

Opens ``data/raw/td_V2.db`` and produces a canonical inventory of tables,
columns, row counts, and sample rows. The purpose is to verify the exact
schema (names, types) before writing any downstream SQL or feature code.

Outputs
-------
- ``results/tables/db_schema.csv`` - one row per (table, column), with
  column types and example value extracted from the first sample row.
- ``results/tables/db_table_counts.csv`` - one row per table with row count.
- ``results/tables/db_samples/<table>.csv`` - 3 sample rows per table.

Run
---
.. code-block:: bash

    .\\venv\\Scripts\\python.exe scripts/01_inspect_db.py
"""

import sys
from pathlib import Path

import pandas as pd


SAMPLES_DIR = TABLES_DIR / "db_samples"


def run_stage_1() -> None:
    print(f"[Stage 1] Opening: {TD_DATASET_PATH}")
    print(f"[Stage 1] Size (MB): {TD_DATASET_PATH.stat().st_size / (1024 ** 2):.1f}")

    SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

    with get_connection() as conn:
        tables = get_table_names(conn)
        print(f"[Stage 1] Found {len(tables)} tables:")
        for t in tables:
            print(f"   - {t}")

        schema_rows: list[dict] = []
        count_rows: list[dict] = []

        for table in tables:
            try:
                count = get_row_count(conn, table)
            except Exception as exc:
                print(f"   ! COUNT failed for {table}: {exc}")
                count = -1
            count_rows.append({"table": table, "row_count": count})

            try:
                schema = get_table_schema(conn, table)
                sample = get_sample_rows(conn, table, 3)
            except Exception as exc:
                print(f"   ! PRAGMA/sample failed for {table}: {exc}")
                continue

            sample.to_csv(SAMPLES_DIR / f"{table}.csv", index=False)

            first_row = sample.iloc[0] if len(sample) > 0 else pd.Series(dtype=object)
            for _, col in schema.iterrows():
                col_name = col["name"]
                example = first_row.get(col_name, None) if len(first_row) else None
                if isinstance(example, str) and len(example) > 120:
                    example = example[:117] + "..."
                schema_rows.append({
                    "table": table,
                    "column": col_name,
                    "type": col["type"],
                    "notnull": bool(col["notnull"]),
                    "pk": bool(col["pk"]),
                    "example_value": example,
                })

    schema_df = pd.DataFrame(schema_rows)
    counts_df = pd.DataFrame(count_rows).sort_values("row_count", ascending=False)

    schema_path = TABLES_DIR / "db_schema.csv"
    counts_path = TABLES_DIR / "db_table_counts.csv"
    schema_df.to_csv(schema_path, index=False)
    counts_df.to_csv(counts_path, index=False)

    print("\n[Stage 1] Row-count summary:")
    for _, r in counts_df.iterrows():
        print(f"   {r['table']:<40} {r['row_count']:>15,}")

    print(f"\n[Stage 1] Schema written to: {schema_path}")
    print(f"[Stage 1] Counts written to : {counts_path}")
    print(f"[Stage 1] Samples written to: {SAMPLES_DIR}")
    print("[Stage 1] Complete.")


In [ ]:
run_stage_1()
print()
print('--- db_table_counts.csv ---')
show_table('results/tables/db_table_counts.csv')


## Section 2 — Project Profiling

Profile 33 Apache projects and select the 22 eligible ones using the
500-pre / 50-post commit thresholds. Writes `project_stats.csv` and
`corpus_summary.csv`.

In [ ]:
# Pipeline code

"""
Snapshot date selection and temporal-split utilities.

Implements the temporal-split protocol described in the approved proposal
Section 3.4: for each project, a snapshot time ``t`` is selected such that
features are computed from commits with ``AUTHOR_DATE <= t`` and labels are
derived from commits in the observation window ``(t, t + W]`` where ``W``
is the observation window (6 months primary).

The default strategy is the **median master-branch commit date** per project,
which guarantees every project has both a meaningful pre-snapshot history
(for features) and a meaningful post-snapshot window (for labels).

References
----------
- Zimmermann, T., Nagappan, N. (2008). Predicting defects using network analysis
  on dependency graphs. ICSE 2008 - basis for 6-month observation window.
- Jiang, Z., Chen, T., Zhou, Y. (2024). Improving technical debt prediction with
  graph-based and social-network metrics. Empir. Softw. Eng. 29 - uses median
  snapshot strategy.
"""

import sqlite3
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd


@dataclass
class ProjectSnapshot:
    """Per-project snapshot metadata."""

    project_id: str
    snapshot_date: pd.Timestamp
    window_months: int
    first_commit: pd.Timestamp
    last_commit: pd.Timestamp
    total_commits: int
    pre_snapshot_commits: int
    post_snapshot_commits: int
    distinct_files_pre: int
    distinct_authors_pre: int
    eligible: bool
    exclusion_reason: str = ""

    def to_dict(self) -> dict:
        return {
            "project_id": self.project_id,
            "snapshot_date": self.snapshot_date,
            "window_months": self.window_months,
            "window_end": self.window_end,
            "first_commit": self.first_commit,
            "last_commit": self.last_commit,
            "total_commits": self.total_commits,
            "pre_snapshot_commits": self.pre_snapshot_commits,
            "post_snapshot_commits": self.post_snapshot_commits,
            "distinct_files_pre": self.distinct_files_pre,
            "distinct_authors_pre": self.distinct_authors_pre,
            "eligible": self.eligible,
            "exclusion_reason": self.exclusion_reason,
        }

    @property
    def window_end(self) -> pd.Timestamp:
        return self.snapshot_date + pd.DateOffset(months=self.window_months)


def _add_months(t: pd.Timestamp, months: int) -> pd.Timestamp:
    return t + pd.DateOffset(months=months)


def compute_project_snapshot(
    conn: sqlite3.Connection,
    project_id: str,
    strategy: str = SNAPSHOT_STRATEGY,
    window_months: int = OBSERVATION_WINDOW_MONTHS,
    min_pre: int = MIN_PRE_SNAPSHOT_COMMITS,
    min_post: int = MIN_POST_SNAPSHOT_COMMITS,
) -> ProjectSnapshot:
    """Compute the snapshot date for a single project.

    Parameters
    ----------
    conn :
        SQLite connection to the Technical Debt Dataset.
    project_id :
        PROJECT_ID value (e.g. ``'org.apache:batik'``).
    strategy :
        One of ``'median'`` (default), ``'fixed'`` (uses ``window_months`` back
        from the last commit), or ``'release'`` (not implemented yet).
    window_months :
        Length of the post-snapshot observation window.
    min_pre, min_post :
        Minimum number of pre- and post-snapshot commits required for a
        project to be considered eligible.
    """
    query = (
        "SELECT AUTHOR_DATE, COMMIT_HASH, AUTHOR "
        "FROM GIT_COMMITS "
        "WHERE PROJECT_ID = ? AND IN_MAIN_BRANCH = 'True' "
        "ORDER BY AUTHOR_DATE ASC"
    )
    commits = pd.read_sql_query(query, conn, params=[project_id])
    commits["AUTHOR_DATE"] = pd.to_datetime(commits["AUTHOR_DATE"], errors="coerce", utc=True)
    commits = commits.dropna(subset=["AUTHOR_DATE"])

    if len(commits) == 0:
        return ProjectSnapshot(
            project_id=project_id,
            snapshot_date=pd.NaT,
            window_months=window_months,
            first_commit=pd.NaT,
            last_commit=pd.NaT,
            total_commits=0,
            pre_snapshot_commits=0,
            post_snapshot_commits=0,
            distinct_files_pre=0,
            distinct_authors_pre=0,
            eligible=False,
            exclusion_reason="no_commits",
        )

    first = commits["AUTHOR_DATE"].min()
    last = commits["AUTHOR_DATE"].max()

    if strategy == "median":
        snapshot = commits["AUTHOR_DATE"].quantile(0.5, interpolation="nearest")
    elif strategy == "fixed":
        snapshot = _add_months(last, -window_months)
    elif strategy == "release":
        raise NotImplementedError("release-tag strategy not implemented yet")
    else:
        raise ValueError(f"Unknown snapshot strategy: {strategy}")

    snapshot = pd.Timestamp(snapshot)
    window_end = _add_months(snapshot, window_months)

    pre_mask = commits["AUTHOR_DATE"] <= snapshot
    post_mask = (commits["AUTHOR_DATE"] > snapshot) & (commits["AUTHOR_DATE"] <= window_end)
    pre_count = int(pre_mask.sum())
    post_count = int(post_mask.sum())

    # Distinct pre-snapshot files / authors for profiling
    if pre_count > 0:
        pre_hashes = commits.loc[pre_mask, "COMMIT_HASH"].tolist()
        if len(pre_hashes) > 0:
            placeholders = ",".join("?" * len(pre_hashes))
            file_q = (
                f"SELECT COUNT(DISTINCT FILE) AS n "
                f"FROM GIT_COMMITS_CHANGES "
                f"WHERE PROJECT_ID = ? AND COMMIT_HASH IN ({placeholders})"
            )
            distinct_files = pd.read_sql_query(file_q, conn, params=[project_id, *pre_hashes]).iloc[0]["n"]
        else:
            distinct_files = 0
        distinct_authors = int(commits.loc[pre_mask, "AUTHOR"].nunique())
    else:
        distinct_files = 0
        distinct_authors = 0

    eligible = pre_count >= min_pre and post_count >= min_post
    reason = ""
    if not eligible:
        reasons = []
        if pre_count < min_pre:
            reasons.append(f"pre<{min_pre}")
        if post_count < min_post:
            reasons.append(f"post<{min_post}")
        reason = ",".join(reasons)

    return ProjectSnapshot(
        project_id=project_id,
        snapshot_date=snapshot,
        window_months=window_months,
        first_commit=pd.Timestamp(first),
        last_commit=pd.Timestamp(last),
        total_commits=int(len(commits)),
        pre_snapshot_commits=pre_count,
        post_snapshot_commits=post_count,
        distinct_files_pre=int(distinct_files),
        distinct_authors_pre=int(distinct_authors),
        eligible=bool(eligible),
        exclusion_reason=reason,
    )


def compute_all_snapshots(
    conn: sqlite3.Connection,
    projects: Optional[list[str]] = None,
    strategy: str = SNAPSHOT_STRATEGY,
    window_months: int = OBSERVATION_WINDOW_MONTHS,
) -> pd.DataFrame:
    """Compute snapshots for every project in the dataset (or a given list)."""
    if projects is None:
        cur = conn.cursor()
        cur.execute("SELECT DISTINCT PROJECT_ID FROM GIT_COMMITS ORDER BY PROJECT_ID")
        projects = [r[0] for r in cur.fetchall()]

    rows: list[dict] = []
    for pid in projects:
        snap = compute_project_snapshot(conn, pid, strategy, window_months)
        rows.append(snap.to_dict())
    return pd.DataFrame(rows)


def load_snapshots(path: Path) -> pd.DataFrame:
    """Load previously computed snapshots from disk (Parquet)."""
    df = pd.read_parquet(path)
    for col in ("snapshot_date", "window_end", "first_commit", "last_commit"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    return df


"""
Stage 2 - Per-project profiling and snapshot-date selection.

For each project, compute:
- First / last master-branch commit date
- Total commit count (master branch)
- Snapshot date ``t`` (median of master-branch commit dates)
- Pre-snapshot commits (features are drawn from here)
- Post-snapshot commits within a 6-month window (labels are drawn from here)
- Distinct pre-snapshot files and authors
- Eligibility flag: requires >=500 pre-snapshot and >=100 post-snapshot commits

Outputs
-------
- ``data/processed/project_snapshots.parquet``: machine-readable snapshot metadata
- ``results/tables/project_stats.csv``: paper-ready descriptive-statistics table

Run
---
.. code-block:: bash

    .\\venv\\Scripts\\python.exe scripts/02_profile_projects.py
"""

import sys
import time
from pathlib import Path

import pandas as pd


def run_stage_2() -> None:
    t0 = time.time()
    print(f"[Stage 2] Strategy        : {SNAPSHOT_STRATEGY}")
    print(f"[Stage 2] Observation win : {OBSERVATION_WINDOW_MONTHS} months")

    with get_connection() as conn:
        print("[Stage 2] Computing snapshots for every project (master branch only) ...")
        df = compute_all_snapshots(conn)

    print(f"[Stage 2] Computed {len(df)} project profiles in {time.time() - t0:.1f}s")

    out_parquet = PROCESSED_DATA_DIR / "project_snapshots.parquet"
    out_csv = TABLES_DIR / "project_stats.csv"
    df.to_parquet(out_parquet, index=False)
    df.to_csv(out_csv, index=False)

    # Corpus summary: one row per eligible project for the thesis tables.
    corpus = df[df["eligible"]].copy()
    corpus_summary = pd.DataFrame(
        {
            "project_id": corpus["project_id"],
            "snapshot_date": corpus["snapshot_date"].dt.strftime("%Y-%m-%d"),
            "n_pre_commits": corpus["pre_snapshot_commits"].astype("int64"),
            "n_post_commits": corpus["post_snapshot_commits"].astype("int64"),
            "n_java_files": corpus["distinct_files_pre"].astype("int64"),
        }
    ).sort_values("project_id").reset_index(drop=True)
    corpus_summary.to_csv(TABLES_DIR / "corpus_summary.csv", index=False)

    eligible = df[df["eligible"]].copy()
    excluded = df[~df["eligible"]].copy()

    print(f"\n[Stage 2] Eligibility summary:")
    print(f"  Eligible projects : {len(eligible)} / {len(df)}")
    print(f"  Excluded projects : {len(excluded)}")
    if len(excluded):
        print("\n  Excluded details:")
        for _, r in excluded.iterrows():
            print(f"    - {r['project_id']:<35}  reason: {r['exclusion_reason']}")

    print(f"\n[Stage 2] Per-project profile (eligible, sorted by total_commits desc):")
    display_cols = [
        "project_id",
        "first_commit",
        "last_commit",
        "snapshot_date",
        "total_commits",
        "pre_snapshot_commits",
        "post_snapshot_commits",
        "distinct_files_pre",
        "distinct_authors_pre",
    ]
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 50, "display.width", 200):
        print(eligible[display_cols].sort_values("total_commits", ascending=False).to_string(index=False))

    print(f"\n[Stage 2] Saved: {out_parquet}")
    print(f"[Stage 2] Saved: {out_csv}")
    print(f"[Stage 2] Total elapsed : {time.time() - t0:.1f}s")
    print("[Stage 2] Complete.")


In [ ]:
run_stage_2()
print()
print('--- corpus_summary.csv (22 eligible) ---')
show_table('results/tables/corpus_summary.csv')


## Section 3 — Data Cleaning and Collision Resolution

Resolve basename path collisions and filter every table to clean
file-level instances. Writes the six `clean_*.parquet` files and
`collision_report.csv`.

In [ ]:
# Pipeline code

"""
Data cleaning and normalization for the Technical Debt Dataset v2.0.

This module converts raw SQLite tables into tidy pandas DataFrames suitable
for feature extraction (Stage 5), labeling (Stage 4) and cross-table joins
(Stage 6). The main transformations are:

1. **Path normalization** - ``SONAR_ISSUES.COMPONENT`` stores file identifiers
   as ``SonarProjectKey:path/to/File.java`` whereas ``GIT_COMMITS_CHANGES.FILE``
   stores them as plain repo-relative paths. We strip the prefix so the two
   tables join cleanly on ``(PROJECT_ID, file_path)``.
2. **Temporal enrichment** - ``SONAR_MEASURES`` rows carry an analysis key
   but not a date. We join ``SONAR_ANALYSIS`` to attach ``DATE`` and
   ``REVISION`` (Git hash of the analyzed commit) so measures can be
   temporally aligned to the snapshot ``t``.
3. **Bug-fix tagging** - regex-based flagging of commit messages plus a
   Jira-link flag derived from the pre-populated ``JIRA_ISSUES.HASH``
   column and Jira key mentions in commit messages.
4. **Scope filtering** - restricts to Java source files (excluding
   tests, generated sources, build artefacts) per ``config.py``.
5. **Project filtering** - keeps only projects flagged *eligible* by
   Stage 2.
6. **Type coercion** - parses dates to tz-aware ``datetime64[ns, UTC]``,
   casts numeric columns, removes obvious duplicates.

Each cleaned DataFrame is persisted as Parquet in ``data/processed/``.

References
----------
- Lenarduzzi, V., et al. (2019). The Technical Debt Dataset. PROMISE 2019.
- Mockus, A., Votta, L. (2000). Identifying reasons for software changes
  using historic databases. ICSM 2000 - bug-fix keyword regex basis.
"""

import re
import sqlite3
import sys
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd


_BUGFIX_REGEX = re.compile(BUGFIX_REGEX, re.IGNORECASE)
_JIRA_KEY_REGEX = re.compile(JIRA_ISSUE_KEY_PATTERN)


# ---------------------------------------------------------------------------
# Path normalization
# ---------------------------------------------------------------------------
def normalize_component_path(component: Optional[str]) -> Optional[str]:
    """Strip the ``SonarProjectKey:`` prefix from ``SONAR_ISSUES.COMPONENT``.

    Returns the repo-relative file path, e.g.
    ``"Apache_Cayenne:framework/.../Fault.java"`` -> ``"framework/.../Fault.java"``.
    Components without a colon are returned unchanged. ``None`` values are
    preserved.
    """
    if component is None or not isinstance(component, str):
        return component
    idx = component.find(":")
    if idx < 0:
        return component
    return component[idx + 1 :]


def is_java_source(path: Optional[str]) -> bool:
    """True if ``path`` is a non-test, non-generated Java source file."""
    if path is None or not isinstance(path, str) or not path:
        return False
    p = path.replace("\\", "/").lower()
    if not any(p.endswith(ext) for ext in SOURCE_FILE_EXTENSIONS):
        return False
    return not any(bad in p for bad in PATH_EXCLUSION_PATTERNS)


def extract_basename(path: Optional[str]) -> Optional[str]:
    """Return the final segment of a slash- or backslash-separated path.

    This is the canonical file-identifier key used across the pipeline because
    ``GIT_COMMITS_CHANGES.FILE`` in TD Dataset v2.0 stores only the basename
    for most projects (see RESEARCH_LOG.md 2026-04-23 path-format entry).
    """
    if path is None or not isinstance(path, str) or not path:
        return path
    p = path.replace("\\", "/")
    return p.rsplit("/", 1)[-1]


# ---------------------------------------------------------------------------
# Commit tagging
# ---------------------------------------------------------------------------
def flag_bugfix_commits(messages: pd.Series) -> pd.Series:
    """Return a boolean Series marking messages that look like bug-fixes.

    Uses the union regex of ``BUG_FIX_KEYWORDS`` (Mockus & Votta 2000).
    Empty / NaN messages return ``False``.
    """
    return messages.fillna("").astype(str).str.contains(_BUGFIX_REGEX, regex=True)


def extract_jira_keys(messages: pd.Series) -> pd.Series:
    """Return a Series of lists of Jira keys mentioned in each commit message."""

    def _find(msg: str) -> list[str]:
        if not isinstance(msg, str) or not msg:
            return []
        return [f"{m.group(1)}-{m.group(2)}" for m in _JIRA_KEY_REGEX.finditer(msg)]

    return messages.apply(_find)


# ---------------------------------------------------------------------------
# Per-table cleaners
# ---------------------------------------------------------------------------
def clean_git_commits(
    conn: sqlite3.Connection,
    projects: Iterable[str],
) -> pd.DataFrame:
    """Load master-branch commits for ``projects`` and add cleaning columns.

    Columns added:
    - ``AUTHOR_DATE`` parsed to UTC-aware datetime
    - ``is_bugfix`` (bool)
    - ``jira_keys`` (list[str])
    """
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, COMMIT_HASH, COMMIT_MESSAGE, AUTHOR, AUTHOR_DATE, "
        "       COMMITTER, COMMITTER_DATE, MERGE "
        "FROM GIT_COMMITS "
        f"WHERE IN_MAIN_BRANCH = 'True' AND PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df["AUTHOR_DATE"] = pd.to_datetime(df["AUTHOR_DATE"], errors="coerce", utc=True)
    df["COMMITTER_DATE"] = pd.to_datetime(df["COMMITTER_DATE"], errors="coerce", utc=True)
    df = df.dropna(subset=["AUTHOR_DATE", "COMMIT_HASH"]).copy()
    df["is_bugfix"] = flag_bugfix_commits(df["COMMIT_MESSAGE"])
    df["jira_keys"] = extract_jira_keys(df["COMMIT_MESSAGE"])
    df["MERGE"] = df["MERGE"].astype(str).str.lower().isin(("true", "1"))
    df = df.drop_duplicates(subset=["PROJECT_ID", "COMMIT_HASH"])
    return df.reset_index(drop=True)


def clean_git_commits_changes(
    conn: sqlite3.Connection,
    projects: Iterable[str],
    java_only: bool = True,
) -> pd.DataFrame:
    """Load per-file changes for ``projects``; optionally keep only Java sources.

    Output columns: ``PROJECT_ID``, ``COMMIT_HASH``, ``file_path`` (normalized),
    ``DATE`` (UTC), ``COMMITTER_ID``, ``LINES_ADDED``, ``LINES_REMOVED``.
    """
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, COMMIT_HASH, FILE, DATE, COMMITTER_ID, "
        "       LINES_ADDED, LINES_REMOVED "
        "FROM GIT_COMMITS_CHANGES "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df = df.rename(columns={"FILE": "file_path"})
    df["file_path"] = df["file_path"].astype(str).str.replace("\\", "/", regex=False)
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce", utc=True)
    for c in ("LINES_ADDED", "LINES_REMOVED"):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int64")
    df = df.dropna(subset=["file_path", "COMMIT_HASH", "DATE"])
    df = df[df["file_path"].str.len() > 0]
    if java_only:
        df = df[df["file_path"].apply(is_java_source)]
    df["basename"] = df["file_path"].apply(extract_basename)
    df = df.drop_duplicates(subset=["PROJECT_ID", "COMMIT_HASH", "file_path"])
    return df.reset_index(drop=True)


def clean_sonar_issues(
    conn: sqlite3.Connection,
    projects: Iterable[str],
    java_only: bool = True,
) -> pd.DataFrame:
    """Load SONAR_ISSUES, normalize COMPONENT to ``file_path``, coerce types."""
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, ISSUE_KEY, TYPE, RULE, SEVERITY, STATUS, RESOLUTION, "
        "       EFFORT, DEBT, CREATION_DATE, CLOSE_DATE, COMPONENT, "
        "       START_LINE, END_LINE "
        "FROM SONAR_ISSUES "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df["file_path"] = df["COMPONENT"].apply(normalize_component_path)
    df["file_path"] = df["file_path"].astype(str).str.replace("\\", "/", regex=False)
    df["CREATION_DATE"] = pd.to_datetime(df["CREATION_DATE"], errors="coerce", utc=True)
    df["CLOSE_DATE"] = pd.to_datetime(df["CLOSE_DATE"], errors="coerce", utc=True)
    for c in ("EFFORT", "DEBT", "START_LINE", "END_LINE"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["file_path", "ISSUE_KEY"])
    df = df[df["file_path"].str.len() > 0]
    if java_only:
        df = df[df["file_path"].apply(is_java_source)]
    df["basename"] = df["file_path"].apply(extract_basename)
    return df.drop(columns=["COMPONENT"]).reset_index(drop=True)


def clean_sonar_measures_with_dates(
    conn: sqlite3.Connection,
    projects: Iterable[str],
) -> pd.DataFrame:
    """Load SONAR_MEASURES and join SONAR_ANALYSIS to attach DATE + REVISION.

    Returns one row per ``(PROJECT_ID, ANALYSIS_KEY)`` with ``analysis_date``
    (UTC-aware) and the canonical subset of project-level metrics used as
    features (NCLOC, COMPLEXITY, SQALE_INDEX, ...). Leaky raw severity counts
    are kept here because they will be selectively dropped at dataset-build
    time depending on the label variant.
    """
    ids = ",".join(f"'{p}'" for p in projects)
    metric_cols = [
        "NCLOC",
        "LINES",
        "CLASSES",
        "FILES",
        "FUNCTIONS",
        "STATEMENTS",
        "COMPLEXITY",
        "COGNITIVE_COMPLEXITY",
        "FILE_COMPLEXITY",
        "FUNCTION_COMPLEXITY",
        "CLASS_COMPLEXITY",
        "COMMENT_LINES",
        "COMMENT_LINES_DENSITY",
        "DUPLICATED_LINES",
        "DUPLICATED_LINES_DENSITY",
        "DUPLICATED_BLOCKS",
        "DUPLICATED_FILES",
        "COVERAGE",
        "LINE_COVERAGE",
        "LINES_TO_COVER",
        "UNCOVERED_LINES",
        "VIOLATIONS",
        "BLOCKER_VIOLATIONS",
        "CRITICAL_VIOLATIONS",
        "MAJOR_VIOLATIONS",
        "MINOR_VIOLATIONS",
        "INFO_VIOLATIONS",
        "CODE_SMELLS",
        "BUGS",
        "VULNERABILITIES",
        "SQALE_INDEX",
        "SQALE_DEBT_RATIO",
        "SQALE_RATING",
        "RELIABILITY_RATING",
        "SECURITY_RATING",
        "RELIABILITY_REMEDIATION_EFFORT",
        "SECURITY_REMEDIATION_EFFORT",
        "OPEN_ISSUES",
    ]
    cols_sql = ", ".join(f"m.{c}" for c in metric_cols)
    sql = (
        f"SELECT m.PROJECT_ID, m.ANALYSIS_KEY, a.DATE AS analysis_date, "
        f"       a.REVISION AS analysis_revision, {cols_sql} "
        "FROM SONAR_MEASURES m "
        "LEFT JOIN SONAR_ANALYSIS a "
        "       ON a.PROJECT_ID = m.PROJECT_ID "
        "      AND a.ANALYSIS_KEY = m.ANALYSIS_KEY "
        f"WHERE m.PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df["analysis_date"] = pd.to_datetime(df["analysis_date"], errors="coerce", utc=True)
    for c in metric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["analysis_date"]).copy()
    df.columns = [c.lower() if c.isupper() else c for c in df.columns]
    return df.sort_values(["project_id", "analysis_date"]).reset_index(drop=True)


def clean_szz_with_dates(
    conn: sqlite3.Connection,
    projects: Iterable[str],
    git_commits_clean: pd.DataFrame,
) -> pd.DataFrame:
    """Load SZZ links and enrich with fault-fixing and fault-inducing dates.

    Joins to ``git_commits_clean`` on ``(PROJECT_ID, COMMIT_HASH)`` to attach
    ``AUTHOR_DATE`` for both sides of the pair.
    """
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, FAULT_FIXING_COMMIT_HASH, FAULT_INDUCING_COMMIT_HASH "
        "FROM SZZ_FAULT_INDUCING_COMMITS "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)

    commits_small = git_commits_clean[["PROJECT_ID", "COMMIT_HASH", "AUTHOR_DATE"]]
    df = df.merge(
        commits_small.rename(
            columns={"COMMIT_HASH": "FAULT_FIXING_COMMIT_HASH", "AUTHOR_DATE": "fix_date"}
        ),
        on=["PROJECT_ID", "FAULT_FIXING_COMMIT_HASH"],
        how="left",
    )
    df = df.merge(
        commits_small.rename(
            columns={"COMMIT_HASH": "FAULT_INDUCING_COMMIT_HASH", "AUTHOR_DATE": "induce_date"}
        ),
        on=["PROJECT_ID", "FAULT_INDUCING_COMMIT_HASH"],
        how="left",
    )
    return df.drop_duplicates().reset_index(drop=True)


def clean_jira_issues(
    conn: sqlite3.Connection,
    projects: Iterable[str],
) -> pd.DataFrame:
    """Load JIRA_ISSUES for projects with dates parsed and a ``is_bug`` flag."""
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, KEY, PRIORITY, TYPE, STATUS, RESOLUTION, "
        "       CREATION_DATE, RESOLUTION_DATE, UPDATE_DATE, HASH, COMMIT_DATE "
        "FROM JIRA_ISSUES "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    for c in ("CREATION_DATE", "RESOLUTION_DATE", "UPDATE_DATE", "COMMIT_DATE"):
        df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)
    df["is_bug"] = df["TYPE"].fillna("").str.lower().eq("bug")
    return df.reset_index(drop=True)


"""
Stage 3 - Clean, normalize and persist the raw dataset tables.

Pipeline:
  1. Resolve basename collisions in SONAR_ISSUES.COMPONENT (full paths)
     vs GIT_COMMITS_CHANGES.FILE (basenames in most projects) and write
     a per-project ``clean_basenames`` table that downstream stages use
     to filter rows to the unambiguous / resolved subset.
  2. Clean each raw table (paths normalised, dates parsed, Java-only).
  3. Filter every per-file frame to rows whose
     ``(project_id, basename)`` has ``is_clean == 1``.
  4. Persist cleaned parquets and the collision report.

Outputs
-------
- ``data/processed/clean_basenames.parquet``
    (project_id, basename, is_clean, resolution_status, resolved_full_path)
- ``data/processed/collision_report.csv``
    (project_id, n_total, n_kept, n_dropped, drop_rate_pct)
- ``data/processed/clean_git_commits.parquet``
- ``data/processed/clean_git_commits_changes.parquet``
- ``data/processed/clean_sonar_issues.parquet``
- ``data/processed/clean_sonar_measures.parquet``
- ``data/processed/clean_szz.parquet``
- ``data/processed/clean_jira_issues.parquet``
"""

import sys
import time
from pathlib import Path
from typing import List, Optional, Tuple

import pandas as pd


def _fmt_rows(n: int) -> str:
    return f"{n:>12,}"


# ---------------------------------------------------------------------------
# Basename collision resolution
# ---------------------------------------------------------------------------
def _resolve_basename(paths: List[str]) -> Tuple[Optional[str], str]:
    """Pick a canonical full_path for an ambiguous basename, or drop it.

    Rules (in order):
      1. Unique ``src/main`` non-test path -> resolved_main
      2. Multiple ``src/main`` paths       -> shortest one (resolved_shortest)
      3. Unique non-test path              -> resolved_nontest
      4. Otherwise                         -> dropped
    """
    main = [p for p in paths if "src/main" in p and "test" not in p.lower()]
    if len(main) == 1:
        return main[0], "resolved_main"
    if len(main) > 1:
        return min(main, key=lambda p: p.count("/")), "resolved_shortest"
    non_test = [p for p in paths if "test" not in p.lower()]
    if len(non_test) == 1:
        return non_test[0], "resolved_nontest"
    return None, "dropped"


def build_clean_basenames(conn) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build the (basename -> canonical path) map and a per-project report.

    Reads SONAR_ISSUES directly (only table that carries full paths),
    derives basename, counts paths per basename, and applies the
    resolution rules above. Java-source filtering is applied so we
    never adjudicate test files.

    Returns
    -------
    clean : DataFrame with columns
        (project_id, basename, is_clean, resolution_status, resolved_full_path)
    report : DataFrame with columns
        (project_id, n_total, n_kept, n_dropped, drop_rate_pct)
    """
    sonar_paths = pd.read_sql(
        """
        SELECT PROJECT_ID  AS project_id,
               COMPONENT   AS raw_component
          FROM SONAR_ISSUES
         GROUP BY PROJECT_ID, COMPONENT
        """,
        conn,
    )
    sonar_paths["full_path"] = sonar_paths["raw_component"].apply(normalize_component_path)
    sonar_paths["full_path"] = (
        sonar_paths["full_path"].astype(str).str.replace("\\", "/", regex=False)
    )
    sonar_paths = sonar_paths[sonar_paths["full_path"].apply(is_java_source)].copy()
    sonar_paths["basename"] = sonar_paths["full_path"].apply(extract_basename)
    sonar_paths = sonar_paths.dropna(subset=["basename"]).drop(columns=["raw_component"])

    grouped = (
        sonar_paths.groupby(["project_id", "basename"])["full_path"]
        .apply(lambda s: list(dict.fromkeys(s)))  # deduped, order-preserving
        .reset_index(name="paths")
    )
    grouped["n_paths"] = grouped["paths"].map(len)

    statuses: List[str] = []
    resolved_paths: List[Optional[str]] = []
    for paths in grouped["paths"]:
        if len(paths) == 1:
            statuses.append("unambiguous")
            resolved_paths.append(paths[0])
        else:
            chosen, status = _resolve_basename(paths)
            statuses.append(status)
            resolved_paths.append(chosen)
    grouped["resolution_status"] = statuses
    grouped["resolved_full_path"] = resolved_paths
    grouped["is_clean"] = (grouped["resolution_status"] != "dropped").astype("int64")

    clean = grouped[
        ["project_id", "basename", "is_clean", "resolution_status", "resolved_full_path"]
    ].copy()

    report_rows = []
    for pid, sub in clean.groupby("project_id"):
        n_total = int(len(sub))
        n_kept = int(sub["is_clean"].sum())
        n_dropped = n_total - n_kept
        report_rows.append(
            {
                "project_id": pid,
                "n_total": n_total,
                "n_kept": n_kept,
                "n_dropped": n_dropped,
                "drop_rate_pct": round(100 * n_dropped / n_total, 2) if n_total else 0.0,
            }
        )
    report = pd.DataFrame(report_rows).sort_values("project_id").reset_index(drop=True)
    return clean, report


def _filter_to_clean(
    df: pd.DataFrame,
    clean_basenames: pd.DataFrame,
    pid_col: str = "PROJECT_ID",
) -> pd.DataFrame:
    """Keep rows whose ``(project_id, basename)`` has ``is_clean == 1``.

    ``df`` is expected to carry both the uppercase project key (``PROJECT_ID``
    in cleaned tables) and a ``basename`` column.
    """
    if "basename" not in df.columns:
        raise KeyError(f"_filter_to_clean: 'basename' column missing in frame with cols {list(df.columns)[:8]}")
    keep = clean_basenames[clean_basenames["is_clean"] == 1][["project_id", "basename"]]
    keep = keep.rename(columns={"project_id": pid_col})
    return df.merge(keep, on=[pid_col, "basename"], how="inner").reset_index(drop=True)


# ---------------------------------------------------------------------------
# Stage driver
# ---------------------------------------------------------------------------
def run_stage_3() -> None:
    t0 = time.time()
    snap_path = PROCESSED_DATA_DIR / "project_snapshots.parquet"
    if not snap_path.exists():
        raise FileNotFoundError(
            f"Missing {snap_path}. Run scripts/02_profile_projects.py first."
        )

    snaps = load_snapshots(snap_path)
    eligible = snaps[snaps["eligible"]].copy()
    projects = eligible["project_id"].tolist()
    print(f"[Stage 3] Eligible projects    : {len(projects)}")

    with get_connection() as conn:
        # --- Step 1: basename collision resolution -------------------------
        print("[Stage 3] (1/7) Resolving basename collisions ...", end=" ", flush=True)
        t = time.time()
        clean_basenames, collision_report = build_clean_basenames(conn)
        # Restrict to eligible projects only (the SQL pulled all projects).
        clean_basenames = clean_basenames[clean_basenames["project_id"].isin(projects)].copy()
        collision_report = collision_report[
            collision_report["project_id"].isin(projects)
        ].reset_index(drop=True)
        print(
            f"{_fmt_rows(len(clean_basenames))} basenames "
            f"(kept={int(clean_basenames['is_clean'].sum()):,}, "
            f"dropped={int((1 - clean_basenames['is_clean']).sum()):,})  "
            f"({time.time() - t:.1f}s)"
        )

        print("[Stage 3] (2/7) GIT_COMMITS ...", end=" ", flush=True)
        t = time.time()
        gc = clean_git_commits(conn, projects)
        print(f"{_fmt_rows(len(gc))} rows  ({time.time()-t:.1f}s)")

        print("[Stage 3] (3/7) GIT_COMMITS_CHANGES (Java only) ...", end=" ", flush=True)
        t = time.time()
        gcc = clean_git_commits_changes(conn, projects, java_only=True)
        print(f"{_fmt_rows(len(gcc))} rows  ({time.time()-t:.1f}s)")

        print("[Stage 3] (4/7) SONAR_ISSUES (Java only) ...", end=" ", flush=True)
        t = time.time()
        si = clean_sonar_issues(conn, projects, java_only=True)
        print(f"{_fmt_rows(len(si))} rows  ({time.time()-t:.1f}s)")

        print("[Stage 3] (5/7) SONAR_MEASURES (with dates) ...", end=" ", flush=True)
        t = time.time()
        sm = clean_sonar_measures_with_dates(conn, projects)
        print(f"{_fmt_rows(len(sm))} rows  ({time.time()-t:.1f}s)")

        print("[Stage 3] (6/7) SZZ (with dates) ...", end=" ", flush=True)
        t = time.time()
        szz = clean_szz_with_dates(conn, projects, gc)
        print(f"{_fmt_rows(len(szz))} rows  ({time.time()-t:.1f}s)")

        print("[Stage 3] (7/7) JIRA_ISSUES ...", end=" ", flush=True)
        t = time.time()
        ji = clean_jira_issues(conn, projects)
        print(f"{_fmt_rows(len(ji))} rows  ({time.time()-t:.1f}s)")

    # --- Filter to is_clean == 1 for per-file frames -----------------------
    print("[Stage 3] Filtering per-file frames to resolved basenames ...")
    gcc_pre = len(gcc)
    si_pre = len(si)
    gcc = _filter_to_clean(gcc, clean_basenames)
    si = _filter_to_clean(si, clean_basenames)
    print(
        f"   git_commits_changes : {gcc_pre:>10,} -> {len(gcc):>10,} "
        f"({100*(gcc_pre-len(gcc))/max(gcc_pre,1):.1f}% removed)"
    )
    print(
        f"   sonar_issues        : {si_pre:>10,} -> {len(si):>10,} "
        f"({100*(si_pre-len(si))/max(si_pre,1):.1f}% removed)"
    )

    # --- Write outputs -----------------------------------------------------
    print("[Stage 3] Writing parquet outputs ...")
    clean_basenames.to_parquet(PROCESSED_DATA_DIR / "clean_basenames.parquet", index=False)
    collision_report.to_csv(PROCESSED_DATA_DIR / "collision_report.csv", index=False)
    gc.to_parquet(PROCESSED_DATA_DIR / "clean_git_commits.parquet", index=False)
    gcc.to_parquet(PROCESSED_DATA_DIR / "clean_git_commits_changes.parquet", index=False)
    si.to_parquet(PROCESSED_DATA_DIR / "clean_sonar_issues.parquet", index=False)
    sm.to_parquet(PROCESSED_DATA_DIR / "clean_sonar_measures.parquet", index=False)
    szz.to_parquet(PROCESSED_DATA_DIR / "clean_szz.parquet", index=False)
    ji.to_parquet(PROCESSED_DATA_DIR / "clean_jira_issues.parquet", index=False)

    print("\n[Stage 3] Collision report (data/processed/collision_report.csv):")
    with pd.option_context("display.width", 200, "display.max_rows", None):
        print(collision_report.to_string(index=False))

    median_drop = collision_report["drop_rate_pct"].median()
    max_drop = collision_report["drop_rate_pct"].max()
    print(f"\n[Stage 3] Basename drop rate: median={median_drop:.1f}%  max={max_drop:.1f}%")
    if not (0 <= median_drop <= 25):
        print(f"[Stage 3] WARNING: median drop rate {median_drop:.1f}% outside expected 5-15% band.")

    print(f"\n[Stage 3] Total elapsed : {time.time() - t0:.1f}s")
    print("[Stage 3] Complete.")


In [ ]:
run_stage_3()
print()
print('--- collision_report.csv ---')
df = pd.read_csv('data/processed/collision_report.csv')
display(df.style.background_gradient(subset=['drop_rate_pct'], cmap='Reds'))


## Section 4 — High-Risk TD Labeling

Compute the combined-weight high-risk TD label using six binary signals
(three static SonarQube signals + three git-history signals). Empirical
weights derived via point-biserial correlation; falls back to theoretical
Kamei 2013 weights if within 0.05. Writes `labels.parquet`,
`derived_weights.json`, `label_summary.csv`.

In [ ]:
# Pipeline code

"""
SZZ, bug-fix and Jira resolution helpers operating on cleaned parquets.

These helpers bridge commit-level tables (GIT_COMMITS, GIT_COMMITS_CHANGES,
SZZ_FAULT_INDUCING_COMMITS, JIRA_ISSUES) and the basename-level units of
analysis used throughout the pipeline.

Key operations
--------------
- ``bugfix_touches_in_window`` - for a project, a snapshot ``t`` and a
  window length ``W``, return a DataFrame of ``(basename,
  n_bugfix_commits, churn_add, churn_removed)`` covering bug-fix commits
  whose ``AUTHOR_DATE`` falls in ``(t, t+W]``.
- ``churn_in_window`` - per-basename churn (sum of LINES_ADDED +
  LINES_REMOVED) in the post-snapshot window, irrespective of bug-fix
  flag. Used as the "future_churn" component of the consequence score.
- ``szz_events_in_window`` - per-basename count of SZZ fault-fixing
  commits in ``(t, t+W]`` that touched the basename (i.e. the fix
  modified the file). Used as the "szz_defects_future" component.
- ``jira_bug_commits_in_window`` - commits linked to Jira ``Bug`` issues
  that close inside the observation window. Used as a stricter bug-fix
  signal (sensitivity analysis; not in primary risk score to keep the
  score fully reproducible from the DB).

Everything is indexed by ``(project_id, basename)`` to match the chosen
unit of analysis (see RESEARCH_LOG.md 2026-04-23 entry).
"""

from typing import Optional

import pandas as pd


def _window_mask(dates: pd.Series, t: pd.Timestamp, t_end: pd.Timestamp) -> pd.Series:
    """Boolean mask for ``t < dates <= t_end`` with NaT-safe handling."""
    if dates.dtype.kind != "M":
        dates = pd.to_datetime(dates, errors="coerce", utc=True)
    return (dates > t) & (dates <= t_end)


def _ensure_utc(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    if ts.tz is None:
        ts = ts.tz_localize("UTC")
    return ts


def bugfix_touches_in_window(
    commits: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Bug-fix commit count and churn per basename in ``(t, t+W]``.

    Parameters
    ----------
    commits :
        ``clean_git_commits`` DataFrame with an ``is_bugfix`` column.
    changes :
        ``clean_git_commits_changes`` DataFrame with ``basename``.
    project_id :
        Which project to compute for.
    t :
        Snapshot timestamp (UTC-aware).
    window_months :
        Observation window length.

    Returns
    -------
    DataFrame with columns ``basename``, ``n_bugfix_commits_future``,
    ``bugfix_churn_future`` (added + removed lines in bug-fix commits).
    """
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    c = commits[commits["PROJECT_ID"] == project_id].copy()
    c = c[_window_mask(c["AUTHOR_DATE"], t, t_end) & c["is_bugfix"].fillna(False)]
    if c.empty:
        return pd.DataFrame(columns=["basename", "n_bugfix_commits_future", "bugfix_churn_future"])

    ch = changes[changes["PROJECT_ID"] == project_id].copy()
    ch = ch[ch["COMMIT_HASH"].isin(c["COMMIT_HASH"])]

    if ch.empty:
        return pd.DataFrame(columns=["basename", "n_bugfix_commits_future", "bugfix_churn_future"])

    ch = ch.assign(_row_churn=ch["LINES_ADDED"].astype("int64") + ch["LINES_REMOVED"].astype("int64"))
    agg = (
        ch.groupby("basename")
        .agg(
            n_bugfix_commits_future=("COMMIT_HASH", "nunique"),
            bugfix_churn_future=("_row_churn", "sum"),
        )
        .reset_index()
    )
    agg["bugfix_churn_future"] = agg["bugfix_churn_future"].astype("int64")
    return agg


def churn_in_window(
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Total churn per basename in ``(t, t+W]`` (all commits, not only bug-fixes)."""
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    ch = changes[changes["PROJECT_ID"] == project_id].copy()
    ch = ch[_window_mask(ch["DATE"], t, t_end)]
    if ch.empty:
        return pd.DataFrame(
            columns=["basename", "future_churn", "future_add", "future_removed", "future_commits"]
        )

    agg = (
        ch.groupby("basename")
        .agg(
            future_add=("LINES_ADDED", "sum"),
            future_removed=("LINES_REMOVED", "sum"),
            future_commits=("COMMIT_HASH", "nunique"),
        )
        .reset_index()
    )
    agg["future_churn"] = (agg["future_add"] + agg["future_removed"]).astype("int64")
    return agg[["basename", "future_churn", "future_add", "future_removed", "future_commits"]]


def szz_events_in_window(
    szz: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Per-basename count of SZZ fault-fixing commits in ``(t, t+W]`` touching it.

    Uses the ``fix_date`` attached in ``clean_szz_with_dates``. The file
    resolution is done by joining SZZ's ``FAULT_FIXING_COMMIT_HASH`` to
    ``GIT_COMMITS_CHANGES.COMMIT_HASH`` to find touched basenames.
    """
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    s = szz[szz["PROJECT_ID"] == project_id].copy()
    s = s[_window_mask(s["fix_date"], t, t_end)]
    if s.empty:
        return pd.DataFrame(columns=["basename", "n_szz_fixes_future", "n_szz_inducing_past"])

    ch = changes[changes["PROJECT_ID"] == project_id]

    # (1) Basenames touched by fault-fixing commits in window
    fix_hashes = set(s["FAULT_FIXING_COMMIT_HASH"].dropna().unique())
    fix_touches = ch[ch["COMMIT_HASH"].isin(fix_hashes)][["basename", "COMMIT_HASH"]]
    fix_agg = (
        fix_touches.groupby("basename")["COMMIT_HASH"]
        .nunique()
        .rename("n_szz_fixes_future")
        .reset_index()
    )

    # (2) Basenames that were originally modified by fault-inducing commits
    # before the snapshot (these are the "ticking-time-bomb" files)
    induce = s[s["induce_date"].notna() & (s["induce_date"] <= t)]
    induce_hashes = set(induce["FAULT_INDUCING_COMMIT_HASH"].dropna().unique())
    induce_touches = ch[ch["COMMIT_HASH"].isin(induce_hashes)][["basename", "COMMIT_HASH"]]
    induce_agg = (
        induce_touches.groupby("basename")["COMMIT_HASH"]
        .nunique()
        .rename("n_szz_inducing_past")
        .reset_index()
    )

    out = pd.merge(fix_agg, induce_agg, on="basename", how="outer").fillna(0)
    for c in ("n_szz_fixes_future", "n_szz_inducing_past"):
        out[c] = out[c].astype("int64")
    return out


def jira_bug_commits_in_window(
    commits: pd.DataFrame,
    jira: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Per-basename count of commits that close a Jira *Bug* in ``(t, t+W]``.

    Uses the pre-populated ``JIRA_ISSUES.HASH`` column to link tickets to
    commits directly; supplements by scanning for Jira keys in commit
    messages of bug-fix commits. Returned as a separate diagnostic
    feature - not part of the primary risk score to keep the score
    computable for projects that lack rich Jira linkage.
    """
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    j = jira[jira["PROJECT_ID"] == project_id].copy()
    j = j[j["is_bug"]]
    bug_keys = set(j["KEY"].dropna().unique())
    hash_links = set(j["HASH"].dropna().unique())

    c = commits[commits["PROJECT_ID"] == project_id].copy()
    c = c[_window_mask(c["AUTHOR_DATE"], t, t_end)]
    if c.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bug_commits_future"])

    # Commits that either link to a bug ticket via hash OR mention a known bug key
    c["mentions_bug_key"] = c["jira_keys"].apply(
        lambda keys: any(k in bug_keys for k in keys) if isinstance(keys, list) else False
    )
    c["hash_in_bug_links"] = c["COMMIT_HASH"].isin(hash_links)
    c = c[c["mentions_bug_key"] | c["hash_in_bug_links"]]
    if c.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bug_commits_future"])

    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["COMMIT_HASH"].isin(c["COMMIT_HASH"])]
    if ch.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bug_commits_future"])
    agg = (
        ch.groupby("basename")["COMMIT_HASH"]
        .nunique()
        .rename("n_jira_bug_commits_future")
        .reset_index()
    )
    return agg


def basename_universe_at_snapshot(
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
) -> pd.DataFrame:
    """Return the set of basenames that exist in ``project_id`` at time ``t``.

    A basename "exists at t" if any change with ``DATE <= t`` touched it.
    Output columns: ``project_id``, ``basename``.
    """
    t = _ensure_utc(t)
    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["DATE"] <= t]
    universe = ch[["basename"]].drop_duplicates().copy()
    universe["project_id"] = project_id
    return universe[["project_id", "basename"]].reset_index(drop=True)


def open_issues_at_snapshot(
    issues: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
) -> pd.DataFrame:
    """Filter ``clean_sonar_issues`` to issues open at time ``t``.

    An issue is "open at ``t``" iff ``CREATION_DATE <= t`` and
    (``CLOSE_DATE`` is NaT or ``CLOSE_DATE > t``).
    """
    t = _ensure_utc(t)
    df = issues[issues["PROJECT_ID"] == project_id]
    creation_ok = df["CREATION_DATE"] <= t
    close_ok = df["CLOSE_DATE"].isna() | (df["CLOSE_DATE"] > t)
    return df[creation_ok & close_ok].copy()


"""
Dual-signal combined-weight binary label for technical debt.

Binary output: ``is_high_risk`` in {0, 1}. Built from six binary signals
spanning two independent sources:

  Static (SonarQube, at snapshot t):
    S1_severity     - file has >= 1 OPEN BLOCKER/CRITICAL issue
    S2_debt         - total SonarQube DEBT minutes > project median
    S3_smells       - count of CODE_SMELL issues > project median

  History (Git, AUTHOR_DATE <= t):
    S4_bugfix       - bug-fix commit count > project median
    S5_churn        - lifetime churn > project 75th percentile
    S6_contributors - distinct authors > project median

A weighted sum ``risk_score = sum(w_i * S_i)`` in [0, 1] is thresholded
at 0.50 (relaxed to 0.45 / 0.40 per project if positives < 5).

Weights are derived empirically via point-biserial correlation between
each signal and a 6-month post-snapshot bug-fix surrogate (Kamei TSE
2013 weighting approach). If the empirical weights are within 0.05 of
the theoretically motivated baseline, the theoretical weights are used
and the empirical run is logged as confirmation. The surrogate is used
only for weight derivation and is never persisted alongside features.
"""

import re
from typing import Dict, Iterable, Tuple

import numpy as np
import pandas as pd
from scipy.stats import pointbiserialr


_BUGFIX_RE = re.compile(BUGFIX_REGEX, re.IGNORECASE)

SIGNAL_COLUMNS: Tuple[str, ...] = (
    "S1_severity",
    "S2_debt",
    "S3_smells",
    "S4_bugfix",
    "S5_churn",
    "S6_contributors",
)

# All six signals use p75 as the within-project
# threshold. This ensures only genuinely elevated
# files are flagged (top 25% on each dimension).
# Using the median would flag ~50% of files per
# signal by construction, which is not a useful
# high-risk indicator. p75 alignment follows the
# design rationale of S5 and is consistent with
# percentile-based prioritization in Kamei 2013.
# (S1 is a boolean presence flag for BLOCKER/CRITICAL
# issues - not percentile-based by construction.)


# ---------------------------------------------------------------------------
# Six per-file binary signals at snapshot t
# ---------------------------------------------------------------------------
def _project_basenames(
    sonar_issues_p: pd.DataFrame,
    changes_p: pd.DataFrame,
) -> pd.Index:
    """Union of basenames seen in either Sonar issues or Git changes for one project."""
    sonar_b = sonar_issues_p["basename"].dropna().unique() if "basename" in sonar_issues_p else []
    git_b = changes_p["basename"].dropna().unique() if "basename" in changes_p else []
    return pd.Index(sorted(set(sonar_b) | set(git_b)), name="basename")


def _static_signals(
    sonar_issues_p: pd.DataFrame,
    t: pd.Timestamp,
    basenames: pd.Index,
) -> pd.DataFrame:
    """Compute S1, S2, S3 over basenames using SONAR_ISSUES rows with CREATION_DATE <= t."""
    df = sonar_issues_p.copy()
    if "CREATION_DATE" in df.columns:
        df = df[df["CREATION_DATE"] <= t]
    sev = df["SEVERITY"].astype(str).str.upper()
    status = df["STATUS"].astype(str).str.upper()
    issue_type = df["TYPE"].astype(str).str.upper()

    # S1 - open BLOCKER/CRITICAL count
    sev_mask = sev.isin(SEVERITY_LABEL_LEVELS) & (status == "OPEN")
    n_sev = df.loc[sev_mask].groupby("basename").size()

    # S2 - total debt
    debt = df.groupby("basename")["DEBT"].sum(min_count=1).fillna(0.0)

    # S3 - smell count
    smell_mask = issue_type == "CODE_SMELL"
    n_smells = df.loc[smell_mask].groupby("basename").size()

    out = pd.DataFrame(index=basenames)
    out["n_severity"] = n_sev.reindex(basenames).fillna(0).astype("int64")
    out["total_debt"] = debt.reindex(basenames).fillna(0.0).astype("float64")
    out["n_smells"] = n_smells.reindex(basenames).fillna(0).astype("int64")

    out["S1_severity"] = (out["n_severity"] >= 1).astype("int64")

    debt_p75 = out["total_debt"].quantile(0.75)
    smell_p75 = out["n_smells"].quantile(0.75)
    out["S2_debt"] = (out["total_debt"] > debt_p75).astype("int64")
    out["S3_smells"] = (out["n_smells"] > smell_p75).astype("int64")
    return out


def _history_signals(
    commits_p: pd.DataFrame,
    changes_p: pd.DataFrame,
    t: pd.Timestamp,
    basenames: pd.Index,
) -> pd.DataFrame:
    """Compute S4, S5, S6 over basenames using commits/changes with AUTHOR_DATE/DATE <= t."""
    if "AUTHOR_DATE" in commits_p.columns:
        commits_pre = commits_p[commits_p["AUTHOR_DATE"] <= t]
    else:
        commits_pre = commits_p.iloc[0:0]

    if "DATE" in changes_p.columns:
        changes_pre = changes_p[changes_p["DATE"] <= t]
    else:
        changes_pre = changes_p.iloc[0:0]

    commit_meta = commits_pre[["COMMIT_HASH", "AUTHOR"]].copy()
    if "is_bugfix" in commits_pre.columns:
        commit_meta["is_bugfix"] = commits_pre["is_bugfix"].astype(bool).values
    else:
        msgs = commits_pre.get("COMMIT_MESSAGE", pd.Series([""] * len(commits_pre)))
        commit_meta["is_bugfix"] = msgs.fillna("").astype(str).str.contains(_BUGFIX_RE, regex=True)

    file_commit_pairs = changes_pre[
        ["COMMIT_HASH", "basename", "LINES_ADDED", "LINES_REMOVED"]
    ].merge(commit_meta, on="COMMIT_HASH", how="left")
    file_commit_pairs["is_bugfix"] = file_commit_pairs["is_bugfix"].fillna(False).astype(bool)

    # S4 - bug-fix commit count per basename
    bf = (
        file_commit_pairs[file_commit_pairs["is_bugfix"]]
        .drop_duplicates(["basename", "COMMIT_HASH"])
        .groupby("basename")
        .size()
    )

    # S5 - total churn (added + removed) per basename
    file_commit_pairs["churn"] = (
        file_commit_pairs["LINES_ADDED"].fillna(0).astype("int64")
        + file_commit_pairs["LINES_REMOVED"].fillna(0).astype("int64")
    )
    churn = file_commit_pairs.groupby("basename")["churn"].sum()

    # S6 - distinct authors per basename
    authors = (
        file_commit_pairs.dropna(subset=["AUTHOR"])
        .groupby("basename")["AUTHOR"]
        .nunique()
    )

    out = pd.DataFrame(index=basenames)
    out["n_bugfix"] = bf.reindex(basenames).fillna(0).astype("int64")
    out["total_churn"] = churn.reindex(basenames).fillna(0).astype("int64")
    out["n_authors"] = authors.reindex(basenames).fillna(0).astype("int64")

    bf_p75 = out["n_bugfix"].quantile(0.75)
    churn_p75 = out["total_churn"].quantile(0.75)
    auth_p75 = out["n_authors"].quantile(0.75)

    out["S4_bugfix"] = (out["n_bugfix"] > bf_p75).astype("int64")
    out["S5_churn"] = (out["total_churn"] > churn_p75).astype("int64")
    out["S6_contributors"] = (out["n_authors"] > auth_p75).astype("int64")
    return out


def _future_bugfix_count(
    commits_p: pd.DataFrame,
    changes_p: pd.DataFrame,
    t: pd.Timestamp,
    window_months: int,
    basenames: pd.Index,
) -> pd.Series:
    """Surrogate: bug-fix commits per basename in (t, t + window]. Weight-derivation only."""
    if "AUTHOR_DATE" not in commits_p.columns or "DATE" not in changes_p.columns:
        return pd.Series(0, index=basenames, dtype="int64", name="future_bugfix_count")

    t_end = t + pd.DateOffset(months=window_months)
    post_commits = commits_p[(commits_p["AUTHOR_DATE"] > t) & (commits_p["AUTHOR_DATE"] <= t_end)]
    if "is_bugfix" in post_commits.columns:
        bf_hashes = post_commits.loc[post_commits["is_bugfix"].astype(bool), "COMMIT_HASH"]
    else:
        msgs = post_commits.get("COMMIT_MESSAGE", pd.Series([""] * len(post_commits)))
        is_bf = msgs.fillna("").astype(str).str.contains(_BUGFIX_RE, regex=True)
        bf_hashes = post_commits.loc[is_bf, "COMMIT_HASH"]

    if len(bf_hashes) == 0:
        return pd.Series(0, index=basenames, dtype="int64", name="future_bugfix_count")

    post_changes = changes_p[
        (changes_p["DATE"] > t)
        & (changes_p["DATE"] <= t_end)
        & changes_p["COMMIT_HASH"].isin(bf_hashes)
    ]
    counts = (
        post_changes.drop_duplicates(["basename", "COMMIT_HASH"])
        .groupby("basename")
        .size()
    )
    return counts.reindex(basenames).fillna(0).astype("int64").rename("future_bugfix_count")


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------
def compute_dual_signal_signals(
    project_id: str,
    t: pd.Timestamp,
    commits: pd.DataFrame,
    changes: pd.DataFrame,
    sonar_issues: pd.DataFrame,
    surrogate_window_months: int = LABEL_SURROGATE_WINDOW_MONTHS,
) -> pd.DataFrame:
    """Return per-basename signals S1..S6 plus the weight-derivation surrogate.

    The surrogate column ``future_bugfix_count`` is returned alongside S1..S6
    so the caller can stack frames across projects to fit the empirical
    weights. It must be dropped before merging with the feature matrix
    (see assertions in scripts/06_build_dataset.py).
    """
    cp = commits[commits["PROJECT_ID"] == project_id] if "PROJECT_ID" in commits.columns else commits
    ch = changes[changes["PROJECT_ID"] == project_id] if "PROJECT_ID" in changes.columns else changes
    si = (
        sonar_issues[sonar_issues["PROJECT_ID"] == project_id]
        if "PROJECT_ID" in sonar_issues.columns
        else sonar_issues
    )

    basenames = _project_basenames(si, ch)
    if len(basenames) == 0:
        cols = ["project_id", "basename", *SIGNAL_COLUMNS, "future_bugfix_count"]
        return pd.DataFrame({c: pd.Series(dtype="int64") for c in cols})

    static_df = _static_signals(si, t, basenames)
    history_df = _history_signals(cp, ch, t, basenames)
    surrogate = _future_bugfix_count(cp, ch, t, surrogate_window_months, basenames)

    out = pd.DataFrame(index=basenames)
    for col in SIGNAL_COLUMNS:
        out[col] = static_df[col] if col in static_df.columns else history_df[col]
    out["future_bugfix_count"] = surrogate
    out.insert(0, "project_id", project_id)
    out.index.name = "basename"
    return out.reset_index()


def derive_weights_empirically(
    df_with_signals: pd.DataFrame,
    df_with_surrogate: pd.DataFrame,
) -> Dict[str, float]:
    """Empirical weights via |point-biserial correlation| with future bug-fix count.

    Both inputs must align row-by-row on ``(project_id, basename)``. Each
    signal's |r| is floored at 0.01, normalised to sum to 1.0, rounded to
    two decimals; rounding drift is absorbed by the largest-weight signal.
    """
    if len(df_with_signals) != len(df_with_surrogate):
        raise ValueError("signals and surrogate frames must have the same length")
    y = df_with_surrogate["future_bugfix_count"].astype(float).values
    corrs: Dict[str, float] = {}
    for s in SIGNAL_COLUMNS:
        x = df_with_signals[s].astype(float).values
        if np.var(x) == 0 or np.var(y) == 0:
            r = 0.0
        else:
            r, _ = pointbiserialr(x, y)
        corrs[s] = max(abs(float(r)), 0.01)

    total = sum(corrs.values())
    weights = {k: round(v / total, 2) for k, v in corrs.items()}
    drift = 1.0 - sum(weights.values())
    top = max(weights, key=weights.get)
    weights[top] = round(weights[top] + drift, 2)
    return weights


def choose_weights(
    derived: Dict[str, float],
    theoretical: Dict[str, float] = THEORETICAL_WEIGHTS,
    tolerance: float = 0.05,
) -> Tuple[Dict[str, float], str]:
    """Pick theoretical weights if every signal is within ``tolerance``, else derived."""
    max_diff = max(abs(derived[s] - theoretical[s]) for s in SIGNAL_COLUMNS)
    if max_diff <= tolerance:
        return dict(theoretical), "theoretical"
    return dict(derived), "empirical"


def apply_label_thresholding(
    risk_scores: pd.Series,
    min_positives: int = LABEL_MIN_POSITIVES_PER_PROJECT,
    primary_threshold: float = LABEL_RISK_THRESHOLD,
    fallback_thresholds: Iterable[float] = LABEL_THRESHOLD_FALLBACKS,
) -> Tuple[pd.Series, float, bool]:
    """Return (is_high_risk, threshold_used, satisfied_min_positives)."""
    thresholds = [primary_threshold, *fallback_thresholds]
    for thr in thresholds:
        labels = (risk_scores >= thr).astype("int64")
        if int(labels.sum()) >= min_positives:
            return labels, thr, True
    labels = (risk_scores >= thresholds[-1]).astype("int64")
    return labels, thresholds[-1], False


def compute_dual_signal_labels(
    signals_df: pd.DataFrame,
    weights: Dict[str, float],
) -> pd.DataFrame:
    """Score and threshold per project. Returns the labels.parquet row set.

    ``signals_df`` must contain columns ``project_id``, ``basename``, and
    ``S1_severity..S6_contributors``. The output adds ``risk_score``,
    ``is_high_risk``, ``threshold_used``, ``min_positives_satisfied``.
    """
    score = np.zeros(len(signals_df), dtype="float64")
    for s, w in weights.items():
        score += float(w) * signals_df[s].astype("float64").values
    out = signals_df[["project_id", "basename", *SIGNAL_COLUMNS]].copy()
    out["risk_score"] = score

    is_high = np.zeros(len(out), dtype="int64")
    threshold_used = np.zeros(len(out), dtype="float64")
    min_positives_ok = np.ones(len(out), dtype="int64")
    out = out.reset_index(drop=True)
    for _, idx in out.groupby("project_id").groups.items():
        positions = np.asarray(idx)
        rs = out.loc[positions, "risk_score"]
        labels, thr, ok = apply_label_thresholding(rs)
        is_high[positions] = labels.values
        threshold_used[positions] = thr
        if not ok:
            min_positives_ok[positions] = 0
    out["is_high_risk"] = is_high
    out["threshold_used"] = threshold_used
    out["min_positives_satisfied"] = min_positives_ok
    return out


# =================================================
# COMBINED-WEIGHT HIGH-RISK TD LABEL
# =================================================
# Binary output: is_high_risk = 1 or 0
#
# Six binary signals from two independent sources:
#   Static (SonarQube at t):   S1, S2, S3
#   History (Git before t):    S4, S5, S6
#
# Weights derived empirically via point-biserial
# correlation with post-snapshot bug-fix activity
# (Kamei et al. TSE 2013 weighting approach).
# Confirmed against theoretically motivated baseline.
#
# Label is binary 1/0 - not probability, not score.
# risk_score is internal computation only.
# =================================================
"""
Stage 4 - Compute the single dual-signal binary label.

Outputs
-------
- ``data/processed/labels.parquet``
    (project_id, basename, is_high_risk, risk_score, S1..S6)
- ``data/processed/label_statistics.csv``
    (project_id, n_files, n_positive, positive_rate_pct,
     n_s1_only, n_s4_only, n_both_static_history, threshold_used)
- ``data/processed/derived_weights.json``
- ``data/processed/weight_comparison.csv``
- ``results/tables/label_summary.csv``
"""

import json
import sys
import time
from pathlib import Path

import pandas as pd


# SNAPSHOT: t = median commit date per project
# Rationale: ensures equal proportional history
# across all 22 projects for LOPO comparability.
# Follows Tsoukalas 2020 and Jiang 2024 convention.
# 6-month label window follows Kamei et al. 2013.
# Data after t+6 months discarded: beyond this
# window, activity reflects new debt after t.


def _load_clean() -> dict[str, pd.DataFrame]:
    commits = pd.read_parquet(PROCESSED_DATA_DIR / "clean_git_commits.parquet")
    changes = pd.read_parquet(PROCESSED_DATA_DIR / "clean_git_commits_changes.parquet")
    sonar_issues = pd.read_parquet(PROCESSED_DATA_DIR / "clean_sonar_issues.parquet")
    for df, cols in (
        (commits, ["AUTHOR_DATE", "COMMITTER_DATE"]),
        (changes, ["DATE"]),
        (sonar_issues, ["CREATION_DATE", "CLOSE_DATE"]),
    ):
        for col in cols:
            if col in df.columns and df[col].dtype.kind == "M" and df[col].dt.tz is None:
                df[col] = df[col].dt.tz_localize("UTC")
    return {"commits": commits, "changes": changes, "sonar_issues": sonar_issues}


def run_stage_4() -> None:
    t0 = time.time()
    print(f"[Stage 4] Surrogate window     : {LABEL_SURROGATE_WINDOW_MONTHS} months")
    print(f"[Stage 4] Theoretical baseline : {THEORETICAL_WEIGHTS}")

    snaps = load_snapshots(PROCESSED_DATA_DIR / "project_snapshots.parquet")
    eligible = snaps[snaps["eligible"]].copy()
    print(f"[Stage 4] Eligible projects    : {len(eligible)}")

    print("[Stage 4] Loading cleaned parquets ...")
    data = _load_clean()
    print(f"   commits      : {len(data['commits']):>10,}")
    print(f"   changes      : {len(data['changes']):>10,}")
    print(f"   sonar_issues : {len(data['sonar_issues']):>10,}")

    # --- Build per-project signals (incl. surrogate) -----------------------
    print("\n[Stage 4] Computing per-project signals ...")
    parts: list[pd.DataFrame] = []
    for _, row in eligible.sort_values("project_id").iterrows():
        pid = row["project_id"]
        t = row["snapshot_date"]
        sub = compute_dual_signal_signals(
            pid, t, data["commits"], data["changes"], data["sonar_issues"]
        )
        parts.append(sub)
        print(
            f"   {pid:<35}  N={len(sub):>6}  "
            f"S1={int(sub['S1_severity'].sum()):>4}  "
            f"S4={int(sub['S4_bugfix'].sum()):>4}  "
            f"surrogate={int(sub['future_bugfix_count'].sum()):>5}"
        )
    signals_all = pd.concat(parts, ignore_index=True)

    # --- Empirical weight derivation --------------------------------------
    print("\n[Stage 4] Deriving empirical weights via point-biserial ...")
    derived = derive_weights_empirically(signals_all, signals_all[["future_bugfix_count"]])
    chosen_weights, source = choose_weights(derived, THEORETICAL_WEIGHTS, tolerance=0.05)
    if source == "theoretical":
        print("[Stage 4] Empirical weights confirmed theoretical baseline (max diff <= 0.05).")
    else:
        print("[Stage 4] Empirical weights diverge from theoretical (max diff > 0.05); using empirical.")

    print("[Stage 4] Weights (in use):")
    for s in SIGNAL_COLUMNS:
        print(f"   {s:<20} theoretical={THEORETICAL_WEIGHTS[s]:.2f}  derived={derived[s]:.2f}  chosen={chosen_weights[s]:.2f}")

    weights_path = PROCESSED_DATA_DIR / "derived_weights.json"
    with weights_path.open("w", encoding="utf-8") as fh:
        json.dump({**chosen_weights, "_source": source}, fh, indent=2)

    comparison = pd.DataFrame(
        {
            "signal": list(SIGNAL_COLUMNS),
            "theoretical": [THEORETICAL_WEIGHTS[s] for s in SIGNAL_COLUMNS],
            "derived": [derived[s] for s in SIGNAL_COLUMNS],
            "chosen": [chosen_weights[s] for s in SIGNAL_COLUMNS],
        }
    )
    comparison["difference"] = (comparison["derived"] - comparison["theoretical"]).round(3)

    def _interpret(diff: float) -> str:
        # Magnitude bands chosen so anything beyond 0.05 (the
        # theoretical->empirical fallback trigger) gets called out
        # explicitly, with sign telling the reader whether the surrogate
        # placed more or less weight on the signal than literature did.
        ad = abs(diff)
        if ad <= 0.02:
            return "matches theoretical baseline (no meaningful divergence)"
        if diff < 0:
            kind = "weaker"
        else:
            kind = "stronger"
        if ad <= 0.05:
            magnitude = "modestly"
        elif ad <= 0.10:
            magnitude = "noticeably"
        else:
            magnitude = "substantially"
        return f"{magnitude} {kind} empirical correlation with post-snapshot bug-fix activity than Kamei 2013 baseline"

    comparison["interpretation"] = comparison["difference"].apply(_interpret)
    comparison.to_csv(PROCESSED_DATA_DIR / "weight_comparison.csv", index=False)

    # --- Compute labels using chosen weights -------------------------------
    print("\n[Stage 4] Scoring + thresholding per project ...")
    # Drop surrogate before label scoring so it never leaks into the saved frame.
    signals_only = signals_all.drop(columns=["future_bugfix_count"])
    labeled = compute_dual_signal_labels(signals_only, chosen_weights)
    assert "future_bugfix_count" not in labeled.columns, "surrogate leaked into labels"
    assert set(labeled["is_high_risk"].unique()) <= {0, 1}

    # --- Per-project label_statistics --------------------------------------
    static_flag = (labeled[["S1_severity", "S2_debt", "S3_smells"]].sum(axis=1) >= 1).astype("int64")
    history_flag = (labeled[["S4_bugfix", "S5_churn", "S6_contributors"]].sum(axis=1) >= 1).astype("int64")
    labeled["_static_any"] = static_flag
    labeled["_history_any"] = history_flag

    stats_rows = []
    for pid, sub in labeled.groupby("project_id"):
        s1_only = int(((sub["S1_severity"] == 1) & (sub["_history_any"] == 0)).sum())
        s4_only = int(((sub["S4_bugfix"] == 1) & (sub["_static_any"] == 0)).sum())
        both = int(((sub["_static_any"] == 1) & (sub["_history_any"] == 1) & (sub["is_high_risk"] == 1)).sum())
        thr = float(sub["threshold_used"].iloc[0])
        stats_rows.append(
            {
                "project_id": pid,
                "n_files": int(len(sub)),
                "n_positive": int(sub["is_high_risk"].sum()),
                "positive_rate_pct": round(100 * float(sub["is_high_risk"].mean()), 2),
                "n_s1_only": s1_only,
                "n_s4_only": s4_only,
                "n_both_static_history": both,
                "threshold_used": thr,
                "min_positives_satisfied": int(sub["min_positives_satisfied"].iloc[0]),
            }
        )
    stats = pd.DataFrame(stats_rows).sort_values("project_id").reset_index(drop=True)
    stats.to_csv(PROCESSED_DATA_DIR / "label_statistics.csv", index=False)

    # Concise per-project summary for the thesis tables/ folder.
    label_summary = stats[["project_id", "n_files", "n_positive", "positive_rate_pct", "threshold_used"]].copy()
    label_summary.to_csv(TABLES_DIR / "label_summary.csv", index=False)

    # --- Persist labels (drop helper cols) ---------------------------------
    keep_cols = [
        "project_id",
        "basename",
        "is_high_risk",
        "risk_score",
        *SIGNAL_COLUMNS,
    ]
    labels_to_save = labeled[keep_cols].copy()
    labels_to_save.to_parquet(PROCESSED_DATA_DIR / "labels.parquet", index=False)

    # --- Stage summary -----------------------------------------------------
    print("\n[Stage 4] Per-project label statistics:")
    with pd.option_context("display.width", 200, "display.max_rows", None):
        print(stats.to_string(index=False))

    overall_rate = float(labeled["is_high_risk"].mean()) * 100
    print(f"\n[Stage 4] Overall positive rate : {overall_rate:.2f}%  "
          f"({int(labeled['is_high_risk'].sum()):,} of {len(labeled):,})")

    low_pos = stats[stats["min_positives_satisfied"] == 0]
    if len(low_pos) > 0:
        print(f"[Stage 4] WARNING: {len(low_pos)} project(s) have < 5 positives even at lowest threshold:")
        print(low_pos[["project_id", "n_positive", "threshold_used"]].to_string(index=False))

    print(f"\n[Stage 4] Total elapsed : {time.time() - t0:.1f}s")
    print("[Stage 4] Complete.")


In [ ]:
run_stage_4()
print()
print('--- derived_weights.json ---')
print(json.dumps(json.loads(Path('data/processed/derived_weights.json').read_text()), indent=2))
print()
print('--- label_summary.csv ---')
show_table('results/tables/label_summary.csv')

# fig_01/02/03 are produced by Stage 10's report renderer (Section 12);
# they will not exist at this point. show_figure prints a notice and
# moves on rather than raising.
print()
show_figure('results/figures/fig_01_positive_rates.png')
show_figure('results/figures/fig_02_label_signal_breakdown.png')
show_figure('results/figures/fig_03_risk_score_distribution.png')


## Section 5 — Feature Engineering

Extract 27 features in 5 families (size/complexity, static debt, historical,
co-change graph, prior defect) at the project/basename snapshot. Writes
four `features_*.parquet` files + `feature_summary.csv`.

In [ ]:
# Pipeline code

from joblib import Parallel, delayed

"""
Static feature extraction at (project, basename) granularity.

Emits two of the five feature families (10 features total):

Family 1 - Size / Complexity (project-level context replicated per file):
    ncloc, complexity, cognitive_complexity, functions, classes

Family 2 - Static Debt (per-file aggregates over SONAR_ISSUES at t):
    n_code_smells, n_bugs, total_debt_minutes, issue_density,
    duplicated_lines_density

``ncloc`` and ``duplicated_lines_density`` come from the most recent
SONAR_ANALYSIS at or before ``t`` (project-level snapshot). All issue
aggregates use ``CREATION_DATE <= t`` and respect the issue's
``CLOSE_DATE`` window (open at snapshot).

Leakage note: raw severity counts (n_blocker, n_critical, etc.) are
deliberately NOT emitted - the dual-signal label S1 keys on
BLOCKER/CRITICAL counts, so including them as features would leak the
label. See assertion in scripts/06_build_dataset.py.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Family 2 - Static Debt (per-file from SONAR_ISSUES)
# ---------------------------------------------------------------------------
def _issue_aggregates_at_snapshot(
    sonar_issues: pd.DataFrame,
    project_id: str,
    snapshot: pd.Timestamp,
) -> pd.DataFrame:
    """Per-basename counts of CODE_SMELL, BUG, and total debt minutes."""
    open_iss = open_issues_at_snapshot(sonar_issues, project_id, snapshot)
    cols = ["basename", "n_code_smells", "n_bugs", "total_debt_minutes", "n_issues_open"]
    if open_iss.empty:
        return pd.DataFrame(columns=cols)

    issue_type = open_iss["TYPE"].astype(str).str.upper()
    smell_mask = issue_type == "CODE_SMELL"
    bug_mask = issue_type == "BUG"

    n_smells = open_iss.loc[smell_mask].groupby("basename").size().rename("n_code_smells")
    n_bugs = open_iss.loc[bug_mask].groupby("basename").size().rename("n_bugs")
    debt = open_iss.groupby("basename")["DEBT"].sum(min_count=1).rename("total_debt_minutes")
    n_open = open_iss.groupby("basename").size().rename("n_issues_open")

    out = pd.concat([n_smells, n_bugs, debt, n_open], axis=1).reset_index()
    out["n_code_smells"] = out["n_code_smells"].fillna(0).astype("int64")
    out["n_bugs"] = out["n_bugs"].fillna(0).astype("int64")
    out["n_issues_open"] = out["n_issues_open"].fillna(0).astype("int64")
    out["total_debt_minutes"] = out["total_debt_minutes"].fillna(0.0).astype(float)
    return out


# ---------------------------------------------------------------------------
# Family 1 - Size / Complexity (project-level context at t)
# ---------------------------------------------------------------------------
_PROJECT_LEVEL_METRICS = (
    "ncloc",
    "complexity",
    "cognitive_complexity",
    "functions",
    "classes",
    "duplicated_lines_density",
)


def _project_metrics_at_snapshot(
    sonar_measures: pd.DataFrame,
    project_id: str,
    snapshot: pd.Timestamp,
) -> dict:
    """Most recent SONAR_ANALYSIS metrics at or before ``t``.

    TD Dataset v2 stores SonarQube measures at project granularity (no
    per-file COMPONENT column), so the same row is replicated for every
    basename in the project. Empty dict if no analysis exists before t.
    """
    t = snapshot
    if t.tz is None:
        t = t.tz_localize("UTC")
    m = sonar_measures[sonar_measures["project_id"] == project_id]
    m = m[m["analysis_date"] <= t]
    if m.empty:
        return {}
    row = m.sort_values("analysis_date").iloc[-1]
    return {c: row.get(c) for c in _PROJECT_LEVEL_METRICS if c in m.columns}


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------
def build_static_features_for_project(
    project_id: str,
    snapshot: pd.Timestamp,
    sonar_issues: pd.DataFrame,
    sonar_measures: pd.DataFrame,
    changes: pd.DataFrame,
) -> pd.DataFrame:
    """Return ``(project_id, basename, <10 static features>)`` for one project."""
    universe = basename_universe_at_snapshot(changes, project_id, snapshot)
    if universe.empty:
        return universe.assign(
            **{
                "ncloc": 0.0,
                "complexity": 0.0,
                "cognitive_complexity": 0.0,
                "functions": 0.0,
                "classes": 0.0,
                "n_code_smells": 0,
                "n_bugs": 0,
                "total_debt_minutes": 0.0,
                "issue_density": 0.0,
                "duplicated_lines_density": 0.0,
            }
        )

    issue_feats = _issue_aggregates_at_snapshot(sonar_issues, project_id, snapshot)
    ctx = _project_metrics_at_snapshot(sonar_measures, project_id, snapshot)

    df = universe.merge(issue_feats, on="basename", how="left")
    for c in ("n_code_smells", "n_bugs", "n_issues_open"):
        df[c] = df[c].fillna(0).astype("int64")
    df["total_debt_minutes"] = df["total_debt_minutes"].fillna(0.0).astype(float)

    # Project-level metrics replicated per file (NaN -> 0.0 if no analysis <= t).
    for col in ("ncloc", "complexity", "cognitive_complexity", "functions", "classes",
                "duplicated_lines_density"):
        val = ctx.get(col)
        df[col] = float(val) if val is not None and not pd.isna(val) else 0.0

    # Issue density (per spec). Guard ncloc == 0.
    ncloc_safe = df["ncloc"].replace(0, np.nan)
    df["issue_density"] = np.where(
        df["ncloc"] > 0,
        (df["n_issues_open"] / ncloc_safe).fillna(0.0),
        0.0,
    ).astype(float)

    # Drop helper column not in the 27 features.
    df = df.drop(columns=["n_issues_open"])
    df["snapshot_date"] = snapshot
    return df


"""
Family 3 - Historical change metrics at (project, basename) granularity.

8 features derived from ``GIT_COMMITS`` and ``GIT_COMMITS_CHANGES``
restricted to ``AUTHOR_DATE <= snapshot``:

    total_commits_pre       - distinct commits touching the file
    code_churn_pre          - lifetime added + removed lines
    recent_churn_90d        - churn in last 90 days before t
    commit_frequency_30d    - commit count in last 30 days before t
    file_age_days           - days between first commit and t
    days_since_last_change  - days between most recent commit and t
    contributor_count       - distinct authors
    ownership_ratio         - max single author commits / total
                              (1.0 if total_commits_pre == 0)

References: Kamei TSE 2013, Hassan ICSE 2009.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _ensure_utc(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    if ts.tz is None:
        ts = ts.tz_localize("UTC")
    return ts


def build_historical_features_for_project(
    project_id: str,
    snapshot: pd.Timestamp,
    commits: pd.DataFrame,
    changes: pd.DataFrame,
) -> pd.DataFrame:
    t = _ensure_utc(snapshot)
    universe = basename_universe_at_snapshot(changes, project_id, t)
    if universe.empty:
        return universe

    c = commits[commits["PROJECT_ID"] == project_id]
    c = c[c["AUTHOR_DATE"] <= t][["COMMIT_HASH", "AUTHOR_DATE", "AUTHOR"]]

    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["DATE"] <= t].copy()
    ch = ch.merge(c, on="COMMIT_HASH", how="left")
    ch["row_churn"] = ch["LINES_ADDED"].fillna(0).astype("int64") + ch["LINES_REMOVED"].fillna(0).astype("int64")

    t_30 = t - pd.Timedelta(days=30)
    t_90 = t - pd.Timedelta(days=90)
    ch["in_30d"] = (ch["AUTHOR_DATE"] > t_30) & (ch["AUTHOR_DATE"] <= t)
    ch["in_90d"] = (ch["AUTHOR_DATE"] > t_90) & (ch["AUTHOR_DATE"] <= t)

    base = ch.groupby("basename").agg(
        total_commits_pre=("COMMIT_HASH", "nunique"),
        contributor_count=("AUTHOR", "nunique"),
        code_added_pre=("LINES_ADDED", "sum"),
        code_removed_pre=("LINES_REMOVED", "sum"),
        first_commit_date=("AUTHOR_DATE", "min"),
        last_commit_date=("AUTHOR_DATE", "max"),
    )
    base["code_churn_pre"] = (
        base["code_added_pre"].fillna(0).astype("int64")
        + base["code_removed_pre"].fillna(0).astype("int64")
    )

    w90 = (
        ch[ch["in_90d"]]
        .groupby("basename")
        .agg(recent_churn_90d=("row_churn", "sum"))
    )
    w30 = (
        ch[ch["in_30d"]]
        .groupby("basename")
        .agg(commit_frequency_30d=("COMMIT_HASH", "nunique"))
    )

    author_commits = (
        ch.dropna(subset=["AUTHOR"])
        .groupby(["basename", "AUTHOR"])["COMMIT_HASH"]
        .nunique()
        .reset_index(name="author_commits")
    )
    top_author = (
        author_commits.groupby("basename")["author_commits"].max().rename("max_commits_by_author")
    )

    df = universe.merge(base.reset_index(), on="basename", how="left")
    df = df.merge(w90.reset_index(), on="basename", how="left")
    df = df.merge(w30.reset_index(), on="basename", how="left")
    df = df.merge(top_author.reset_index(), on="basename", how="left")

    df["file_age_days"] = (t - df["first_commit_date"]).dt.days
    df["days_since_last_change"] = (t - df["last_commit_date"]).dt.days
    df["ownership_ratio"] = np.where(
        df["total_commits_pre"].fillna(0) > 0,
        df["max_commits_by_author"] / df["total_commits_pre"].replace(0, np.nan),
        1.0,
    )

    int_cols = [
        "total_commits_pre",
        "contributor_count",
        "code_churn_pre",
        "recent_churn_90d",
        "commit_frequency_30d",
        "file_age_days",
        "days_since_last_change",
    ]
    for col in int_cols:
        df[col] = df[col].fillna(0).astype("int64")
    df["ownership_ratio"] = df["ownership_ratio"].fillna(1.0).astype(float)

    keep = [
        "project_id",
        "basename",
        "total_commits_pre",
        "code_churn_pre",
        "recent_churn_90d",
        "commit_frequency_30d",
        "file_age_days",
        "days_since_last_change",
        "contributor_count",
        "ownership_ratio",
    ]
    return df[keep]


"""
Family 4 - Co-change graph features at (project, basename) granularity.

For each project at snapshot ``t``, an undirected weighted co-change
graph is built using commits with ``DATE <= t``:

- Nodes: distinct basenames touched by any pre-``t`` commit.
- Edges: for every commit that touches multiple files, every pair of
  basenames gets +1 edge weight (so weight = number of shared commits).

Four features (per spec):
    cocg_degree       - number of co-change neighbours
    cocg_pagerank     - weighted PageRank (damping 0.85)
    cocg_betweenness  - normalised betweenness centrality
    cocg_entropy      - co-change scattering entropy
                        (Ethari & Bhardwaj 2025):
                        H = -sum(p_i * log2(p_i)) over normalised
                        edge weights to neighbours; 0.0 if degree==0.

igraph is used for the centralities (PRPACK + Brandes); a parity check
against NetworkX runs at module import on a 50-node BA graph.
"""

import math
import sys
from collections import Counter
from itertools import combinations
from pathlib import Path

import igraph as ig
import networkx as nx
import numpy as np
import pandas as pd


_PARITY_TOL_BC = 1e-9
_PARITY_TOL_PR = 5e-6

GRAPH_FEATURE_COLS: tuple[str, ...] = (
    "cocg_degree",
    "cocg_pagerank",
    "cocg_betweenness",
    "cocg_entropy",
)


def _ensure_utc(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    if ts.tz is None:
        ts = ts.tz_localize("UTC")
    return ts


def _empty_features_for_universe(universe: pd.DataFrame) -> pd.DataFrame:
    out = universe.copy()
    out["cocg_degree"] = 0
    out["cocg_pagerank"] = 0.0
    out["cocg_betweenness"] = 0.0
    out["cocg_entropy"] = 0.0
    out["cocg_degree"] = out["cocg_degree"].astype("int64")
    return out


def _build_graph_from_changes(ch: pd.DataFrame) -> nx.Graph:
    by_commit = ch.groupby("COMMIT_HASH")["basename"].unique()
    edge_counts: Counter[tuple[str, str]] = Counter()
    all_nodes: set[str] = set()
    for files in by_commit:
        if files is None:
            continue
        all_nodes.update(files)
        if len(files) < 2:
            continue
        sorted_files = sorted(files)
        edge_counts.update(combinations(sorted_files, 2))

    G = nx.Graph()
    G.add_nodes_from(all_nodes)
    if edge_counts:
        G.add_weighted_edges_from(((u, v, w) for (u, v), w in edge_counts.items()))
    return G


def _compute_centralities_igraph(G: nx.Graph) -> tuple[dict[str, float], dict[str, float]]:
    """Return (betweenness, pagerank) dicts keyed by basename."""
    nodes = list(G.nodes())
    n = len(nodes)
    if n == 0:
        return {}, {}

    idx = {nm: i for i, nm in enumerate(nodes)}
    edge_list: list[tuple[int, int]] = []
    weights: list[float] = []
    for u, v, d in G.edges(data=True):
        edge_list.append((idx[u], idx[v]))
        weights.append(float(d.get("weight", 1.0)))
    ig_G = ig.Graph(n=n, edges=edge_list, directed=False)

    if n > 2:
        bc_raw = ig_G.betweenness(directed=False)
        denom = (n - 1) * (n - 2) / 2.0
        betweenness = {nodes[i]: float(bc_raw[i] / denom) for i in range(n)}
    else:
        betweenness = {nodes[i]: 0.0 for i in range(n)}

    pr_weights = weights if weights else None
    try:
        pr = ig_G.pagerank(damping=0.85, weights=pr_weights)
    except Exception:
        pr = ig_G.pagerank(damping=0.85)
    pagerank = {nodes[i]: float(pr[i]) for i in range(n)}
    return betweenness, pagerank


def _compute_entropy(G: nx.Graph) -> dict[str, float]:
    """Per-node Shannon entropy over normalised edge weights to neighbours.

    H_v = -sum_i p_i * log2(p_i) where p_i = w_i / sum(w).
    Returns 0.0 for isolated nodes (degree == 0). Single-neighbour nodes
    have p_1 = 1.0 -> 0.0 entropy as well.
    """
    out: dict[str, float] = {}
    for node in G.nodes():
        neighbours = list(G[node])
        if not neighbours:
            out[node] = 0.0
            continue
        weights = [float(G[node][nb].get("weight", 1.0)) for nb in neighbours]
        total = sum(weights)
        if total <= 0:
            out[node] = 0.0
            continue
        h = 0.0
        for w in weights:
            p = w / total
            if p > 0:
                h -= p * math.log2(p)
        out[node] = float(h)
    return out


def _parity_test() -> None:
    """Assert igraph centralities match NetworkX on a small graph at import."""
    rng = np.random.default_rng(42)
    G = nx.barabasi_albert_graph(50, 3, seed=42)
    for u, v in G.edges():
        G[u][v]["weight"] = float(rng.integers(1, 10))

    ig_bc, ig_pr = _compute_centralities_igraph(G)
    nx_bc = nx.betweenness_centrality(G, normalized=True)
    nx_pr = nx.pagerank(G, weight="weight")

    def _max_diff(a: dict, b: dict) -> float:
        return max(abs(a[k] - b[k]) for k in a)

    diff_bc = _max_diff(ig_bc, nx_bc)
    diff_pr = _max_diff(ig_pr, nx_pr)
    failures = []
    if diff_bc > _PARITY_TOL_BC:
        failures.append(f"betweenness: {diff_bc:.3e} > tol {_PARITY_TOL_BC:.0e}")
    if diff_pr > _PARITY_TOL_PR:
        failures.append(f"pagerank: {diff_pr:.3e} > tol {_PARITY_TOL_PR:.0e}")
    if failures:
        raise RuntimeError(
            "graph_features parity test FAILED (igraph vs networkx):\n  "
            + "\n  ".join(failures)
        )


_parity_test()


def build_graph_features_for_project(
    project_id: str,
    snapshot: pd.Timestamp,
    changes: pd.DataFrame,
    verbose: bool = True,
) -> pd.DataFrame:
    """Per-basename co-change graph features for one project at snapshot ``t``."""
    import time as _time

    def _log(msg: str) -> None:
        if verbose:
            print(f"      [graph:{project_id}] {msg}", flush=True)

    t = _ensure_utc(snapshot)
    universe = basename_universe_at_snapshot(changes, project_id, t)
    if universe.empty:
        return universe

    ch = changes[(changes["PROJECT_ID"] == project_id) & (changes["DATE"] <= t)]
    if ch.empty:
        return _empty_features_for_universe(universe)

    _ts = _time.time()
    G = _build_graph_from_changes(ch[["COMMIT_HASH", "basename"]])
    _log(f"graph built V={G.number_of_nodes()} E={G.number_of_edges()} ({_time.time()-_ts:.1f}s)")
    if G.number_of_edges() == 0:
        return _empty_features_for_universe(universe)

    _ts = _time.time()
    betweenness, pagerank = _compute_centralities_igraph(G)
    _log(f"centralities ({_time.time()-_ts:.1f}s)")

    _ts = _time.time()
    entropy = _compute_entropy(G)
    _log(f"entropy ({_time.time()-_ts:.1f}s)")

    degree = dict(G.degree())

    rows = []
    for node in G.nodes():
        rows.append(
            {
                "basename": node,
                "cocg_degree": int(degree.get(node, 0)),
                "cocg_pagerank": float(pagerank.get(node, 0.0)),
                "cocg_betweenness": float(betweenness.get(node, 0.0)),
                "cocg_entropy": float(entropy.get(node, 0.0)),
            }
        )
    feats = pd.DataFrame(rows)

    out = universe.merge(feats, on="basename", how="left")
    out["cocg_degree"] = out["cocg_degree"].fillna(0).astype("int64")
    for col in ("cocg_pagerank", "cocg_betweenness", "cocg_entropy"):
        out[col] = out[col].fillna(0.0).astype(float)
    return out


"""
Family 5 - Prior defect history features at (project, basename) granularity.

Five features computed strictly from events with date ``<= t``:

    bugfix_commits_pre   - lifetime bug-fix commit count
    bugfix_commits_90d   - bug-fix commits in last 90 days before t
    bug_density_pre      - bugfix_commits_pre / total_commits_pre
                           (0.0 if total_commits_pre == 0)
    n_jira_bugs_pre      - JIRA Bug-type tickets linked to the file
                           before t
    jira_blocker_flag    - 1 if any linked JIRA Bug has
                           PRIORITY in {Blocker, Critical}, else 0

References: Hassan ICSE 2009, Falessi et al. ESEM 2020.
The same bug-fix regex used for the dual-signal label is reused via
the ``is_bugfix`` column produced in stage 3.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd


PRIOR_DEFECT_FEATURE_COLS: tuple[str, ...] = (
    "bugfix_commits_pre",
    "bugfix_commits_90d",
    "bug_density_pre",
    "n_jira_bugs_pre",
    "jira_blocker_flag",
)


def _ensure_utc(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    if ts.tz is None:
        ts = ts.tz_localize("UTC")
    return ts


def _empty_features_for_universe(universe: pd.DataFrame) -> pd.DataFrame:
    out = universe.copy()
    out["bugfix_commits_pre"] = 0
    out["bugfix_commits_90d"] = 0
    out["bug_density_pre"] = 0.0
    out["n_jira_bugs_pre"] = 0
    out["jira_blocker_flag"] = 0
    for col in ("bugfix_commits_pre", "bugfix_commits_90d", "n_jira_bugs_pre", "jira_blocker_flag"):
        out[col] = out[col].astype("int64")
    return out


def _bugfix_aggregates(
    commits: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
) -> pd.DataFrame:
    """Per-basename bug-fix counts + total commits for the bug_density_pre denominator."""
    c_all = commits[commits["PROJECT_ID"] == project_id]
    c_all = c_all[c_all["AUTHOR_DATE"] <= t][["COMMIT_HASH", "AUTHOR_DATE", "is_bugfix"]]

    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["DATE"] <= t][["COMMIT_HASH", "basename"]]
    if ch.empty or c_all.empty:
        return pd.DataFrame(
            columns=["basename", "bugfix_commits_pre", "bugfix_commits_90d", "total_commits_pre"]
        )

    touches = ch.merge(c_all, on="COMMIT_HASH", how="inner")
    touches = touches.drop_duplicates(["basename", "COMMIT_HASH"])
    t_90 = t - pd.Timedelta(days=90)
    touches["is_bugfix"] = touches["is_bugfix"].fillna(False).astype(bool)
    touches["in_90d"] = touches["AUTHOR_DATE"] > t_90

    agg = touches.groupby("basename").agg(
        total_commits_pre=("COMMIT_HASH", "size"),
        bugfix_commits_pre=("is_bugfix", "sum"),
        bugfix_commits_90d=("is_bugfix", lambda s: int((s & touches.loc[s.index, "in_90d"]).sum())),
    )
    for col in ("total_commits_pre", "bugfix_commits_pre", "bugfix_commits_90d"):
        agg[col] = agg[col].fillna(0).astype("int64")
    return agg.reset_index()


def _jira_bug_aggregates(
    commits: pd.DataFrame,
    jira: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
) -> pd.DataFrame:
    """Per-basename JIRA Bug counts and Blocker/Critical priority flag, pre-t."""
    j = jira[jira["PROJECT_ID"] == project_id].copy()
    j = j[j["is_bug"] & j["HASH"].notna()]
    if j.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bugs_pre", "jira_blocker_flag"])

    # Resolved before t (or commit-date before t) -- use the earlier of the two if available.
    j["link_date"] = j[["RESOLUTION_DATE", "COMMIT_DATE", "UPDATE_DATE"]].min(axis=1)
    j = j[j["link_date"].isna() | (j["link_date"] <= t)]
    if j.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bugs_pre", "jira_blocker_flag"])

    j["priority_high"] = (
        j["PRIORITY"].fillna("").astype(str).str.upper().isin({"BLOCKER", "CRITICAL"})
    )

    bug_hashes = j[["HASH", "priority_high"]].drop_duplicates(subset=["HASH"])

    c = commits[commits["PROJECT_ID"] == project_id]
    c = c[(c["AUTHOR_DATE"] <= t) & c["COMMIT_HASH"].isin(bug_hashes["HASH"])][
        ["COMMIT_HASH"]
    ]
    if c.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bugs_pre", "jira_blocker_flag"])

    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[(ch["DATE"] <= t) & ch["COMMIT_HASH"].isin(c["COMMIT_HASH"])][
        ["COMMIT_HASH", "basename"]
    ]
    if ch.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bugs_pre", "jira_blocker_flag"])

    ch = ch.merge(bug_hashes, left_on="COMMIT_HASH", right_on="HASH", how="left")

    agg = ch.groupby("basename").agg(
        n_jira_bugs_pre=("HASH", "nunique"),
        jira_blocker_flag=("priority_high", "any"),
    )
    agg["n_jira_bugs_pre"] = agg["n_jira_bugs_pre"].fillna(0).astype("int64")
    agg["jira_blocker_flag"] = agg["jira_blocker_flag"].fillna(False).astype("int64")
    return agg.reset_index()


def build_priordefect_features_for_project(
    project_id: str,
    snapshot: pd.Timestamp,
    commits: pd.DataFrame,
    changes: pd.DataFrame,
    szz: pd.DataFrame,  # kept in signature for stage-5 caller compatibility; not used
    jira: pd.DataFrame,
) -> pd.DataFrame:
    t = _ensure_utc(snapshot)
    universe = basename_universe_at_snapshot(changes, project_id, t)
    if universe.empty:
        return _empty_features_for_universe(universe)

    bf = _bugfix_aggregates(commits, changes, project_id, t)
    jb = _jira_bug_aggregates(commits, jira, changes, project_id, t)

    out = universe.merge(bf, on="basename", how="left")
    out = out.merge(jb, on="basename", how="left")

    for col in ("bugfix_commits_pre", "bugfix_commits_90d", "total_commits_pre",
                "n_jira_bugs_pre", "jira_blocker_flag"):
        if col not in out.columns:
            out[col] = 0
        out[col] = out[col].fillna(0).astype("int64")

    out["bug_density_pre"] = np.where(
        out["total_commits_pre"] > 0,
        out["bugfix_commits_pre"] / out["total_commits_pre"].replace(0, np.nan),
        0.0,
    ).astype(float)
    out["bug_density_pre"] = out["bug_density_pre"].fillna(0.0)

    keep = [
        "project_id",
        "basename",
        "bugfix_commits_pre",
        "bugfix_commits_90d",
        "bug_density_pre",
        "n_jira_bugs_pre",
        "jira_blocker_flag",
    ]
    return out[keep]


"""
Stage 5 - Snapshot-aware feature extraction.

For each eligible project (``snapshot`` from Stage 2), builds:

1. Per-(project, basename) *static* features (SonarQube issue aggregates
   at snapshot + git-derived size + project-level context from the
   latest analysis <= ``t``).
2. Per-(project, basename) *historical* (process) features restricted
   to commits with ``AUTHOR_DATE <= t``.
3. Per-(project, basename) *co-change graph* features (centrality and
   coupling metrics on the snapshot-time co-change graph).
4. Per-(project, basename) *prior-defect* features (pre-snapshot
   bug-fix counts, SZZ-inducing flags, linked Jira tickets).

Outputs
-------
- ``data/processed/features_static.parquet``
- ``data/processed/features_historical.parquet``
- ``data/processed/features_graph.parquet``
- ``data/processed/features_priordefect.parquet``
- ``results/tables/feature_summary.csv`` - per-project row counts and
  quick statistics.

Parallelism
-----------
Projects are independent: each project's features are computed from a
self-contained slice of the cleaned dataframes. We process them in
parallel with ``joblib.Parallel`` using ``STAGE5_N_JOBS`` workers
(default ``cpu_count // 2``; override with the ``TD_N_JOBS`` env
var). Workers see only their project's pre-filtered slice, which
keeps per-worker RAM small and pickling fast.

Run
---
.. code-block:: bash

    .\\venv\\Scripts\\python.exe scripts/05_features.py
"""

import sys
import time
from pathlib import Path
from typing import Any

import pandas as pd
from joblib import Parallel, delayed


def _load_clean() -> dict[str, pd.DataFrame]:
    commits = pd.read_parquet(PROCESSED_DATA_DIR / "clean_git_commits.parquet")
    changes = pd.read_parquet(PROCESSED_DATA_DIR / "clean_git_commits_changes.parquet")
    sonar_issues = pd.read_parquet(PROCESSED_DATA_DIR / "clean_sonar_issues.parquet")
    sonar_measures = pd.read_parquet(PROCESSED_DATA_DIR / "clean_sonar_measures.parquet")
    szz = pd.read_parquet(PROCESSED_DATA_DIR / "clean_szz.parquet")
    jira = pd.read_parquet(PROCESSED_DATA_DIR / "clean_jira_issues.parquet")

    for df, cols in (
        (commits, ["AUTHOR_DATE", "COMMITTER_DATE"]),
        (changes, ["DATE"]),
        (sonar_issues, ["CREATION_DATE", "CLOSE_DATE"]),
        (sonar_measures, ["analysis_date"]),
        (szz, ["fix_date", "induce_date"]),
        (jira, ["CREATION_DATE", "RESOLUTION_DATE", "UPDATE_DATE", "COMMIT_DATE"]),
    ):
        for col in cols:
            if col in df.columns and df[col].dtype.kind == "M" and df[col].dt.tz is None:
                df[col] = df[col].dt.tz_localize("UTC")
    return {
        "commits": commits,
        "changes": changes,
        "sonar_issues": sonar_issues,
        "sonar_measures": sonar_measures,
        "szz": szz,
        "jira": jira,
    }


def _slice_for_project(data: dict[str, pd.DataFrame], pid: str) -> dict[str, pd.DataFrame]:
    """Return per-project slices of every cleaned frame.

    The feature builders all filter on ``PROJECT_ID`` internally, so
    pre-slicing here is purely a memory and pickling optimisation:
    each worker receives only its project's data instead of all 22.
    """
    out: dict[str, pd.DataFrame] = {}
    for name, df in data.items():
        if "PROJECT_ID" in df.columns:
            out[name] = df[df["PROJECT_ID"] == pid].copy()
        else:
            out[name] = df
    return out


def _one_project(pid: str, t: pd.Timestamp, sliced: dict[str, pd.DataFrame]) -> dict[str, Any]:
    """Compute all four feature families for a single project.

    Pure function: no shared mutable state, no I/O. Safe to invoke in
    parallel worker processes via joblib.
    """
    ts = time.time()
    static_df = build_static_features_for_project(
        pid, t, sliced["sonar_issues"], sliced["sonar_measures"], sliced["changes"]
    )
    hist_df = build_historical_features_for_project(
        pid, t, sliced["commits"], sliced["changes"]
    )
    graph_df = build_graph_features_for_project(
        pid, t, sliced["changes"], verbose=False
    )
    prior_df = build_priordefect_features_for_project(
        pid, t, sliced["commits"], sliced["changes"], sliced["szz"], sliced["jira"]
    )
    elapsed = time.time() - ts

    summary_row = {
        "project_id": pid,
        "n_basenames": len(static_df),
        "static_cols": len(static_df.columns),
        "hist_cols": len(hist_df.columns),
        "graph_cols": len(graph_df.columns),
        "prior_cols": len(prior_df.columns),
        "mean_n_code_smells": round(float(static_df["n_code_smells"].mean()), 2)
        if "n_code_smells" in static_df.columns and len(static_df)
        else 0.0,
        "mean_total_commits_pre": round(float(hist_df["total_commits_pre"].mean()), 2)
        if "total_commits_pre" in hist_df.columns and len(hist_df)
        else 0.0,
        "mean_cocg_degree": round(float(graph_df["cocg_degree"].mean()), 2)
        if "cocg_degree" in graph_df.columns and len(graph_df)
        else 0.0,
        "mean_bugfix_pre": round(float(prior_df["bugfix_commits_pre"].mean()), 2)
        if "bugfix_commits_pre" in prior_df.columns and len(prior_df)
        else 0.0,
        "elapsed_s": round(elapsed, 2),
    }

    print(
        f"   done {pid:<35}  N={len(static_df):>6}  "
        f"static={len(static_df.columns):>3}  "
        f"hist={len(hist_df.columns):>3}  "
        f"graph={len(graph_df.columns):>3}  "
        f"prior={len(prior_df.columns):>3}  "
        f"({elapsed:.1f}s)",
        flush=True,
    )

    return {
        "pid": pid,
        "static": static_df,
        "hist": hist_df,
        "graph": graph_df,
        "prior": prior_df,
        "summary": summary_row,
    }


def run_stage_5() -> None:
    t0 = time.time()
    snaps = load_snapshots(PROCESSED_DATA_DIR / "project_snapshots.parquet")
    eligible = snaps[snaps["eligible"]].copy()
    print(f"[Stage 5] Eligible projects: {len(eligible)}", flush=True)
    print(f"[Stage 5] Parallel workers (STAGE5_N_JOBS): {STAGE5_N_JOBS}", flush=True)

    print("[Stage 5] Loading cleaned parquets ...", flush=True)
    data = _load_clean()
    print(f"   commits        : {len(data['commits']):>10,}", flush=True)
    print(f"   changes        : {len(data['changes']):>10,}", flush=True)
    print(f"   sonar_issues   : {len(data['sonar_issues']):>10,}", flush=True)
    print(f"   sonar_measures : {len(data['sonar_measures']):>10,}", flush=True)
    print(f"   szz            : {len(data['szz']):>10,}", flush=True)
    print(f"   jira           : {len(data['jira']):>10,}", flush=True)

    eligible_sorted = eligible.sort_values("project_id").reset_index(drop=True)
    n_total = len(eligible_sorted)

    print("[Stage 5] Pre-slicing data per project ...", flush=True)
    tasks: list[tuple[str, pd.Timestamp, dict[str, pd.DataFrame]]] = []
    for _, row in eligible_sorted.iterrows():
        pid = row["project_id"]
        t = row["snapshot_date"]
        sliced = _slice_for_project(data, pid)
        n_pid_changes = len(sliced["changes"])
        tasks.append((pid, t, sliced))
        print(
            f"   queued {pid:<35} (snapshot={t:%Y-%m-%d}, "
            f"pre-snap changes={n_pid_changes:,})",
            flush=True,
        )
    # Drop the global frames once all per-project slices are built;
    # the workers no longer need them and this halves peak RAM.
    del data

    print(
        f"[Stage 5] Computing features in parallel "
        f"(n_jobs={STAGE5_N_JOBS}, n_projects={n_total}) ...",
        flush=True,
    )
    results = Parallel(n_jobs=STAGE5_N_JOBS, verbose=5)(
        delayed(_one_project)(pid, t, sliced) for (pid, t, sliced) in tasks
    )

    # Sort by project_id to make the concatenated outputs deterministic
    # regardless of worker completion order.
    results.sort(key=lambda r: r["pid"])

    static_all = pd.concat([r["static"] for r in results], ignore_index=True)
    hist_all = pd.concat([r["hist"] for r in results], ignore_index=True)
    graph_all = pd.concat([r["graph"] for r in results], ignore_index=True)
    prior_all = pd.concat([r["prior"] for r in results], ignore_index=True)

    static_all.to_parquet(PROCESSED_DATA_DIR / "features_static.parquet", index=False)
    hist_all.to_parquet(PROCESSED_DATA_DIR / "features_historical.parquet", index=False)
    graph_all.to_parquet(PROCESSED_DATA_DIR / "features_graph.parquet", index=False)
    prior_all.to_parquet(PROCESSED_DATA_DIR / "features_priordefect.parquet", index=False)

    summary_df = pd.DataFrame([r["summary"] for r in results])
    summary_df.to_csv(TABLES_DIR / "feature_summary.csv", index=False)

    print("\n[Stage 5] Feature summary:")
    with pd.option_context("display.width", 200, "display.max_rows", None):
        print(summary_df.to_string(index=False))

    print(
        f"\n[Stage 5] Total rows - static: {len(static_all):,} | "
        f"historical: {len(hist_all):,} | graph: {len(graph_all):,} | "
        f"prior_defect: {len(prior_all):,}"
    )
    print(f"[Stage 5] Static columns      : {list(static_all.columns)[:12]} ... total {len(static_all.columns)}")
    print(f"[Stage 5] Historical columns  : {list(hist_all.columns)[:12]} ... total {len(hist_all.columns)}")
    print(f"[Stage 5] Graph columns       : {list(graph_all.columns)[:12]} ... total {len(graph_all.columns)}")
    print(f"[Stage 5] Prior-defect columns: {list(prior_all.columns)[:12]} ... total {len(prior_all.columns)}")
    print(f"\n[Stage 5] Elapsed: {time.time() - t0:.1f}s")
    print("[Stage 5] Complete.")


In [ ]:
run_stage_5()
print()
print('--- feature_summary.csv ---')
show_table('results/tables/feature_summary.csv')


## Section 6 — Dataset Assembly

Merge features and labels into `dataset_final.parquet`. Apply `log1p` to
heavy-tailed columns and assert no leakage (raw severity counts and the
weight-derivation surrogate are absent). Writes the dataset and
`feature_catalog.csv`.

In [ ]:
# Pipeline code

"""
Stage 6 - Assemble the final modeling dataset.

Merges the four feature-family parquets with ``labels.parquet`` into a
single ``dataset_final.parquet`` with exactly 27 feature columns +
``is_high_risk`` (28 modelling columns) plus ``project_id`` and
``basename`` for downstream grouping.

Pre-processing:
    1. NaN -> 0 for every feature column
    2. log1p transform for heavy-tailed counts (LOG1P_FEATURES)

Scaling is NOT applied here: ``StandardScaler`` is fitted per CV fold
inside the training pipelines so it never leaks across folds.

Leakage assertions:
    - No raw severity counts (n_blocker / n_critical) in the matrix.
    - The weight-derivation surrogate (future_bugfix_count) is absent.

Outputs
-------
- ``data/processed/dataset_final.parquet``
- ``data/processed/feature_catalog.csv``
"""

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


KEY_COLS = ["project_id", "basename"]
LABEL_COL = "is_high_risk"


FEATURE_CATALOG_ROWS = [
    # Family 1: Size / Complexity (project-level context replicated per file)
    ("ncloc", "size_complexity", "SONAR_MEASURES",
     "Nagappan ICSE 2006",
     "Larger files accumulate more debt opportunities"),
    ("complexity", "size_complexity", "SONAR_MEASURES",
     "McCabe 1976",
     "Cyclomatic complexity increases change effort and error-proneness"),
    ("cognitive_complexity", "size_complexity", "SONAR_MEASURES",
     "Campbell 2018 (SonarSource)",
     "Human-perceived complexity; hard to read = hard to maintain"),
    ("functions", "size_complexity", "SONAR_MEASURES",
     "Nagappan ICSE 2006",
     "Method count - more methods = more potential debt entry points"),
    ("classes", "size_complexity", "SONAR_MEASURES",
     "Nagappan ICSE 2006",
     "Class count - captures OO design scale"),

    # Family 2: Static debt
    ("n_code_smells", "static_debt", "SONAR_ISSUES",
     "Tsoukalas JSS 2020",
     "Maintainability violations (CODE_SMELL type)"),
    ("n_bugs", "static_debt", "SONAR_ISSUES",
     "Tsoukalas JSS 2020",
     "BUG-type issues regardless of severity"),
    ("total_debt_minutes", "static_debt", "SONAR_ISSUES",
     "Tsoukalas JSS 2020",
     "Remediation effort estimate; debt principal in minutes"),
    ("issue_density", "static_debt", "SONAR_ISSUES / SONAR_MEASURES",
     "Tsoukalas JSS 2020",
     "total_issues / ncloc; size-normalized debt intensity"),
    ("duplicated_lines_density", "static_debt", "SONAR_MEASURES",
     "Fowler 1999",
     "Duplication raises cost of propagating fixes"),

    # Family 3: Historical change
    ("total_commits_pre", "historical", "GIT_COMMITS",
     "Kamei TSE 2013",
     "Total commits touching the file before t"),
    ("code_churn_pre", "historical", "GIT_COMMITS_CHANGES",
     "Kamei TSE 2013",
     "Lifetime lines added + removed; total volatility"),
    ("recent_churn_90d", "historical", "GIT_COMMITS_CHANGES",
     "Hassan ICSE 2009",
     "Churn in last 90 days; current hotspot signal"),
    ("commit_frequency_30d", "historical", "GIT_COMMITS",
     "Hassan ICSE 2009",
     "Commits in last 30 days; recent activity level"),
    ("file_age_days", "historical", "GIT_COMMITS",
     "Kamei TSE 2013",
     "Days from first commit to t; older files carry more debt"),
    ("days_since_last_change", "historical", "GIT_COMMITS",
     "Kamei TSE 2013",
     "Days from last commit to t; stale files may need attention"),
    ("contributor_count", "historical", "GIT_COMMITS",
     "Bird FSE 2011",
     "Distinct authors before t; coordination overhead risk"),
    ("ownership_ratio", "historical", "GIT_COMMITS",
     "Bird FSE 2011",
     "max_single_author_commits / total_commits_pre; diffuse responsibility risk"),

    # Family 4: Co-change graph
    ("cocg_degree", "graph", "GIT_COMMITS_CHANGES (co-change graph)",
     "Jiang EMSE 2024",
     "Number of co-change neighbours; architectural coupling"),
    ("cocg_pagerank", "graph", "GIT_COMMITS_CHANGES (co-change graph)",
     "Jiang EMSE 2024",
     "Recursive importance via weighted PageRank"),
    ("cocg_betweenness", "graph", "GIT_COMMITS_CHANGES (co-change graph)",
     "Jiang EMSE 2024",
     "Bridge status; changes here ripple to many modules"),
    ("cocg_entropy", "graph", "GIT_COMMITS_CHANGES (co-change graph)",
     "Ethari & Bhardwaj 2025",
     "Shannon entropy over neighbour edge weights; high = scattered coupling"),

    # Family 5: Prior defect
    ("bugfix_commits_pre", "prior_defect", "GIT_COMMITS (regex)",
     "Hassan ICSE 2009",
     "Lifetime bug-fix commit count using the canonical regex"),
    ("bugfix_commits_90d", "prior_defect", "GIT_COMMITS (regex)",
     "Hassan ICSE 2009",
     "Bug-fix commits last 90 days; recent defect activity"),
    ("bug_density_pre", "prior_defect", "GIT_COMMITS (regex)",
     "Hassan ICSE 2009",
     "bugfix_commits_pre / total_commits_pre; normalized"),
    ("n_jira_bugs_pre", "prior_defect", "JIRA_ISSUES",
     "Falessi ESEM 2020",
     "Officially confirmed bug tickets linked to the file before t"),
    ("jira_blocker_flag", "prior_defect", "JIRA_ISSUES",
     "Falessi ESEM 2020",
     "Flag for any linked JIRA Bug with PRIORITY Blocker or Critical"),
]


def _load_features() -> pd.DataFrame:
    static = pd.read_parquet(PROCESSED_DATA_DIR / "features_static.parquet")
    hist = pd.read_parquet(PROCESSED_DATA_DIR / "features_historical.parquet")
    graph = pd.read_parquet(PROCESSED_DATA_DIR / "features_graph.parquet")
    prior = pd.read_parquet(PROCESSED_DATA_DIR / "features_priordefect.parquet")

    # Drop helper columns that may still be present.
    for df in (static, hist, graph, prior):
        for col in ("snapshot_date",):
            if col in df.columns:
                df.drop(columns=[col], inplace=True)

    out = static.merge(hist, on=KEY_COLS, how="outer")
    out = out.merge(graph, on=KEY_COLS, how="outer")
    out = out.merge(prior, on=KEY_COLS, how="outer")
    return out


def run_stage_6() -> None:
    t0 = time.time()
    print("[Stage 6] Loading feature parquets ...")
    features = _load_features()
    labels = pd.read_parquet(PROCESSED_DATA_DIR / "labels.parquet")[
        KEY_COLS + [LABEL_COL]
    ]
    print(f"   features rows={len(features):,}  cols={len(features.columns)}")
    print(f"   labels   rows={len(labels):,}  positive_rate={100*labels[LABEL_COL].mean():.2f}%")

    df = features.merge(labels, on=KEY_COLS, how="inner")
    print(f"[Stage 6] After merge: rows={len(df):,}")

    # ----- Restrict to the 27 declared features -----
    missing = [c for c in ALL_FEATURES if c not in df.columns]
    if missing:
        raise KeyError(f"Stage 6: missing expected feature columns: {missing}")
    df = df[KEY_COLS + ALL_FEATURES + [LABEL_COL]].copy()

    # ----- NaN -> 0 across the 27 features -----
    df[ALL_FEATURES] = df[ALL_FEATURES].fillna(0)

    # ----- log1p heavy-tailed columns -----
    print(f"[Stage 6] Applying log1p to {len(LOG1P_FEATURES)} columns ...")
    for col in LOG1P_FEATURES:
        df[col] = np.log1p(df[col].astype(float))

    # ----- Leakage assertions -----
    assert "n_blocker" not in df.columns and "n_critical" not in df.columns, (
        "Severity leakage: raw severity counts found in feature matrix"
    )
    assert "future_bugfix_count" not in df.columns, (
        "Surrogate leakage: future data found in feature matrix"
    )
    feat_cols = [c for c in df.columns if c not in KEY_COLS + [LABEL_COL]]
    assert len(feat_cols) == 27, f"expected 27 feature cols, got {len(feat_cols)}: {feat_cols}"
    assert df.isna().sum().sum() == 0, "NaN detected in dataset_final after preprocessing"

    # ----- Persist -----
    out_path = PROCESSED_DATA_DIR / "dataset_final.parquet"
    df.to_parquet(out_path, index=False)
    print(f"[Stage 6] Wrote {out_path}  rows={len(df):,}  cols={len(df.columns)}")

    # ----- Feature catalog -----
    catalog = pd.DataFrame(
        FEATURE_CATALOG_ROWS,
        columns=["feature_name", "family", "source_table", "literature_citation", "rationale"],
    )
    assert len(catalog) == 27, f"feature_catalog has {len(catalog)} rows, expected 27"
    # Sanity: every catalog row corresponds to a column in the dataset
    miss = set(catalog["feature_name"]) - set(df.columns)
    assert not miss, f"catalog features not in dataset: {miss}"
    catalog.to_csv(PROCESSED_DATA_DIR / "feature_catalog.csv", index=False)
    print(f"[Stage 6] Wrote feature_catalog.csv ({len(catalog)} features)")

    # ----- Per-family summary log -----
    print("\n[Stage 6] Feature family breakdown:")
    for fam, cols in FEATURE_FAMILIES.items():
        nz = (df[cols] != 0).any(axis=1).mean() * 100
        print(f"   {fam:<16}  {len(cols)} features  non_zero_any={nz:.1f}%")

    # ----- Prior-defect coverage diagnostic -----
    # The prior-defect family depends on JIRA and the bug-fix regex
    # matching commits. Projects with sparse JIRA linkage or terse commit
    # messages may have most rows all-zero across this family - good to
    # know before reading family ablation results.
    prior_cols = FEATURE_FAMILIES["prior_defect"]
    pf_rows = []
    for pid, sub in df.groupby("project_id"):
        all_zero = (sub[prior_cols] == 0).all(axis=1)
        pf_rows.append(
            {
                "project_id": pid,
                "n_files": int(len(sub)),
                "n_all_zero_prior_defect": int(all_zero.sum()),
                "pct_all_zero": round(100 * float(all_zero.mean()), 2),
            }
        )
    pf_coverage = pd.DataFrame(pf_rows).sort_values("project_id").reset_index(drop=True)
    pf_coverage.to_csv(PROCESSED_DATA_DIR / "prior_defect_coverage.csv", index=False)

    high_zero = pf_coverage[pf_coverage["pct_all_zero"] > 80]
    if len(high_zero) > 0:
        print(f"\n[Stage 6] WARNING: {len(high_zero)} project(s) have > 80% rows with all-zero prior-defect features:")
        print(high_zero.to_string(index=False))

    pos_rate = 100 * df[LABEL_COL].mean()
    print(f"\n[Stage 6] Final positive rate : {pos_rate:.2f}%  ({int(df[LABEL_COL].sum()):,} of {len(df):,})")

    # Also emit a tiny tables/ ack so the verifier can find the summary.
    TABLES_DIR.mkdir(parents=True, exist_ok=True)

    print(f"\n[Stage 6] Elapsed: {time.time() - t0:.1f}s")
    print("[Stage 6] Complete.")


In [ ]:
run_stage_6()
print()
ds = pd.read_parquet('data/processed/dataset_final.parquet')
print(f'shape: {ds.shape}   positive rate: {100*ds["is_high_risk"].mean():.2f}%')

# fig_04 is produced later by 10_report.py (Section 12); show if available.
show_figure('results/figures/fig_04_feature_correlation_heatmap.png')

# Drive checkpoint
shutil.copytree('data/processed', f'{DRIVE_BASE}/data/processed', dirs_exist_ok=True)
print('\nDataset checkpointed to Drive.')


## Section 7 — Default Model Training

Train all 4 models with default hyperparameters under stratified 10-fold
within-project CV as a pre-tuning baseline. Writes `default_cv_results.csv`.
processed data from Drive if the local copy is missing.

In [ ]:
# Resume guard — load processed data from Drive if missing locally
if not Path('data/processed/dataset_final.parquet').exists():
    print('Local data missing. Loading from Drive...')
    shutil.copytree(f'{DRIVE_BASE}/data/processed', 'data/processed', dirs_exist_ok=True)
    print('Loaded from Drive.')
else:
    print('Local data found. Proceeding.')


In [ ]:
# Pipeline code

"""
Within-project stratified K-fold training core.

Builds models, runs K-fold CV, computes the four-metric battery
(F1, ROC-AUC, PR-AUC, CE@20) at a fixed threshold of 0.5. Two
regimes are supported:

- ``stratified_kfold_cv(..., params=None)`` -- default hyperparams.
- ``stratified_kfold_cv(..., params=<dict>)`` -- tuned hyperparams from
  the Optuna driver (scripts/07b_tune.py).

Threshold policy: classification uses the standard 0.5 cutoff on
``predict_proba``. Tree ensembles with ``class_weight='balanced'``
(or ``scale_pos_weight`` for XGBoost) are already well-calibrated at
the 17% positive rate of this dataset; per-fold threshold sweeping
added complexity without measurable F1 gain in our pilot
(within-project F1 deltas were +0.002 / -0.002 / -0.012 / +0.034
for xgboost / lightgbm / random_forest / logistic_regression), and
the negative deltas mean per-fold threshold variance was hurting
the ensembles. We therefore report F1 at the unambiguous 0.5
threshold.
"""

import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


CLASSIFICATION_THRESHOLD = 0.5


KEY_COLS = ("project_id", "basename")
LABEL_COL = "is_high_risk"

MODEL_NAMES = ("logistic_regression", "random_forest", "xgboost", "lightgbm")


# ---------------------------------------------------------------------------
# Model zoo (4 models)
# ---------------------------------------------------------------------------
def _merged_params(defaults: dict[str, Any], overrides: Optional[dict[str, Any]]) -> dict[str, Any]:
    out = dict(defaults)
    if overrides:
        out.update(overrides)
    return out


def _make_model(name: str, params: Optional[dict[str, Any]] = None, *, y_train: Optional[np.ndarray] = None):
    """Build an sklearn estimator by name.

    Parameters
    ----------
    name : one of MODEL_NAMES.
    params : optional overrides merged over the estimator defaults.
    y_train : optional training labels used to compute ``scale_pos_weight``
        for XGBoost (``n_neg / n_pos``). Only used when ``name == "xgboost"``.

    Logistic regression is wrapped in a Pipeline with StandardScaler so
    per-fold scaling is honest. Tree ensembles don't need scaling but
    are returned bare for speed.
    """
    name = name.lower()
    if name == "logistic_regression":
        defaults = {
            "class_weight": "balanced",
            "max_iter": 1000,
            "random_state": RANDOM_STATE,
            "solver": "lbfgs",
        }
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(**_merged_params(defaults, params))),
            ]
        )
    if name == "random_forest":
        defaults = {
            "class_weight": "balanced",
            "n_jobs": -1,
            "random_state": RANDOM_STATE,
        }
        return RandomForestClassifier(**_merged_params(defaults, params))
    if name == "xgboost":
        try:
            from xgboost import XGBClassifier
        except ImportError:
            return None
        defaults = {
            "eval_metric": "aucpr",
            "random_state": RANDOM_STATE,
            "verbosity": 0,
            "n_jobs": -1,
        }
        if y_train is not None:
            n_pos = int(np.sum(y_train == 1))
            n_neg = int(np.sum(y_train == 0))
            if n_pos > 0:
                defaults["scale_pos_weight"] = float(n_neg) / float(n_pos)
        return XGBClassifier(**_merged_params(defaults, params))
    if name == "lightgbm":
        try:
            from lightgbm import LGBMClassifier
        except ImportError:
            return None
        defaults = {
            "class_weight": "balanced",
            "verbose": -1,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        return LGBMClassifier(**_merged_params(defaults, params))
    raise ValueError(f"Unknown model: {name}")


# ---------------------------------------------------------------------------
# Four-metric battery
# ---------------------------------------------------------------------------
def _recall_at_top_20(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Recall captured in the top-20% highest-scoring rows (CE@20)."""
    n = len(y_true)
    total_pos = int(np.asarray(y_true).sum())
    if n == 0 or total_pos == 0:
        return 0.0
    top_k = max(1, int(n * COST_EFFECTIVENESS_AT))
    idx = np.argsort(y_prob)[::-1]
    top_labels = np.asarray(y_true)[idx[:top_k]]
    return float(top_labels.sum() / total_pos)


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> dict[str, float]:
    """Return F1 (at 0.5), ROC-AUC, PR-AUC, CE@20."""
    y_true = np.asarray(y_true)
    y_pred = (y_prob >= CLASSIFICATION_THRESHOLD).astype(int)
    if y_pred.sum() < 1 or y_true.sum() < 1:
        return {"f1": 0.0, "roc_auc": 0.5, "pr_auc": 0.0, "ce_at_20": 0.0}
    return {
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "ce_at_20": _recall_at_top_20(y_true, y_prob),
    }


# ---------------------------------------------------------------------------
# Data prep
# ---------------------------------------------------------------------------
def load_dataset() -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Return (X, y, project_id) from dataset_final.parquet.

    Uses the 27 feature columns declared in config.ALL_FEATURES in that
    exact order so train and inference paths agree.
    """
    df = pd.read_parquet(PROCESSED_DATA_DIR / "dataset_final.parquet")
    missing = [c for c in ALL_FEATURES if c not in df.columns]
    if missing:
        raise KeyError(f"dataset_final.parquet missing expected features: {missing}")
    X = df[list(ALL_FEATURES)].copy()
    y = df[LABEL_COL].astype(int)
    proj = df["project_id"]
    return X, y, proj


# ---------------------------------------------------------------------------
# K-fold driver with threshold optimisation
# ---------------------------------------------------------------------------
@dataclass
class FoldResult:
    model: str
    fold: int
    n_train: int
    n_test: int
    n_pos_test: int
    metrics: dict[str, float] = field(default_factory=dict)


def _fit_and_score_fold(
    model_name: str,
    X_tr: pd.DataFrame,
    y_tr: np.ndarray,
    X_te: pd.DataFrame,
    y_te: np.ndarray,
    params: Optional[dict[str, Any]],
) -> dict[str, float]:
    """Fit on (X_tr, y_tr); return metric battery on (X_te, y_te) at threshold 0.5."""
    est = _make_model(model_name, params, y_train=y_tr)
    if est is None:
        raise RuntimeError(f"Model {model_name!r} not available (import failed)")
    est.fit(X_tr, y_tr)
    if hasattr(est, "predict_proba"):
        test_probs = est.predict_proba(X_te)[:, 1]
    else:
        test_probs = est.decision_function(X_te)
    return compute_metrics(y_te, test_probs)


def stratified_kfold_cv(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_FOLDS,
    *,
    params: Optional[dict[str, Any]] = None,
) -> list[FoldResult]:
    """Run stratified K-fold CV for one model. F1 at the standard 0.5 threshold."""
    yv = np.asarray(y, dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    out: list[FoldResult] = []
    for fold, (tr, te) in enumerate(skf.split(X, yv), start=1):
        X_tr = X.iloc[tr]
        X_te = X.iloc[te]
        y_tr = yv[tr]
        y_te = yv[te]
        try:
            metrics = _fit_and_score_fold(model_name, X_tr, y_tr, X_te, y_te, params)
        except RuntimeError:
            return []
        out.append(
            FoldResult(
                model=model_name,
                fold=fold,
                n_train=len(tr),
                n_test=len(te),
                n_pos_test=int(y_te.sum()),
                metrics=metrics,
            )
        )
    return out


def fold_results_to_frame(results: Iterable[FoldResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        rows.append(
            {
                "model": r.model,
                "fold": r.fold,
                "n_train": r.n_train,
                "n_test": r.n_test,
                "n_pos_test": r.n_pos_test,
                "f1": r.metrics["f1"],
                "roc_auc": r.metrics["roc_auc"],
                "pr_auc": r.metrics["pr_auc"],
                "ce_at_20": r.metrics["ce_at_20"],
            }
        )
    return pd.DataFrame(rows)


def summarize_by_model(fold_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-fold rows into mean +/- std per model."""
    metric_cols = ["f1", "roc_auc", "pr_auc", "ce_at_20"]
    grp = fold_df.groupby("model")[metric_cols]
    means = grp.mean().add_suffix("_mean")
    stds = grp.std().add_suffix("_std")
    return pd.concat([means, stds], axis=1).reset_index()


"""
Stage 7 - Within-project stratified 10-fold CV with threshold optimisation.

Two modes:

- ``python scripts/07_train.py``            -> Stage A (default hyperparams)
    Writes ``results/tables/default_cv_results.csv``.

- ``python scripts/07_train.py --tuned``    -> Stage C (tuned hyperparams)
    Reads tuned params from ``results/tables/tuned_hyperparameters.csv``
    and rewrites ``results/tables/within_project_results.csv``.

Both modes use threshold optimisation per fold (see Part 6 of the spec).
"""

import json
import sys
import time
from pathlib import Path

import pandas as pd


def _load_tuned_params() -> dict[str, dict]:
    """Return {model_name -> best_params dict} from the tuning CSV."""
    path = TABLES_DIR / "tuned_hyperparameters.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run scripts/07b_tune.py before --tuned mode."
        )
    df = pd.read_csv(path)
    out: dict[str, dict] = {}
    for _, row in df.iterrows():
        out[row["model"]] = json.loads(row["best_params_json"])
    return out


def run_stage_7(tuned: bool = False) -> None:
    class _A: pass
    args = _A(); args.tuned = tuned

    t0 = time.time()
    print(f"[Stage 7] Mode      : {'tuned' if args.tuned else 'default'}")
    print(f"[Stage 7] CV folds  : {CV_FOLDS}")
    print(f"[Stage 7] Models    : {list(MODEL_ORDER)}")

    X, y, _ = load_dataset()
    pos_rate = 100 * float(y.mean())
    print(f"[Stage 7] Dataset   : rows={len(X):,}  feats={X.shape[1]}  positives={int(y.sum())} ({pos_rate:.2f}%)")

    tuned_params = _load_tuned_params() if args.tuned else {}
    all_folds: list[pd.DataFrame] = []
    for model_name in MODEL_ORDER:
        t1 = time.time()
        params = tuned_params.get(model_name) if args.tuned else None
        results = stratified_kfold_cv(model_name, X, y, n_splits=CV_FOLDS, params=params)
        if not results:
            print(f"   {model_name:<22}  SKIPPED (estimator unavailable)")
            continue
        df = fold_results_to_frame(results)
        means = df[["f1", "roc_auc", "pr_auc", "ce_at_20"]].mean()
        print(
            f"   {model_name:<22}  "
            f"F1={means['f1']:.3f}  "
            f"ROC={means['roc_auc']:.3f}  "
            f"PR={means['pr_auc']:.3f}  "
            f"CE@20={means['ce_at_20']:.3f}  "
            f"({time.time() - t1:.1f}s)"
        )
        all_folds.append(df)

    fold_df = pd.concat(all_folds, ignore_index=True)
    out_name = "within_project_results.csv" if args.tuned else "default_cv_results.csv"
    fold_df.to_csv(TABLES_DIR / out_name, index=False)
    print(f"\n[Stage 7] Wrote {TABLES_DIR / out_name}  ({len(fold_df)} rows)")

    summary = summarize_by_model(fold_df)
    print("\n[Stage 7] Per-model summary (mean across folds):")
    with pd.option_context("display.width", 220, "display.max_rows", None):
        cols_show = ["model", "f1_mean", "roc_auc_mean", "pr_auc_mean", "ce_at_20_mean"]
        cols_show = [c for c in cols_show if c in summary.columns]
        print(summary[cols_show].round(3).to_string(index=False))

    print(f"\n[Stage 7] Elapsed: {time.time() - t0:.1f}s")
    print("[Stage 7] Complete.")


In [ ]:
run_stage_7(tuned=False)
print()
print('--- default_cv_results.csv (per-fold summary) ---')
df = pd.read_csv('results/tables/default_cv_results.csv')
summary = df.groupby('model')[['f1','roc_auc','pr_auc','ce_at_20']].mean().round(4).reset_index()
display(summary.style.highlight_max(subset=['f1'], color='lightgreen'))


## Section 8 — Hyperparameter Tuning

Tune all 4 models with Optuna (30 trials each, PR-AUC objective, 5-fold
inner CV); logistic regression uses GridSearchCV over `C`. Writes
`tuned_hyperparameters.csv`.
step in the pipeline. Do not close the browser tab during tuning.

In [ ]:
# Pipeline code

"""
Optuna hyperparameter tuning for the 4-model pipeline (Stage 7b).

Search spaces match Part 5 of the rebuild spec exactly. Objective:
mean PR-AUC over a stratified ``TUNING_INNER_CV_FOLDS``-fold inner CV
disjoint from the outer 10-fold validation in 07_train.py.

PR-AUC is the recommended objective for class-imbalanced binary
classification (Saito & Rehmsmeier 2015); ROC-AUC is optimistic when
positive rate < 25%.

Logistic regression uses a small grid (``C in {0.01, 0.1, 1, 10, 100}``)
per spec; tree ensembles use Optuna TPE with the spec's exact ranges.
"""

import sys
import time
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold


# ---------------------------------------------------------------------------
# Search spaces (verbatim from Part 5)
# ---------------------------------------------------------------------------
def _suggest_params(trial, model_name: str) -> dict[str, Any]:
    name = model_name.lower()
    if name == "random_forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
            "max_depth": trial.suggest_int("max_depth", 5, 25),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.3]),
        }
    if name == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        }
    if name == "lightgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
            "num_leaves": trial.suggest_int("num_leaves", 20, 100),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        }
    raise ValueError(f"No Optuna space defined for model: {model_name}")


# ---------------------------------------------------------------------------
# Inner-CV PR-AUC objective
# ---------------------------------------------------------------------------
def _objective_factory(model_name: str, X: pd.DataFrame, y: np.ndarray, n_splits: int):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial) -> float:
        params = _suggest_params(trial, model_name)
        scores: list[float] = []
        for tr, te in skf.split(X, y):
            X_tr = X.iloc[tr]
            X_te = X.iloc[te]
            y_tr = y[tr]
            est = _make_model(model_name, params, y_train=y_tr)
            if est is None:
                return float("-inf")
            est.fit(X_tr, y_tr)
            if hasattr(est, "predict_proba"):
                proba = est.predict_proba(X_te)[:, 1]
            else:
                proba = est.decision_function(X_te)
            try:
                scores.append(float(average_precision_score(y[te], proba)))
            except ValueError:
                scores.append(0.0)
        return float(np.mean(scores))

    return objective


# ---------------------------------------------------------------------------
# Public tuner
# ---------------------------------------------------------------------------
def tune_model(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_trials: int = TUNING_TRIALS,
    inner_splits: int = TUNING_INNER_CV_FOLDS,
    timeout_s: Optional[int] = None,
) -> dict[str, Any]:
    """Run an Optuna study (or GridSearchCV for LR); return best params + diagnostics.

    Returns
    -------
    dict with keys: model, best_params, best_pr_auc, n_trials_completed,
                    n_trials_requested, inner_splits, elapsed_s, method.
    """
    name = model_name.lower()
    yv = np.asarray(y, dtype=int)

    if name == "logistic_regression":
        return _tune_logistic_grid(X, yv, inner_splits)

    import optuna  # local import; cheap cold-start

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    objective = _objective_factory(model_name, X, yv, inner_splits)

    t0 = time.time()
    study.optimize(objective, n_trials=n_trials, timeout=timeout_s, show_progress_bar=False)
    elapsed = time.time() - t0

    best = study.best_trial
    return {
        "model": model_name,
        "best_params": dict(best.params),
        "best_pr_auc": float(best.value) if best.value is not None else float("nan"),
        "n_trials_completed": len(study.trials),
        "n_trials_requested": n_trials,
        "inner_splits": inner_splits,
        "elapsed_s": round(elapsed, 2),
        "method": "optuna_tpe",
    }


def _tune_logistic_grid(X: pd.DataFrame, y: np.ndarray, inner_splits: int) -> dict[str, Any]:
    """Logistic regression via GridSearchCV on C in {0.01, 0.1, 1, 10, 100} per spec."""
    t0 = time.time()
    est = _make_model("logistic_regression")
    # _make_model wraps LR in a Pipeline named ("scaler", "clf"); GridSearchCV
    # needs "clf__C" to address the inner classifier.
    param_grid = {"clf__C": [0.01, 0.1, 1, 10, 100]}
    gs = GridSearchCV(
        est,
        param_grid=param_grid,
        scoring="average_precision",
        cv=StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
        refit=False,
    )
    gs.fit(X, y)
    elapsed = time.time() - t0
    best_c = float(gs.best_params_["clf__C"])
    return {
        "model": "logistic_regression",
        "best_params": {"C": best_c},
        "best_pr_auc": float(gs.best_score_),
        "n_trials_completed": len(gs.cv_results_["params"]),
        "n_trials_requested": len(gs.cv_results_["params"]),
        "inner_splits": inner_splits,
        "elapsed_s": round(elapsed, 2),
        "method": "grid_search",
    }


"""
Stage 7b - Hyperparameter tuning for the 4 models.

For each model in config.MODEL_ORDER, run an inner-CV PR-AUC tuning
search (Optuna TPE for tree ensembles; GridSearchCV for logistic
regression). Writes one row per model to
``results/tables/tuned_hyperparameters.csv`` with the JSON-encoded
best params.
"""

import json
import sys
import time
from pathlib import Path

import pandas as pd


def run_stage_7b() -> None:
    t0 = time.time()
    print(f"[Stage 7b] Models       : {list(MODEL_ORDER)}")
    print(f"[Stage 7b] Trials/model : {TUNING_TRIALS} (LR uses grid search)")
    print(f"[Stage 7b] Inner folds  : {TUNING_INNER_CV_FOLDS}")
    print(f"[Stage 7b] Objective    : PR-AUC (Saito & Rehmsmeier 2015)")

    X, y, _ = load_dataset()
    print(f"[Stage 7b] Dataset      : rows={len(X):,}  feats={X.shape[1]}  positives={int(y.sum())}")

    rows: list[dict] = []
    for model_name in MODEL_ORDER:
        t1 = time.time()
        print(f"\n[Stage 7b] Tuning {model_name} ...")
        try:
            result = tune_model(model_name, X, y)
        except Exception as e:
            print(f"   FAILED: {type(e).__name__}: {e}")
            continue
        print(
            f"   best_pr_auc={result['best_pr_auc']:.4f}  "
            f"trials={result['n_trials_completed']}  "
            f"method={result['method']}  "
            f"elapsed={result['elapsed_s']}s"
        )
        print(f"   best_params: {result['best_params']}")
        rows.append(
            {
                "model": result["model"],
                "method": result["method"],
                "best_pr_auc": round(result["best_pr_auc"], 6),
                "best_params_json": json.dumps(result["best_params"]),
                "n_trials_completed": result["n_trials_completed"],
                "n_trials_requested": result["n_trials_requested"],
                "inner_splits": result["inner_splits"],
                "elapsed_s": result["elapsed_s"],
            }
        )

    df = pd.DataFrame(rows)
    out_path = TABLES_DIR / "tuned_hyperparameters.csv"
    df.to_csv(out_path, index=False)
    print(f"\n[Stage 7b] Wrote {out_path}")
    print(f"[Stage 7b] Total elapsed: {time.time() - t0:.1f}s")
    print("[Stage 7b] Complete.")


In [ ]:
run_stage_7b()
print()
print('--- tuned_hyperparameters.csv ---')
show_table('results/tables/tuned_hyperparameters.csv')


## Section 9 — Tuned Within-Project Validation

Re-evaluate all 4 tuned models under stratified 10-fold within-project CV.
Writes `within_project_results.csv`.

In [ ]:
# Reuses run_stage_7 defined in Section 7
run_stage_7(tuned=True)
print()
print('--- within_project_results.csv (mean per model) ---')
df = pd.read_csv('results/tables/within_project_results.csv')
summary = df.groupby('model')[['f1','roc_auc','pr_auc','ce_at_20']].mean().round(4).reset_index()
display(summary.style.highlight_max(subset=['f1'], color='lightgreen'))

# fig_06 is produced later by 10_report.py (Section 12); show if available.
show_figure('results/figures/fig_06_model_comparison_f1.png')


## Section 10 — LOPO Cross-Project Validation

Leave-One-Project-Out validation across 22 projects with similarity-weighted
training (org.apache:daemon excluded from test, kept in train). This is the
primary evaluation — the model is tested on projects it has never seen.

Writes `lopo_results.csv`, `lopo_per_project.csv`, `model_comparison.csv`.

In [ ]:
# Resume guard
if not Path('data/processed/dataset_final.parquet').exists():
    print('Local data missing. Loading from Drive...')
    shutil.copytree(f'{DRIVE_BASE}/data/processed', 'data/processed', dirs_exist_ok=True)


In [ ]:
# Pipeline code

"""
Leave-One-Project-Out cross-project validation core.

For each held-out project ``p``:
  1. Compute project-level feature vectors ``[log(n_files),
     log(pre_commits), positive_rate]`` for every training project
     (positive_rate uses ONLY training labels - the held-out project
     never contributes to weight estimation).
  2. Cosine similarity between the held-out project's feature vector
     and each training project's vector yields a per-project weight in
     [0, 1]. Negative cosines are clipped to a small positive floor so
     dissimilar projects still contribute but with reduced influence.
  3. Broadcast the per-project weight to every training row of that
     project. The resulting ``sample_weight`` is one float per training
     row (shape == (n_train,)) and is passed to ``model.fit``.
  4. Classification uses the standard 0.5 threshold (see train.py
     docstring for rationale).
"""

import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, List, Optional

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


SIMILARITY_FLOOR = 0.05  # weight given to dissimilar projects (no zero-weight rows)


@dataclass
class LopoResult:
    model: str
    held_out_project: str
    n_train: int
    n_test: int
    n_pos_test: int
    metrics: dict[str, float] = field(default_factory=dict)
    note: str = ""  # e.g. "training_only" for daemon


# ---------------------------------------------------------------------------
# Project-level similarity weighting
# ---------------------------------------------------------------------------
def _project_level_features(
    X: pd.DataFrame,
    y: pd.Series,
    proj: pd.Series,
) -> pd.DataFrame:
    """Return a DataFrame indexed by project_id with columns
    ``[log_n_files, log_pre_commits, positive_rate]``.

    ``log_pre_commits`` uses ``total_commits_pre`` summed per project
    (already log1p'd in stage 6, so we exponentiate first then re-log
    after summing). ``positive_rate`` comes from the labels supplied.
    """
    df = X[["total_commits_pre"]].copy()
    df["project_id"] = proj.values
    df["y"] = np.asarray(y, dtype=int)
    # total_commits_pre is log1p-transformed in stage 6; reverse it for summing.
    df["raw_commits"] = np.expm1(df["total_commits_pre"])
    grp = df.groupby("project_id")
    out = pd.DataFrame(
        {
            "log_n_files": np.log1p(grp.size().astype(float)),
            "log_pre_commits": np.log1p(grp["raw_commits"].sum()),
            "positive_rate": grp["y"].mean(),
        }
    )
    return out


def _similarity_weights(
    train_proj_features: pd.DataFrame,
    test_features: np.ndarray,
) -> dict[str, float]:
    """Cosine similarity between test_features and each training project."""
    train_mat = train_proj_features.values
    sims = cosine_similarity(test_features.reshape(1, -1), train_mat)[0]
    sims = np.clip(sims, SIMILARITY_FLOOR, 1.0)
    return {pid: float(s) for pid, s in zip(train_proj_features.index, sims)}


def _build_sample_weights(
    train_proj: pd.Series,
    weights_by_project: dict[str, float],
) -> np.ndarray:
    """Broadcast per-project weight to one float per training row.

    Returns an ndarray of shape (n_train,). The caller asserts the
    shape before passing to ``model.fit(sample_weight=...)``.
    """
    return np.array([weights_by_project[pid] for pid in train_proj], dtype=float)


# ---------------------------------------------------------------------------
# Per-fold fit & score
# ---------------------------------------------------------------------------
def _fit_lopo_fold(
    model_name: str,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    proj_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    test_pid: str,
    proj_features_all: pd.DataFrame,
    params: Optional[dict[str, Any]],
) -> dict[str, float]:
    """Fit on training rows with similarity-weighted sample_weight; score test at 0.5."""
    # Project-level features for the test project, derived from the data
    # itself (not from labels), so no leakage even though we use it to
    # weight training rows.
    if test_pid in proj_features_all.index:
        test_feats = proj_features_all.loc[test_pid].values.astype(float)
    else:
        test_feats = proj_features_all.mean().values.astype(float)

    train_pids = sorted(proj_train.unique())
    train_proj_features = proj_features_all.loc[train_pids]
    weights_by_project = _similarity_weights(train_proj_features, test_feats)

    sample_weight = _build_sample_weights(proj_train, weights_by_project)

    # ITEM 5: row-level shape assertion. sample_weight MUST be one float per
    # training row, NOT one float per project. The assertion below catches
    # an entire class of bugs (passing project-level weights by mistake).
    assert sample_weight.shape == (len(X_train),), (
        f"sample_weight shape {sample_weight.shape} != (n_train,) ({len(X_train)},). "
        "Weights must be one float per training row, not per project."
    )
    assert np.all(np.isfinite(sample_weight)) and (sample_weight > 0).all(), (
        "sample_weight contains non-positive or non-finite values"
    )

    est = _make_model(model_name, params, y_train=y_train)
    if est is None:
        raise RuntimeError(f"Model {model_name!r} not available (import failed)")

    # Pipelines need the sample_weight to be routed to the final step
    # via the named-step convention "<step>__sample_weight". Bare
    # estimators take it directly.
    fit_kwargs: dict[str, Any] = {}
    if hasattr(est, "named_steps"):
        clf_step = list(est.named_steps.keys())[-1]
        fit_kwargs[f"{clf_step}__sample_weight"] = sample_weight
    else:
        fit_kwargs["sample_weight"] = sample_weight

    est.fit(X_train, y_train, **fit_kwargs)
    if hasattr(est, "predict_proba"):
        test_probs = est.predict_proba(X_test)[:, 1]
    else:
        test_probs = est.decision_function(X_test)
    return compute_metrics(y_test, test_probs)


# ---------------------------------------------------------------------------
# Public driver
# ---------------------------------------------------------------------------
def lopo_cv(
    model_name: str,
    *,
    params: Optional[dict[str, Any]] = None,
    skip_test_projects: Iterable[str] = (),
) -> list[LopoResult]:
    """Leave-One-Project-Out validation for a single model.

    Parameters
    ----------
    skip_test_projects :
        Project IDs that should NEVER be held out for evaluation. They
        still appear in training folds for the other projects. Used to
        keep small projects (e.g. daemon with 4 positives) as training-
        only data per item 1 of the May 13 enhancements.
    """
    X, y, proj = load_dataset()
    proj_features_all = _project_level_features(X, y, proj)

    projects = sorted(proj.unique())
    skip = set(skip_test_projects)
    results: list[LopoResult] = []

    for test_pid in projects:
        train_mask = proj.values != test_pid
        test_mask = proj.values == test_pid

        X_train = X.iloc[train_mask].reset_index(drop=True)
        y_train = np.asarray(y.iloc[train_mask], dtype=int)
        proj_train = proj.iloc[train_mask].reset_index(drop=True)
        X_test = X.iloc[test_mask].reset_index(drop=True)
        y_test = np.asarray(y.iloc[test_mask], dtype=int)

        if test_pid in skip:
            # daemon (or similar tiny project) - included in TRAINING for
            # every other fold but never used as the held-out test set.
            results.append(
                LopoResult(
                    model=model_name,
                    held_out_project=test_pid,
                    n_train=int(train_mask.sum()),
                    n_test=int(test_mask.sum()),
                    n_pos_test=int(y_test.sum()),
                    metrics={"f1": float("nan"), "roc_auc": float("nan"),
                             "pr_auc": float("nan"), "ce_at_20": float("nan")},
                    note="training_only",
                )
            )
            continue

        try:
            metrics = _fit_lopo_fold(
                model_name, X_train, y_train, proj_train,
                X_test, y_test, test_pid, proj_features_all, params,
            )
        except RuntimeError:
            return []

        results.append(
            LopoResult(
                model=model_name,
                held_out_project=test_pid,
                n_train=int(train_mask.sum()),
                n_test=int(test_mask.sum()),
                n_pos_test=int(y_test.sum()),
                metrics=metrics,
                note="",
            )
        )
    return results


def lopo_results_to_frame(results: Iterable[LopoResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        rows.append(
            {
                "model": r.model,
                "project_id": r.held_out_project,
                "n_train": r.n_train,
                "n_test": r.n_test,
                "n_pos_test": r.n_pos_test,
                "f1": r.metrics["f1"],
                "roc_auc": r.metrics["roc_auc"],
                "pr_auc": r.metrics["pr_auc"],
                "ce_at_20": r.metrics["ce_at_20"],
                "note": r.note,
            }
        )
    return pd.DataFrame(rows)


def lopo_summary(fold_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-project rows into mean +/- std per model.

    Rows with ``note == "training_only"`` (e.g. daemon) are excluded
    from the aggregate so they do not contaminate the mean.
    """
    eligible = fold_df[fold_df["note"] != "training_only"].copy()
    metric_cols = ["f1", "roc_auc", "pr_auc", "ce_at_20"]
    grp = eligible.groupby("model")
    means = grp[metric_cols].mean().add_suffix("_mean")
    stds = grp[metric_cols].std().add_suffix("_std")
    n_eff = grp.size().rename("n_projects_scored")
    return pd.concat([means, stds, n_eff], axis=1).reset_index()


"""
Stage 8 - Leave-One-Project-Out validation + best-model selection.

For each of the 4 models, run LOPO with similarity-weighted training
(see src/models/cross_project.py). Threshold optimisation is applied
to the (weighted) training set; the resulting threshold scores the
held-out test project.

ITEM 1 (May 13 enhancements): org.apache:daemon is excluded from
LOPO TEST. It still appears in training folds for the other 21
projects, but is never held out (only 4 positives total, the
fallback threshold could not satisfy min 5 positives). Rows for
daemon carry ``note='training_only'`` in lopo_per_project.csv and
are excluded from the model-level aggregate.

After all four models finish, write ``model_comparison.csv`` and log
the best model (highest LOPO F1_optimized).
"""

import json
import sys
import time
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*convergence.*")


SKIP_TEST_PROJECTS: tuple[str, ...] = ("org.apache:daemon",)


def _load_tuned_params() -> dict[str, dict]:
    path = TABLES_DIR / "tuned_hyperparameters.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run scripts/07b_tune.py before Stage 8."
        )
    df = pd.read_csv(path)
    return {row["model"]: json.loads(row["best_params_json"]) for _, row in df.iterrows()}


def run_stage_8() -> None:
    t0 = time.time()
    print(f"[Stage 8] Models       : {list(MODEL_ORDER)}")
    print(f"[Stage 8] Skip-test    : {list(SKIP_TEST_PROJECTS)}")

    tuned_params = _load_tuned_params()
    all_rows: list[pd.DataFrame] = []
    for model_name in MODEL_ORDER:
        t1 = time.time()
        params = tuned_params.get(model_name)
        print(f"\n[Stage 8] LOPO {model_name} (params loaded={params is not None}) ...")
        try:
            results = lopo_cv(
                model_name,
                params=params,
                skip_test_projects=SKIP_TEST_PROJECTS,
            )
        except Exception as e:
            print(f"   FAILED: {type(e).__name__}: {e}")
            continue
        if not results:
            print(f"   SKIPPED (estimator unavailable)")
            continue
        df = lopo_results_to_frame(results)
        eligible = df[df["note"] != "training_only"]
        print(
            f"   scored={len(eligible)} of {len(df)} projects  "
            f"mean_F1={eligible['f1'].mean():.3f}  "
            f"mean_ROC={eligible['roc_auc'].mean():.3f}  "
            f"mean_PR={eligible['pr_auc'].mean():.3f}  "
            f"mean_CE@20={eligible['ce_at_20'].mean():.3f}  "
            f"({time.time() - t1:.1f}s)"
        )
        all_rows.append(df)

    per_project = pd.concat(all_rows, ignore_index=True)
    per_project_path = TABLES_DIR / "lopo_per_project.csv"
    per_project.to_csv(per_project_path, index=False)
    print(f"\n[Stage 8] Wrote {per_project_path}  ({len(per_project)} rows)")

    summary = lopo_summary(per_project)
    summary_path = TABLES_DIR / "lopo_results.csv"
    summary.to_csv(summary_path, index=False)
    print(f"[Stage 8] Wrote {summary_path}  ({len(summary)} rows)")

    # ----- Best-model selection -----
    print("\n[Stage 8] Building model_comparison.csv ...")
    within = pd.read_csv(TABLES_DIR / "within_project_results.csv")
    within_agg = (
        within.groupby("model")[["f1", "pr_auc"]]
        .mean()
        .rename(columns={"f1": "within_f1", "pr_auc": "within_pr_auc"})
    )
    lopo_agg = summary.set_index("model")[
        ["f1_mean", "pr_auc_mean", "ce_at_20_mean", "roc_auc_mean"]
    ].rename(
        columns={
            "f1_mean": "lopo_f1",
            "pr_auc_mean": "lopo_pr_auc",
            "ce_at_20_mean": "lopo_ce_at_20",
            "roc_auc_mean": "lopo_roc_auc",
        }
    )
    cmp_df = within_agg.join(lopo_agg, how="outer").reset_index()
    cmp_df["is_best"] = False
    if cmp_df["lopo_f1"].notna().any():
        best_idx = cmp_df["lopo_f1"].idxmax()
        cmp_df.loc[best_idx, "is_best"] = True
        best_model = cmp_df.loc[best_idx, "model"]
        best_f1 = float(cmp_df.loc[best_idx, "lopo_f1"])
        print(f"[Stage 8] Best model: {best_model}, LOPO F1 = {best_f1:.4f}")

    cmp_path = TABLES_DIR / "model_comparison.csv"
    cmp_df.to_csv(cmp_path, index=False)
    print(f"[Stage 8] Wrote {cmp_path}")

    print("\n[Stage 8] Summary table:")
    with pd.option_context("display.width", 220, "display.max_rows", None):
        print(cmp_df.round(4).to_string(index=False))

    print(f"\n[Stage 8] Total elapsed: {time.time() - t0:.1f}s")
    print("[Stage 8] Complete.")


In [ ]:
run_stage_8()
print()
print('--- lopo_results.csv ---')
df = pd.read_csv('results/tables/lopo_results.csv')
num = df.select_dtypes('number').columns
df[num] = df[num].round(4)
display(df.style.highlight_max(subset=['f1_mean'], color='lightgreen'))

print()
print('--- model_comparison.csv ---')
mc = pd.read_csv('results/tables/model_comparison.csv')
num = mc.select_dtypes('number').columns
mc[num] = mc[num].round(4)
display(mc)

best = mc[mc['is_best']].iloc[0]
print(f"\n{best['model'].upper()} (best)")
print(f"  LOPO F1     = {best['lopo_f1']:.4f}")
print(f"  LOPO CE@20  = {best['lopo_ce_at_20']:.4f}")
print(f"  ({100*best['lopo_ce_at_20']:.1f}% of high-risk files found by reviewing top 20%)")

# fig_07 / fig_08 are produced later by 10_report.py (Section 12); show if available.
show_figure('results/figures/fig_07_lopo_per_project_f1.png')
show_figure('results/figures/fig_08_lopo_ce20_distribution.png')

# Drive checkpoint
shutil.copytree('results', f'{DRIVE_BASE}/results', dirs_exist_ok=True)
print('\nResults checkpointed to Drive.')


## Section 11 — Feature Family Ablation

Systematically remove each feature family (and isolate each family alone)
to measure which groups carry the most predictive signal. 11-row table:
1 baseline + 5 families × 2 modes. Writes `ablation_results.csv`.

In [ ]:
# Pipeline code

"""
Feature-family ablation on the best model.

Five families x two modes plus the all-features baseline = 11 rows:

  - all_features                         (baseline)
  - only_<family> for each of 5 families (isolated contribution)
  - leave_out_<family> for each of 5     (marginal drop)

Within-project 10-fold CV is used because it is fast and the metric
of interest (F1) has the same direction as the LOPO metric for the
ablation purpose. Reports F1, PR-AUC, CE@20 means across folds.
"""

import sys
from pathlib import Path
from typing import Iterable

import pandas as pd


def _run(model_name: str, X: pd.DataFrame, y: pd.Series, params: dict | None) -> dict:
    results = stratified_kfold_cv(model_name, X, y, n_splits=CV_FOLDS, params=params)
    df = fold_results_to_frame(results)
    return {
        "f1": float(df["f1"].mean()),
        "pr_auc": float(df["pr_auc"].mean()),
        "ce_at_20": float(df["ce_at_20"].mean()),
        "f1_std": float(df["f1"].std()),
    }


def run_family_ablation(
    model_name: str,
    params: dict | None,
    families: dict[str, list[str]] = FEATURE_FAMILIES,
) -> pd.DataFrame:
    """Return the 11-row ablation table for ``model_name``."""
    X, y, _ = load_dataset()
    rows: list[dict] = []

    # Baseline
    base = _run(model_name, X[list(ALL_FEATURES)], y, params)
    rows.append({"mode": "all_features", "family": "(all)", "n_features": len(ALL_FEATURES), **base})

    # Per-family only / leave-out
    for fam, cols in families.items():
        cols_in = [c for c in cols if c in X.columns]
        only_X = X[cols_in]
        leave_X = X[[c for c in ALL_FEATURES if c not in cols_in]]

        only = _run(model_name, only_X, y, params)
        rows.append({"mode": "only_this_family", "family": fam, "n_features": len(cols_in), **only})

        leave = _run(model_name, leave_X, y, params)
        rows.append({"mode": "leave_out_family", "family": fam, "n_features": leave_X.shape[1], **leave})

    out = pd.DataFrame(rows)
    return out


"""
Stage 9 - Feature-family ablation on the best model.

Reads the best model + tuned hyperparams from stage 7b / 8 outputs,
runs the 11-row ablation (1 baseline + 5 families x 2 modes), and
writes ``results/tables/ablation_results.csv``.
"""

import json
import sys
import time
from pathlib import Path

import pandas as pd


def run_stage_9() -> None:
    t0 = time.time()

    mc = pd.read_csv(TABLES_DIR / "model_comparison.csv")
    if mc["is_best"].sum() != 1:
        raise RuntimeError("model_comparison.csv must flag exactly one best model")
    best_model = mc[mc["is_best"]].iloc[0]["model"]
    print(f"[Stage 9] Best model: {best_model}")

    tuned = pd.read_csv(TABLES_DIR / "tuned_hyperparameters.csv")
    params_row = tuned[tuned["model"] == best_model]
    params = json.loads(params_row.iloc[0]["best_params_json"]) if len(params_row) else None
    print(f"[Stage 9] Params: {params}")

    print("[Stage 9] Running ablation (within-project 10-fold CV) ...")
    out = run_family_ablation(best_model, params)
    out_path = TABLES_DIR / "ablation_results.csv"
    out.to_csv(out_path, index=False)
    print(f"[Stage 9] Wrote {out_path}  ({len(out)} rows)")

    print("\n[Stage 9] Ablation table:")
    with pd.option_context("display.width", 200, "display.max_rows", None):
        print(out.round(4).to_string(index=False))

    # Quick interpretation: print the marginal F1 drop per family.
    base = out[out["mode"] == "all_features"]["f1"].iloc[0]
    leave = out[out["mode"] == "leave_out_family"][["family", "f1"]].copy()
    leave["f1_drop_vs_all"] = (base - leave["f1"]).round(4)
    print("\n[Stage 9] Marginal F1 drop when family is removed (sorted descending):")
    print(leave.sort_values("f1_drop_vs_all", ascending=False).to_string(index=False))

    print(f"\n[Stage 9] Total elapsed: {time.time() - t0:.1f}s")
    print("[Stage 9] Complete.")


In [ ]:
run_stage_9()
print()
print('--- ablation_results.csv ---')
show_table('results/tables/ablation_results.csv')

# fig_10 is produced later by 10_report.py (Section 12); show if available.
show_figure('results/figures/fig_10_ablation.png')


## Section 12 — Feature Importance and Report Figures

SHAP analysis on the best model (top 15 features → fig_11) and permutation
importance for all 4 models (top 10 each → `feature_importance.csv`).
Pooled LOPO predictions across the 4 models produce fig_12 ROC/PR curves.
Also generates fig_05 and fig_09. All 12 thesis figures are present after
this section.

In [ ]:
# Pipeline code

"""
12 publication-ready figures for the TD prediction thesis.

Every function writes one 300-DPI PNG into ``results/figures/``.
Consistent style: ``seaborn-v0_8-whitegrid``; sizes vary by content
(bar charts 10x6, heatmaps 8x8, multi-subplot 12x5).
"""

import json
import sys
from pathlib import Path
from typing import Optional

import matplotlib
matplotlib.use("Agg")  # headless backend for CI / server runs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


_STYLE = "seaborn-v0_8-whitegrid"
_FAMILY_COLORS = {
    "size_complexity": "#1f77b4",
    "static_debt": "#ff7f0e",
    "historical": "#2ca02c",
    "graph": "#d62728",
    "prior_defect": "#9467bd",
}


def _set_style() -> None:
    plt.style.use(_STYLE)


def _feature_to_family() -> dict[str, str]:
    out: dict[str, str] = {}
    for fam, cols in FEATURE_FAMILIES.items():
        for c in cols:
            out[c] = fam
    return out


def _save(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
# Fig 01 - positive rates per project
# ---------------------------------------------------------------------------
def fig_01_positive_rates() -> Path:
    _set_style()
    stats = pd.read_csv(PROCESSED_DATA_DIR / "label_statistics.csv")
    stats = stats.sort_values("positive_rate_pct", ascending=True)
    colors = ["#2ca02c" if r < 15 else "#ff7f0e" for r in stats["positive_rate_pct"]]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(stats["project_id"], stats["positive_rate_pct"], color=colors)
    mean_rate = stats["positive_rate_pct"].mean()
    ax.axvline(mean_rate, ls="--", color="black", alpha=0.5, label=f"mean = {mean_rate:.1f}%")
    ax.set_xlabel("Positive rate (%)")
    ax.set_title("Per-Project Positive Rate of High-Risk TD Label")
    ax.legend()
    out = FIGURES_DIR / "fig_01_positive_rates.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 02 - label signal breakdown
# ---------------------------------------------------------------------------
def fig_02_label_signal_breakdown() -> Path:
    _set_style()
    labels = pd.read_parquet(PROCESSED_DATA_DIR / "labels.parquet")
    static_any = (labels[["S1_severity", "S2_debt", "S3_smells"]].sum(axis=1) >= 1).astype(int)
    history_any = (labels[["S4_bugfix", "S5_churn", "S6_contributors"]].sum(axis=1) >= 1).astype(int)
    labels = labels.assign(_static=static_any, _history=history_any)

    rows = []
    for pid, sub in labels.groupby("project_id"):
        s_only = int(((sub["_static"] == 1) & (sub["_history"] == 0)).sum())
        h_only = int(((sub["_static"] == 0) & (sub["_history"] == 1)).sum())
        both = int(((sub["_static"] == 1) & (sub["_history"] == 1)).sum())
        neither = int(((sub["_static"] == 0) & (sub["_history"] == 0)).sum())
        rows.append({"project_id": pid, "static_only": s_only, "history_only": h_only,
                     "both": both, "neither": neither})
    df = pd.DataFrame(rows).sort_values("project_id").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))
    bot = np.zeros(len(df))
    for col, color in (("static_only", "#1f77b4"), ("history_only", "#2ca02c"),
                       ("both", "#d62728"), ("neither", "#cccccc")):
        ax.bar(x, df[col], bottom=bot, label=col, color=color)
        bot += df[col].values
    ax.set_xticks(x)
    ax.set_xticklabels(df["project_id"], rotation=70, ha="right", fontsize=8)
    ax.set_ylabel("File count")
    ax.set_title("Dual-Signal Label Composition per Project")
    ax.legend(loc="upper right")
    out = FIGURES_DIR / "fig_02_label_signal_breakdown.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 03 - risk score distribution
# ---------------------------------------------------------------------------
def fig_03_risk_score_distribution() -> Path:
    _set_style()
    labels = pd.read_parquet(PROCESSED_DATA_DIR / "labels.parquet")
    pos = labels.loc[labels["is_high_risk"] == 1, "risk_score"]
    neg = labels.loc[labels["is_high_risk"] == 0, "risk_score"]
    fig, ax = plt.subplots(figsize=(10, 6))
    bins = np.linspace(0, 1, 41)
    ax.hist(neg, bins=bins, alpha=0.6, color="#1f77b4", label="is_high_risk = 0", density=True)
    ax.hist(pos, bins=bins, alpha=0.6, color="#d62728", label="is_high_risk = 1", density=True)
    ax.axvline(0.5, ls="--", color="black", alpha=0.6, label="threshold = 0.50")
    ax.set_xlabel("risk_score")
    ax.set_ylabel("density")
    ax.set_title("Risk Score Distribution by Label Class")
    ax.legend()
    out = FIGURES_DIR / "fig_03_risk_score_distribution.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 04 - feature correlation heatmap
# ---------------------------------------------------------------------------
def fig_04_feature_correlation_heatmap() -> Path:
    _set_style()
    ds = pd.read_parquet(PROCESSED_DATA_DIR / "dataset_final.parquet")
    ordered = list(ALL_FEATURES)
    corr = ds[ordered].corr(method="spearman").values
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(ordered)))
    ax.set_yticks(range(len(ordered)))
    ax.set_xticklabels(ordered, rotation=80, fontsize=7)
    ax.set_yticklabels(ordered, fontsize=7)

    # Family boundary lines
    boundary = 0
    for fam, cols in FEATURE_FAMILIES.items():
        boundary += len(cols)
        if boundary < len(ordered):
            ax.axhline(boundary - 0.5, color="black", lw=1)
            ax.axvline(boundary - 0.5, color="black", lw=1)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title("Feature Spearman Correlation Matrix (family-clustered)")
    out = FIGURES_DIR / "fig_04_feature_correlation_heatmap.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 05 - collision resolution per project
# ---------------------------------------------------------------------------
def fig_05_collision_resolution() -> Path:
    _set_style()
    report = pd.read_csv(PROCESSED_DATA_DIR / "collision_report.csv").sort_values("project_id")
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(report))
    ax.bar(x, report["n_kept"], color="#2ca02c", label="kept")
    ax.bar(x, report["n_dropped"], bottom=report["n_kept"], color="#d62728", label="dropped")
    ax.set_xticks(x)
    ax.set_xticklabels(report["project_id"], rotation=70, ha="right", fontsize=8)
    ax.set_ylabel("Basename count")
    ax.set_title("Basename Resolution Status per Project")
    ax.legend()
    out = FIGURES_DIR / "fig_05_collision_resolution.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 06 - model comparison F1
# ---------------------------------------------------------------------------
def fig_06_model_comparison_f1() -> Path:
    _set_style()
    within = pd.read_csv(TABLES_DIR / "within_project_results.csv")
    lopo_pp = pd.read_csv(TABLES_DIR / "lopo_per_project.csv")
    lopo_pp = lopo_pp[lopo_pp["note"] != "training_only"]

    models = sorted(within["model"].unique())
    w_mean = within.groupby("model")["f1"].mean().reindex(models)
    w_std = within.groupby("model")["f1"].std().reindex(models)
    l_mean = lopo_pp.groupby("model")["f1"].mean().reindex(models)
    l_std = lopo_pp.groupby("model")["f1"].std().reindex(models)

    x = np.arange(len(models))
    w = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - w/2, w_mean.values, w, yerr=w_std.values, color="#1f77b4", label="Within-project (fold std)")
    ax.bar(x + w/2, l_mean.values, w, yerr=l_std.values, color="#d62728", label="LOPO (project std)")
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=20, ha="right")
    ax.set_ylabel("F1")
    ax.set_title("Model F1 Comparison: Within-Project vs LOPO")
    ax.legend()
    ax.set_ylim(0, 1)
    out = FIGURES_DIR / "fig_06_model_comparison_f1.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 07 - per-project LOPO F1 for best model
# ---------------------------------------------------------------------------
def fig_07_lopo_per_project_f1(best_model: str) -> Path:
    _set_style()
    lopo_pp = pd.read_csv(TABLES_DIR / "lopo_per_project.csv")
    sub = lopo_pp[(lopo_pp["model"] == best_model) & (lopo_pp["note"] != "training_only")].copy()
    sub = sub.sort_values("f1", ascending=False)
    mean_f1 = sub["f1"].mean()
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(sub))
    ax.bar(x, sub["f1"], color="#1f77b4")
    ax.axhline(mean_f1, ls="--", color="black", alpha=0.6, label=f"mean = {mean_f1:.3f}")
    ax.set_xticks(x)
    ax.set_xticklabels(sub["project_id"], rotation=70, ha="right", fontsize=8)
    ax.set_ylabel("LOPO F1")
    ax.set_title(f"Per-Project LOPO F1 (Best Model: {best_model})")
    ax.legend()
    ax.set_ylim(0, 1)
    out = FIGURES_DIR / "fig_07_lopo_per_project_f1.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 08 - CE@20 distribution per model
# ---------------------------------------------------------------------------
def fig_08_lopo_ce20_distribution() -> Path:
    _set_style()
    lopo_pp = pd.read_csv(TABLES_DIR / "lopo_per_project.csv")
    lopo_pp = lopo_pp[lopo_pp["note"] != "training_only"]
    models = sorted(lopo_pp["model"].unique())
    data = [lopo_pp.loc[lopo_pp["model"] == m, "ce_at_20"].values for m in models]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.boxplot(data, labels=models, patch_artist=True)
    ax.axhline(0.5, ls="--", color="black", alpha=0.6, label="random baseline = 0.5")
    ax.set_ylabel("CE@20")
    ax.set_title("Cross-Project CE@20 Distribution")
    ax.legend()
    ax.set_ylim(0, 1)
    out = FIGURES_DIR / "fig_08_lopo_ce20_distribution.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 09 - F1 vs threshold curve for best model
# ---------------------------------------------------------------------------
def fig_09_threshold_curve(best_model: str, best_params: dict) -> Path:
    _set_style()
    from sklearn.metrics import f1_score
    from sklearn.model_selection import StratifiedKFold


    X, y, _ = load_dataset()
    yv = np.asarray(y, dtype=int)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    thresholds = np.arange(0.05, 0.80, 0.01)
    f1s = []
    for t in thresholds:
        fold_f1 = []
        for tr, te in skf.split(X, yv):
            est = _make_model(best_model, best_params, y_train=yv[tr])
            est.fit(X.iloc[tr], yv[tr])
            proba = est.predict_proba(X.iloc[te])[:, 1]
            pred = (proba >= t).astype(int)
            if pred.sum() == 0:
                fold_f1.append(0.0)
                continue
            fold_f1.append(f1_score(yv[te], pred, zero_division=0))
        f1s.append(float(np.mean(fold_f1)))
    f1s = np.array(f1s)
    optimal_idx = int(np.argmax(f1s))
    optimal_t = float(thresholds[optimal_idx])
    f1_at_05 = float(f1s[np.argmin(np.abs(thresholds - 0.5))])
    f1_at_opt = float(f1s[optimal_idx])

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(thresholds, f1s, color="#1f77b4", lw=2)
    ax.axvline(optimal_t, ls="-", color="#d62728", alpha=0.7,
               label=f"optimal = {optimal_t:.2f}  (F1 = {f1_at_opt:.3f})")
    ax.axvline(0.5, ls="--", color="black", alpha=0.6,
               label=f"default 0.5  (F1 = {f1_at_05:.3f})")
    delta = f1_at_opt - f1_at_05
    ax.set_xlabel("Classification threshold")
    ax.set_ylabel("Mean 5-fold CV F1")
    ax.set_title(f"F1 vs Threshold ({best_model})  -  delta(opt - 0.5) = {delta:+.4f}")
    ax.legend()
    out = FIGURES_DIR / "fig_09_threshold_curve.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 10 - feature family ablation
# ---------------------------------------------------------------------------
def fig_10_ablation() -> Path:
    _set_style()
    abl = pd.read_csv(TABLES_DIR / "ablation_results.csv")
    families = list(FEATURE_FAMILIES.keys())
    baseline = abl[abl["mode"] == "all_features"].iloc[0]

    only = abl[abl["mode"] == "only_this_family"].set_index("family").reindex(families)
    leave = abl[abl["mode"] == "leave_out_family"].set_index("family").reindex(families)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    x = np.arange(len(families))
    w = 0.27

    # F1 subplot
    ax1.bar(x - w, only["f1"], w, label="only_this_family", color="#1f77b4")
    ax1.bar(x, leave["f1"], w, label="leave_out_family", color="#d62728")
    ax1.bar(x + w, [baseline["f1"]] * len(families), w, label="all_features", color="#2ca02c")
    ax1.set_xticks(x)
    ax1.set_xticklabels(families, rotation=20, ha="right")
    ax1.set_ylabel("F1")
    ax1.set_title("F1 by family ablation")
    ax1.set_ylim(0, 1)
    ax1.legend()

    # CE@20 subplot
    ax2.bar(x - w, only["ce_at_20"], w, label="only_this_family", color="#1f77b4")
    ax2.bar(x, leave["ce_at_20"], w, label="leave_out_family", color="#d62728")
    ax2.bar(x + w, [baseline["ce_at_20"]] * len(families), w, label="all_features", color="#2ca02c")
    ax2.set_xticks(x)
    ax2.set_xticklabels(families, rotation=20, ha="right")
    ax2.set_ylabel("CE@20")
    ax2.set_title("CE@20 by family ablation")
    ax2.set_ylim(0, 1)
    ax2.legend()

    fig.suptitle(f"Feature Family Ablation (best model)")
    fig.tight_layout()
    out = FIGURES_DIR / "fig_10_ablation.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 11 - SHAP top-15
# ---------------------------------------------------------------------------
def fig_11_shap_best_model(best_model: str, best_params: dict, sample_size: int = 2000) -> Path:
    _set_style()
    import shap


    X, y, _ = load_dataset()
    yv = np.asarray(y, dtype=int)
    est = _make_model(best_model, best_params, y_train=yv)
    est.fit(X, yv)

    # Subsample for SHAP if large
    if len(X) > sample_size:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X), size=sample_size, replace=False)
        X_shap = X.iloc[idx]
    else:
        X_shap = X

    explainer = shap.TreeExplainer(est)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):  # multi-class case
        shap_values = shap_values[1]

    mean_abs = np.abs(shap_values).mean(axis=0)
    top_idx = np.argsort(mean_abs)[::-1][:15]
    top_features = [X.columns[i] for i in top_idx]
    top_values = mean_abs[top_idx]
    feat_fam = _feature_to_family()
    colors = [_FAMILY_COLORS.get(feat_fam.get(f, ""), "#888888") for f in top_features]

    fig, ax = plt.subplots(figsize=(10, 7))
    y_pos = np.arange(len(top_features))[::-1]
    ax.barh(y_pos, top_values, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title(f"Feature Importance (SHAP, {best_model})  -  top 15")

    # Family legend
    from matplotlib.patches import Patch
    legend_handles = [Patch(color=col, label=fam) for fam, col in _FAMILY_COLORS.items()]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=9)

    out = FIGURES_DIR / "fig_11_shap_best_model.png"
    _save(fig, out)
    return out


# ---------------------------------------------------------------------------
# Fig 12 - ROC and PR curves for 4 models (LOPO)
# ---------------------------------------------------------------------------
def fig_12_roc_pr_curves(per_model_lopo_curves: dict[str, dict]) -> Path:
    """``per_model_lopo_curves[model] = {"fpr": ..., "tpr": ..., "precision": ...,
    "recall": ..., "positive_rate": float}``"""
    _set_style()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    colors = {"logistic_regression": "#9467bd", "random_forest": "#2ca02c",
              "xgboost": "#d62728", "lightgbm": "#ff7f0e"}

    # ROC
    for model, curves in per_model_lopo_curves.items():
        ax1.plot(curves["fpr"], curves["tpr"], color=colors.get(model, "black"),
                 label=f"{model} (AUC={curves['roc_auc']:.3f})", lw=2)
    ax1.plot([0, 1], [0, 1], ls="--", color="black", alpha=0.5, label="random")
    ax1.set_xlabel("False positive rate")
    ax1.set_ylabel("True positive rate")
    ax1.set_title("ROC curves (LOPO, pooled)")
    ax1.legend()
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1.02)

    # PR
    overall_pos_rate = None
    for model, curves in per_model_lopo_curves.items():
        ax2.plot(curves["recall"], curves["precision"], color=colors.get(model, "black"),
                 label=f"{model} (PR-AUC={curves['pr_auc']:.3f})", lw=2)
        overall_pos_rate = curves.get("positive_rate", overall_pos_rate)
    if overall_pos_rate is not None:
        ax2.axhline(overall_pos_rate, ls="--", color="black", alpha=0.5,
                    label=f"random = {overall_pos_rate:.3f}")
    ax2.set_xlabel("Recall")
    ax2.set_ylabel("Precision")
    ax2.set_title("Precision-Recall curves (LOPO, pooled)")
    ax2.legend()
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1.02)

    fig.suptitle("ROC and PR Curves (LOPO)")
    fig.tight_layout()
    out = FIGURES_DIR / "fig_12_roc_pr_curves.png"
    _save(fig, out)
    return out


"""
Stage 10 - Render the 12 thesis figures + feature importance tables.

Per spec:
  - SHAP for BEST MODEL only (top 15 features) -> fig_11
  - Permutation importance for ALL 4 models (top 10 each)
    -> results/tables/feature_importance.csv
  - Figures 1-12 as defined in src/reporting/figures.py

Pooled LOPO predictions for fig_12: for each of the 4 models, run
LOPO once collecting (y_true, y_prob) across all 22 held-out projects
(excluding daemon), then compute pooled ROC + PR curves.
"""

import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


SKIP_TEST_PROJECTS = ("org.apache:daemon",)


def _load_tuned_params() -> dict[str, dict]:
    df = pd.read_csv(TABLES_DIR / "tuned_hyperparameters.csv")
    return {row["model"]: json.loads(row["best_params_json"]) for _, row in df.iterrows()}


def _best_model_name() -> str:
    mc = pd.read_csv(TABLES_DIR / "model_comparison.csv")
    return mc[mc["is_best"]].iloc[0]["model"]


# ---------------------------------------------------------------------------
# Permutation importance (within-project, single fold for speed)
# ---------------------------------------------------------------------------
def _permutation_importance_all_models(tuned_params: dict[str, dict]) -> pd.DataFrame:
    X, y, _ = load_dataset()
    yv = np.asarray(y, dtype=int)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    train_idx, test_idx = next(iter(skf.split(X, yv)))
    rows: list[dict] = []
    for model_name in MODEL_ORDER:
        params = tuned_params.get(model_name)
        est = _make_model(model_name, params, y_train=yv[train_idx])
        if est is None:
            continue
        est.fit(X.iloc[train_idx], yv[train_idx])
        result = permutation_importance(
            est,
            X.iloc[test_idx],
            yv[test_idx],
            n_repeats=5,
            random_state=RANDOM_STATE,
            scoring="average_precision",
            n_jobs=-1,
        )
        order = np.argsort(result.importances_mean)[::-1][:10]
        for rank, idx in enumerate(order, start=1):
            rows.append(
                {
                    "model": model_name,
                    "rank": rank,
                    "feature": X.columns[idx],
                    "importance_mean": float(result.importances_mean[idx]),
                    "importance_std": float(result.importances_std[idx]),
                }
            )
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Pooled LOPO predictions per model for fig_12
# ---------------------------------------------------------------------------
def _pooled_lopo_curves(tuned_params: dict[str, dict]) -> dict[str, dict]:
    X, y, proj = load_dataset()
    proj_features_all = _project_level_features(X, y, proj)
    projects = sorted(proj.unique())

    out: dict[str, dict] = {}
    for model_name in MODEL_ORDER:
        params = tuned_params.get(model_name)
        all_y_true: list[np.ndarray] = []
        all_y_prob: list[np.ndarray] = []
        for test_pid in projects:
            if test_pid in SKIP_TEST_PROJECTS:
                continue
            train_mask = proj.values != test_pid
            test_mask = proj.values == test_pid

            X_train = X.iloc[train_mask].reset_index(drop=True)
            y_train = np.asarray(y.iloc[train_mask], dtype=int)
            proj_train = proj.iloc[train_mask].reset_index(drop=True)
            X_test = X.iloc[test_mask].reset_index(drop=True)
            y_test = np.asarray(y.iloc[test_mask], dtype=int)

            if test_pid in proj_features_all.index:
                test_feats = proj_features_all.loc[test_pid].values.astype(float)
            else:
                test_feats = proj_features_all.mean().values.astype(float)
            train_pids = sorted(proj_train.unique())
            train_proj_features = proj_features_all.loc[train_pids]
            weights_by_project = _similarity_weights(train_proj_features, test_feats)
            sample_weight = _build_sample_weights(proj_train, weights_by_project)

            est = _make_model(model_name, params, y_train=y_train)
            if est is None:
                break
            fit_kwargs: dict = {}
            if hasattr(est, "named_steps"):
                clf_step = list(est.named_steps.keys())[-1]
                fit_kwargs[f"{clf_step}__sample_weight"] = sample_weight
            else:
                fit_kwargs["sample_weight"] = sample_weight
            est.fit(X_train, y_train, **fit_kwargs)
            if hasattr(est, "predict_proba"):
                probs = est.predict_proba(X_test)[:, 1]
            else:
                probs = est.decision_function(X_test)
            all_y_true.append(y_test)
            all_y_prob.append(probs)

        if not all_y_true:
            continue
        y_true = np.concatenate(all_y_true)
        y_prob = np.concatenate(all_y_prob)
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        out[model_name] = {
            "fpr": fpr,
            "tpr": tpr,
            "precision": precision,
            "recall": recall,
            "roc_auc": float(roc_auc_score(y_true, y_prob)),
            "pr_auc": float(average_precision_score(y_true, y_prob)),
            "positive_rate": float(np.mean(y_true)),
        }
    return out


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def run_stage_10() -> None:
    t0 = time.time()
    tuned_params = _load_tuned_params()
    best_model = _best_model_name()
    best_params = tuned_params.get(best_model, {})
    print(f"[Stage 10] Best model: {best_model}")
    print(f"[Stage 10] Best params: {best_params}")

    generated: list[str] = []
    failed: list[tuple[str, str]] = []

    # --- Static figures (no model training needed) ---
    for fn, name in (
        (fig_01_positive_rates, "fig_01"),
        (fig_02_label_signal_breakdown, "fig_02"),
        (fig_03_risk_score_distribution, "fig_03"),
        (fig_04_feature_correlation_heatmap, "fig_04"),
        (fig_05_collision_resolution, "fig_05"),
        (fig_06_model_comparison_f1, "fig_06"),
        (fig_08_lopo_ce20_distribution, "fig_08"),
        (fig_10_ablation, "fig_10"),
    ):
        try:
            path = fn()
            print(f"   {name} OK -> {path.name}")
            generated.append(name)
        except Exception as e:
            print(f"   {name} FAIL: {type(e).__name__}: {e}")
            failed.append((name, f"{type(e).__name__}: {e}"))

    # --- Best-model dependent figures ---
    try:
        path = fig_07_lopo_per_project_f1(best_model)
        print(f"   fig_07 OK -> {path.name}")
        generated.append("fig_07")
    except Exception as e:
        print(f"   fig_07 FAIL: {type(e).__name__}: {e}")
        failed.append(("fig_07", f"{type(e).__name__}: {e}"))

    try:
        t1 = time.time()
        path = fig_09_threshold_curve(best_model, best_params)
        print(f"   fig_09 OK -> {path.name}  ({time.time() - t1:.1f}s)")
        generated.append("fig_09")
    except Exception as e:
        print(f"   fig_09 FAIL: {type(e).__name__}: {e}")
        failed.append(("fig_09", f"{type(e).__name__}: {e}"))

    try:
        t1 = time.time()
        path = fig_11_shap_best_model(best_model, best_params)
        print(f"   fig_11 OK -> {path.name}  ({time.time() - t1:.1f}s)")
        generated.append("fig_11")
    except Exception as e:
        print(f"   fig_11 FAIL: {type(e).__name__}: {e}")
        failed.append(("fig_11", f"{type(e).__name__}: {e}"))

    # --- Pooled LOPO curves for fig_12 ---
    print("\n[Stage 10] Running pooled LOPO for fig_12 ...")
    try:
        t1 = time.time()
        curves = _pooled_lopo_curves(tuned_params)
        print(f"[Stage 10] Pooled LOPO done ({time.time() - t1:.1f}s, {len(curves)} models)")
        path = fig_12_roc_pr_curves(curves)
        print(f"   fig_12 OK -> {path.name}")
        generated.append("fig_12")
    except Exception as e:
        print(f"   fig_12 FAIL: {type(e).__name__}: {e}")
        failed.append(("fig_12", f"{type(e).__name__}: {e}"))

    # --- Permutation importance table ---
    print("\n[Stage 10] Computing permutation importance (4 models, top 10 each) ...")
    try:
        t1 = time.time()
        perm_df = _permutation_importance_all_models(tuned_params)
        perm_path = TABLES_DIR / "feature_importance.csv"
        perm_df.to_csv(perm_path, index=False)
        print(f"[Stage 10] Wrote {perm_path}  ({len(perm_df)} rows, {time.time() - t1:.1f}s)")
    except Exception as e:
        print(f"[Stage 10] permutation importance FAIL: {type(e).__name__}: {e}")

    # --- Summary ---
    print("\n[Stage 10] Figure generation summary:")
    print(f"   Generated: {len(generated)} / 12")
    for n in sorted(generated):
        print(f"      {n}")
    if failed:
        print(f"   Failed: {len(failed)}")
        for n, msg in failed:
            print(f"      {n}: {msg}")
    print(f"\n[Stage 10] Elapsed: {time.time() - t0:.1f}s")
    print("[Stage 10] Complete.")


In [ ]:
run_stage_10()
print()
# All 12 figures are now generated. Show the three explicitly requested for this section.
show_figure('results/figures/fig_09_threshold_curve.png')
show_figure('results/figures/fig_11_shap_best_model.png')
show_figure('results/figures/fig_12_roc_pr_curves.png')

print()
print('--- feature_importance.csv (top 10) ---')
show_table('results/tables/feature_importance.csv', top=10)


## Section 13 — Model Persistence

Persist the best model, scaler, threshold (0.5), feature names, scoring
helper, and model card to `models/` and to Drive.

In [ ]:
# Pipeline code

"""
Stage 11 - Persist the best model + supporting artefacts.

Reads:
- ``results/tables/model_comparison.csv`` to identify the best model
  (where ``is_best == True``).
- ``results/tables/tuned_hyperparameters.csv`` for tuned hyperparams.
- ``data/processed/dataset_final.parquet`` to refit on the full corpus.

Refits the best model on ALL rows, fits a StandardScaler on the same
matrix (for downstream inference symmetry), and writes:

  models/best_model.pkl
  models/feature_scaler.pkl
  models/optimal_threshold.txt   (always "0.5" - see train.py rationale)
  models/feature_names.csv
  models/model_card.json
  models/score_project.py
"""

import json
import sys
import time
from pathlib import Path

import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler


SCORING_FN_TEMPLATE = '''"""
score_project.py - Inference helper for the persisted best model.

Apply log1p to the same columns the training pipeline did, scale with
the persisted StandardScaler, then call predict_proba. A predicted
risk score >= the persisted threshold is flagged as high-risk TD.
"""
import joblib
import pandas as pd
import numpy as np


LOG1P_COLS = __LOG1P_COLS_PLACEHOLDER__


def score_new_project(features_df: pd.DataFrame) -> pd.DataFrame:
    """Score files in a new Apache Java project.

    Input: a DataFrame with the 27 feature columns produced by stage 5
    (in any order; the model's feature_names.csv defines the canonical
    ordering). Output: original rows annotated with ``risk_score`` and
    ``predicted_high_risk``, sorted by descending risk.
    """
    model = joblib.load("models/best_model.pkl")
    scaler = joblib.load("models/feature_scaler.pkl")
    threshold = float(open("models/optimal_threshold.txt").read())
    feat_names = pd.read_csv("models/feature_names.csv")["feature"].tolist()

    X = features_df[feat_names].fillna(0).astype(float).copy()
    for col in LOG1P_COLS:
        if col in X.columns:
            X[col] = np.log1p(X[col])

    X_scaled = scaler.transform(X)
    probs = model.predict_proba(X_scaled)[:, 1]

    result = features_df.copy()
    result["risk_score"] = probs
    result["predicted_high_risk"] = (probs >= threshold).astype(int)
    return result.sort_values("risk_score", ascending=False)


if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("usage: python models/score_project.py <features.parquet>")
        sys.exit(1)
    df_in = pd.read_parquet(sys.argv[1])
    scored = score_new_project(df_in)
    out_path = sys.argv[1].replace(".parquet", "_scored.csv")
    scored.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")
'''


def run_stage_11() -> None:
    t0 = time.time()
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    # Identify best model
    cmp_df = pd.read_csv(TABLES_DIR / "model_comparison.csv")
    if cmp_df["is_best"].sum() != 1:
        raise RuntimeError(
            f"model_comparison.csv must flag exactly one best model, got {cmp_df['is_best'].sum()}"
        )
    best_row = cmp_df[cmp_df["is_best"]].iloc[0]
    best_model_name = best_row["model"]
    print(f"[Stage 11] Best model: {best_model_name}")

    # Load tuned hyperparams for the best model
    tuned = pd.read_csv(TABLES_DIR / "tuned_hyperparameters.csv")
    params_row = tuned[tuned["model"] == best_model_name]
    params = json.loads(params_row.iloc[0]["best_params_json"]) if len(params_row) else {}
    print(f"[Stage 11] Tuned params: {params}")

    # Load full corpus + refit
    X, y, _ = load_dataset()
    print(f"[Stage 11] Refitting on full corpus: rows={len(X):,} feats={X.shape[1]}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values)

    # For LR (pipeline) we pass through the same _make_model factory but
    # then refit on the scaled matrix to keep the persisted model
    # consistent with the persisted scaler. Tree ensembles ignore the
    # scaling anyway; we still fit on the same scaled X for symmetry.
    est = _make_model(best_model_name, params, y_train=y.values)
    if est is None:
        raise RuntimeError(f"Model {best_model_name!r} unavailable")
    # If the estimator is a Pipeline that already has an internal scaler,
    # we drop it and fit the bare classifier on the externally scaled X.
    if hasattr(est, "named_steps") and "scaler" in getattr(est, "named_steps", {}):
        clf = est.named_steps["clf"]
        clf.fit(X_scaled, y.values)
        final_model = clf
    else:
        est.fit(X_scaled, y.values)
        final_model = est

    # Persist artefacts
    joblib.dump(final_model, MODELS_DIR / "best_model.pkl")
    joblib.dump(scaler, MODELS_DIR / "feature_scaler.pkl")
    with (MODELS_DIR / "optimal_threshold.txt").open("w", encoding="utf-8") as fh:
        fh.write(f"{CLASSIFICATION_THRESHOLD}")
    pd.DataFrame({"feature": list(ALL_FEATURES)}).to_csv(
        MODELS_DIR / "feature_names.csv", index=False
    )

    # Model card
    card = {
        "model_type": type(final_model).__name__,
        "model_name": best_model_name,
        "n_features": len(ALL_FEATURES),
        "feature_families": 5,
        "label": "dual_signal_combined_weight",
        "threshold": float(CLASSIFICATION_THRESHOLD),
        "lopo_f1": round(float(best_row["lopo_f1"]), 4),
        "lopo_roc_auc": round(float(best_row.get("lopo_roc_auc", float("nan"))), 4),
        "lopo_pr_auc": round(float(best_row["lopo_pr_auc"]), 4),
        "lopo_ce_at_20": round(float(best_row["lopo_ce_at_20"]), 4),
        "within_f1": round(float(best_row["within_f1"]), 4),
        "within_pr_auc": round(float(best_row["within_pr_auc"]), 4),
        "training_projects": 22,
        "training_rows": int(len(X)),
        "tuned_params": params,
        "date_trained": str(pd.Timestamp.now().date()),
    }
    with (MODELS_DIR / "model_card.json").open("w", encoding="utf-8") as fh:
        json.dump(card, fh, indent=2)

    # Scoring helper - substitute the log1p column list as a literal
    scoring_src = SCORING_FN_TEMPLATE.replace(
        "__LOG1P_COLS_PLACEHOLDER__", repr(list(LOG1P_FEATURES))
    )
    with (MODELS_DIR / "score_project.py").open("w", encoding="utf-8") as fh:
        fh.write(scoring_src)

    print("\n[Stage 11] Model card:")
    print(json.dumps(card, indent=2))

    print(f"\n[Stage 11] Total elapsed: {time.time() - t0:.1f}s")
    print("[Stage 11] Complete.")


In [ ]:
run_stage_11()
shutil.copytree('models', f'{DRIVE_BASE}/models', dirs_exist_ok=True)
print('\nModels checkpointed to Drive.')

card = json.loads(Path('models/model_card.json').read_text())
print('\n--- model_card.json ---')
print(json.dumps(card, indent=2))


## Section 14 — Score a New Project (Demo)

Demonstrate the persisted model scoring files in a project it was not
trained on — using saved features from one held-out project
(`org.apache:zookeeper`, 170 files, 17.9% positive rate). Displays the
top-10 highest-risk files and a top-20% review summary.

In [ ]:
import joblib

DEMO_PROJECT = 'org.apache:zookeeper'

best_model = joblib.load('models/best_model.pkl')
scaler = joblib.load('models/feature_scaler.pkl')
threshold = float(Path('models/optimal_threshold.txt').read_text())
feat_names = pd.read_csv('models/feature_names.csv')['feature'].tolist()

ds = pd.read_parquet('data/processed/dataset_final.parquet')
demo = ds[ds['project_id'] == DEMO_PROJECT].copy()
print(f'Demo project: {DEMO_PROJECT}  files={len(demo)}  positive_rate={100*demo["is_high_risk"].mean():.2f}%')

X = demo[feat_names].fillna(0).astype(float).copy()
# dataset_final.parquet already has log1p applied, so we do NOT log1p again here.
X_scaled = scaler.transform(X.values)
probs = best_model.predict_proba(X_scaled)[:, 1]

scored = demo[['basename', 'is_high_risk']].copy()
scored['risk_score'] = probs
scored['predicted_high_risk'] = (probs >= threshold).astype(int)
scored = scored.sort_values('risk_score', ascending=False).reset_index(drop=True)

print('\nTop 10 highest-risk files:')
display(scored[['basename', 'risk_score', 'predicted_high_risk']].head(10).style.format({'risk_score': '{:.4f}'}))

# Top-20% CE summary
n = len(scored)
top_k = max(1, int(round(0.20 * n)))
top = scored.head(top_k)
n_pos_total = int(scored['is_high_risk'].sum())
n_pos_found = int(top['is_high_risk'].sum())
ce_at_20 = n_pos_found / n_pos_total if n_pos_total else 0.0
print(f'\nReviewed top 20% of files ({top_k} of {n} files).')
print(f'Would find {n_pos_found} of {n_pos_total} truly high-risk files.')
print(f'CE@20 = {ce_at_20:.4f}')


## Section 15 — Analyze Any Apache Java Project from GitHub

Given a GitHub repository URL of any Apache Java project, this section clones
the repo, computes the 27 features from git history, scores all Java files
using the persisted model, and outputs a ranked risk report. SonarQube
features default to zero if not available — the model still runs using the
14 git-based features only, with slightly reduced accuracy.

In [ ]:
# STEP 1 — User input. Edit this URL to analyze any Apache Java project.
GITHUB_REPO_URL = 'https://github.com/apache/commons-lang'
REPO_NAME = 'commons-lang'
CLONE_DEPTH = 500  # ~500 most recent commits; lower if the repo is huge


In [ ]:
# STEP 2 — Clone the repository
# Shallow clone first for speed, then unshallow so PyDriller can diff every
# commit against its real parent (a grafted-parent boundary in a shallow
# clone breaks `git diff-tree` and crashes PyDriller).
import subprocess

clone_dir = f'/content/repos/{REPO_NAME}'
Path('/content/repos').mkdir(parents=True, exist_ok=True)
if not Path(clone_dir).exists():
    subprocess.run(
        ['git', 'clone', f'--depth={CLONE_DEPTH}', GITHUB_REPO_URL, clone_dir],
        check=True,
    )
    print(f'Cloned {REPO_NAME} into {clone_dir}')
else:
    print(f'Already cloned: {clone_dir}')

# Unshallow if needed so PyDriller can diff every commit against its parent.
is_shallow = (Path(clone_dir) / '.git' / 'shallow').exists()
if is_shallow:
    print('Unshallowing clone (needed for PyDriller diffs) ...')
    subprocess.run(['git', '-C', clone_dir, 'fetch', '--unshallow'], check=True)

n_commits = int(subprocess.check_output(
    ['git', '-C', clone_dir, 'rev-list', '--count', 'HEAD']
).decode().strip())
print(f'Available commits: {n_commits}')


In [ ]:
# STEP 3 — Extract git features (Families 3, 4, 5) with PyDriller.
# Families 1 and 2 (SonarQube) default to 0 — the model still runs but
# accuracy is reduced (~80% of the full pipeline). This is noted below.
from pydriller import Repository

_BUGFIX_RE = re.compile(BUGFIX_REGEX, re.IGNORECASE)

print(f'Traversing {REPO_NAME} commits with PyDriller ...')
file_records = {}  # basename -> dict of per-file aggregates
authors_per_file = {}  # basename -> {author: commit_count}
cochange_edges = Counter()  # (a, b) sorted -> co-change weight
all_basenames = set()

t_now = pd.Timestamp.utcnow()
t_30d = t_now - pd.Timedelta(days=30)
t_90d = t_now - pd.Timedelta(days=90)

# Cap traversal to the most recent CLONE_DEPTH commits to keep runtime
# bounded on large repos even after unshallowing.
head_rev_list = subprocess.check_output(
    ['git', '-C', clone_dir, 'rev-list', f'--max-count={CLONE_DEPTH}', 'HEAD']
).decode().splitlines()
target_hashes = set(head_rev_list)
print(f'Traversing latest {len(target_hashes)} commits ...')

n_seen = 0
n_skipped = 0
for commit in Repository(clone_dir).traverse_commits():
    if commit.hash not in target_hashes:
        continue
    n_seen += 1
    commit_date = pd.Timestamp(commit.author_date).tz_convert('UTC') if commit.author_date.tzinfo else pd.Timestamp(commit.author_date, tz='UTC')
    msg = commit.msg or ''
    is_bugfix = bool(_BUGFIX_RE.search(msg))
    author = commit.author.name if commit.author else 'unknown'
    touched_in_commit = []
    try:
        modified_files = commit.modified_files
    except Exception as _e:
        # Grafted-parent boundary or any other diff failure — skip this commit.
        n_skipped += 1
        continue
    for mf in modified_files:
        path = mf.new_path or mf.old_path or ''
        if not path.endswith('.java'):
            continue
        p = path.replace('\\', '/').lower()
        if any(bad in p for bad in PATH_EXCLUSION_PATTERNS):
            continue
        bn = path.rsplit('/', 1)[-1]
        added = mf.added_lines or 0
        removed = mf.deleted_lines or 0
        rec = file_records.setdefault(bn, {
            'total_commits_pre': 0,
            'code_churn_pre': 0,
            'recent_churn_90d': 0,
            'commit_frequency_30d': 0,
            'first_commit': commit_date,
            'last_commit': commit_date,
            'bugfix_commits_pre': 0,
            'bugfix_commits_90d': 0,
        })
        rec['total_commits_pre'] += 1
        rec['code_churn_pre'] += added + removed
        if commit_date >= t_90d:
            rec['recent_churn_90d'] += added + removed
            if is_bugfix:
                rec['bugfix_commits_90d'] += 1
        if commit_date >= t_30d:
            rec['commit_frequency_30d'] += 1
        if commit_date < rec['first_commit']:
            rec['first_commit'] = commit_date
        if commit_date > rec['last_commit']:
            rec['last_commit'] = commit_date
        if is_bugfix:
            rec['bugfix_commits_pre'] += 1
        authors_per_file.setdefault(bn, Counter())[author] += 1
        touched_in_commit.append(bn)
        all_basenames.add(bn)
    if len(touched_in_commit) > 1:
        for a, b in combinations(sorted(set(touched_in_commit)), 2):
            cochange_edges[(a, b)] += 1

print(f'Traversed {n_seen} commits ({n_skipped} skipped); collected {len(all_basenames)} Java basenames.')

# Co-change graph features (igraph for parity with the training pipeline)
import igraph as ig
node_list = sorted(all_basenames)
node_idx = {n: i for i, n in enumerate(node_list)}
edges = [(node_idx[a], node_idx[b]) for (a, b) in cochange_edges]
weights = list(cochange_edges.values())
G = ig.Graph(n=len(node_list), edges=edges, directed=False)
if len(node_list) > 2 and edges:
    bc_raw = G.betweenness(directed=False)
    denom = (len(node_list) - 1) * (len(node_list) - 2) / 2.0
    bc_norm = [b / denom for b in bc_raw]
else:
    bc_norm = [0.0] * len(node_list)
try:
    pr = G.pagerank(damping=0.85, weights=weights if weights else None)
except Exception:
    pr = G.pagerank(damping=0.85)
deg = G.degree()

# Per-node entropy over normalised neighbour edge weights
neighbour_weights = {n: [] for n in node_list}
for (a, b), w in cochange_edges.items():
    neighbour_weights[a].append(w)
    neighbour_weights[b].append(w)
entropy = {}
for n in node_list:
    ws = neighbour_weights[n]
    s = sum(ws)
    if s <= 0:
        entropy[n] = 0.0
        continue
    h = 0.0
    for w in ws:
        p = w / s
        if p > 0:
            h -= p * math.log2(p)
    entropy[n] = h

# Assemble the 27-column matrix
rows = []
for bn in node_list:
    rec = file_records[bn]
    age_days = max(0, (rec['last_commit'] - rec['first_commit']).days)
    days_since = max(0, (t_now - rec['last_commit']).days)
    authors = authors_per_file.get(bn, Counter())
    n_authors = len(authors)
    max_author = max(authors.values()) if authors else 0
    ownership = (max_author / rec['total_commits_pre']) if rec['total_commits_pre'] else 1.0
    bug_density = (rec['bugfix_commits_pre'] / rec['total_commits_pre']) if rec['total_commits_pre'] else 0.0
    i = node_idx[bn]
    row = {
        'basename': bn,
        # Family 1: size/complexity — SonarQube features default to 0
        'ncloc': 0.0, 'complexity': 0.0, 'cognitive_complexity': 0.0,
        'functions': 0.0, 'classes': 0.0,
        # Family 2: static debt — SonarQube features default to 0
        'n_code_smells': 0, 'n_bugs': 0, 'total_debt_minutes': 0.0,
        'issue_density': 0.0, 'duplicated_lines_density': 0.0,
        # Family 3: historical
        'total_commits_pre': rec['total_commits_pre'],
        'code_churn_pre': rec['code_churn_pre'],
        'recent_churn_90d': rec['recent_churn_90d'],
        'commit_frequency_30d': rec['commit_frequency_30d'],
        'file_age_days': age_days,
        'days_since_last_change': days_since,
        'contributor_count': n_authors,
        'ownership_ratio': ownership,
        # Family 4: co-change graph
        'cocg_degree': deg[i],
        'cocg_pagerank': pr[i],
        'cocg_betweenness': bc_norm[i],
        'cocg_entropy': entropy[bn],
        # Family 5: prior defect
        'bugfix_commits_pre': rec['bugfix_commits_pre'],
        'bugfix_commits_90d': rec['bugfix_commits_90d'],
        'bug_density_pre': bug_density,
        'n_jira_bugs_pre': 0,
        'jira_blocker_flag': 0,
    }
    rows.append(row)

feat_df = pd.DataFrame(rows)
print(f'Built feature matrix: {feat_df.shape}')
print('\nStatic features unavailable without SonarQube. Using git-based features only.')
print('Expected accuracy: ~80% of full model.')


In [ ]:
# STEP 4 — Score files using the persisted model
import joblib

best_model = joblib.load('models/best_model.pkl')
scaler = joblib.load('models/feature_scaler.pkl')
feat_names = pd.read_csv('models/feature_names.csv')['feature'].tolist()
threshold = float(Path('models/optimal_threshold.txt').read_text())

# Align to the trained feature order; missing columns become 0
X = feat_df.reindex(columns=feat_names, fill_value=0).fillna(0).astype(float)
assert list(X.columns) == feat_names, 'Feature column mismatch'

# Apply log1p to the same 9 skewed columns the training pipeline did
for col in LOG1P_FEATURES:
    if col in X.columns:
        X[col] = np.log1p(X[col])

X_scaled = scaler.transform(X.values)
probs = best_model.predict_proba(X_scaled)[:, 1]

scored = pd.DataFrame({
    'basename': feat_df['basename'],
    'risk_score': probs,
    'predicted_high_risk': (probs >= threshold).astype(int),
}).sort_values('risk_score', ascending=False).reset_index(drop=True)

print(f'Scored {len(scored)} Java files.')


In [ ]:
# STEP 5 — Display results
import matplotlib.pyplot as plt

print(f'\nTop 20 highest-risk files in {REPO_NAME}:')
display(scored.head(20).style.format({'risk_score': '{:.4f}'}))

n = len(scored)
k = max(1, int(round(0.20 * n)))
n_high = int(scored['predicted_high_risk'].sum())
print(f'\nTotal Java files analyzed   : {n}')
print(f'Files in top 20% review budget: {k}')
print(f'High-risk files predicted    : {n_high}')
print(f'\nTo prioritize maintenance effort, focus on the top {k} files listed above.')

top20 = scored.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20['basename'][::-1], top20['risk_score'][::-1], color='#d62728')
ax.set_xlabel('Predicted risk score')
ax.set_title(f'Top 20 high-risk files: {REPO_NAME}')
fig.tight_layout()
plt.show()


In [ ]:
# STEP 6 — Save report to Drive
out_path = f'{DRIVE_BASE}/risk_report_{REPO_NAME}.csv'
scored.to_csv(out_path, index=False)
print(f'Full report saved to Drive: risk_report_{REPO_NAME}.csv')


## Taking This Further

This scoring logic can be integrated into a GitHub Actions workflow to
automatically flag high-risk files on every pull request or release. The
workflow would:

1. Checkout the repository
2. Install PyDriller and the model artifacts
3. Run the feature extraction and scoring
4. Comment on the PR with the top high-risk files, or fail the check if any
   file exceeds a defined risk threshold

The persisted model artifacts (saved to `models/` in Section 13) are all that
is needed to run inference on any new project.